<!-- Last changed: Facebook_20260901A -->
> **Enclave notebook — `Facebook_YYYYMMDD<A-Z>.ipynb` series.** This file is *authoring
> only*. It imports `metacontentlibraryapi`, which exists only inside Meta's Secure
> Research Environment; it cannot be executed, tested, or run against the API from
> this repo. Committed with **all outputs cleared** — Meta permits no data egress, so
> nothing produced by running it may be saved back here.
>
> Predecessors: `Facebook_20260831A.ipynb` (block timing, per-category checkpoints),
> `Facebook_20260825G.ipynb` (discovery scope gap), `Facebook_20260824A.ipynb`
> (checkpointing), `Facebook_20260822A.ipynb` (aggregate grain),
> `Claude_Reference/H5N1_Aggregate_20260819.ipynb` (origin).

# H5N1 Farm-Page Post Aggregation — Meta Content Library

Page discovery, post collection, term flagging, sentiment, and a single aggregate table
at the grain **(PageCategory, State, Zip, Date)**.

Every code cell carries a `# BLOCK nn - name` header. Blocks run in order; the two
long-running ones (**07** page discovery, **12** post pull) print a line per slice or
job and are meant to be **collapsed** — each is followed immediately by a summary block
(**08**, **13**) that recomputes and prints everything you actually need to read.

Every block also brackets itself with a wall-clock start and end and reports its own
elapsed hours and minutes; **block 23** prints the full ledger. See *§9 — Timing*.

---

## What changed in `20260901A`

Three things, all of them opening a question the notebook could not previously ask.

**1. Block 07 projects an ETA.** `20260831A` made an interrupted run visible and
resumable; it still gave no way to tell, *while discovery was running*, how much was
left. Every slice now prints `[14/~86, ~2h 38m left]`. The denominator carries a tilde
because it is a projection, not a total: the L2 split fires only for categories that
reach the cap, so how many slices a run will do is not knowable in advance. Rate comes
from the last 20 slices rather than all of them — a capped slice is 100 rate-limited
calls and an empty one is a single call — and slice durations are written into
`PAGE_SLICES`, so a resumed run is calibrated off the checkpoint from its first slice.

**2. Emoji are scored — and tested, not assumed.** Earlier versions said emoji were
"deliberately not scored." What was actually happening is that the tokenizer,
`[a-z0-9']+`, discarded every non-ASCII character, so emoji contributed **exactly zero**
and the question was never tested. The prior cohort found emoji added noise; the
Laison/Lyme work found the opposite. Every post is now scored twice — `sentiment`
(text-only, unchanged meaning) and `sentiment_emoji` — and both reach the export. Block
19A reports whether they differ. Whether they *help* is decided downstream, against the
USDA target, which cannot come into the enclave.

**3. Spanish is collected and scored.** `_base_params` sent `"lang": "en"`, so Spanish
posts were not being scored badly — they were never **collected**. `POST_LANGS` now
drives the request, `POST_TEXT_BY_LANG` the term flag, and `_LEX_BY_LANG` the scoring.
The structure takes a fourth language as a data edit rather than a code change.

> ⚠️ **`n_posts` has never been the "every post these pages made" denominator this
> document claimed** — it was every *English* post. With `POST_LANGS = ["en", "es"]` it
> is every post in those two languages. The ~6,000,000-post estimate on record is
> likewise English-only: **re-run `facebook/posts/estimate` before submitting anything**,
> because that figure is already twelve budget-weeks.

> ⚠️ **Whether `lang` accepts a comma list is unverified** — the docs describe a single
> ISO 639-1 code. `LANG_REQUEST_MODE = "auto"` spends one synchronous call in block 11 to
> find out and falls back to one request per language, which always works but doubles the
> job count and the submission wall-clock. In that mode `job_key` gains a language
> segment (`b003|2024-01-01|es`); keys written by earlier versions have none, so an old
> checkpoint and a new run never collide — but they do not share work either, and the
> English pass is re-submitted.

Nothing here changes what discovery collects or how any existing column is computed. The
page universe is identical, `sentiment` still means what it meant, and every previously
exported column keeps its definition.

---

## What changed in `20260831A`

A `SHARD = "nick"`, `RUN_MODE = "full"` run on 2026-08-31 sat on block 07 for over three
hours and was killed. Nothing was wrong with it — that is what full-scope discovery
costs — but two things made it impossible to know that, and both are fixed here.

**1. Discovery is 3–4 hours by construction, and nothing said so.** `RUN_MODE = "full"`
sets `MAX_API_PAGES = None`, so a slice pages all the way to `SYNC_SEARCH_CAP` — 100 API
calls at `PAGE_SEARCH_SLEEP_S = 1.05` each, three to four minutes per capped slice.
`DISCOVER_ALL_CATEGORIES = True` runs all six production categories, five of which
exceed 1,000 pages and so sub-split across all sixteen `PageDescription` terms: **6 L1 +
80 L2 = 86 slices**. That arithmetic is the run, not a fault in it.

**2. The cold-start path checkpointed nothing until the entire sweep returned.** So
killing it at hour three discarded every row, the query budget spent to get them — page
rows count against the same 500,000-per-rolling-7-days ledger as the async jobs — and
any possibility of resuming, because the scope-gap logic added in `20260825G` reads
`SEARCHED_CATEGORIES` back off level-1 slices that were never written to disk.

> **The fix.** `_discover_categories` now calls `checkpoint()` after **every category**,
> restricted via a new `only=` argument to `PAGE_ROWS` and `PAGE_SLICES` so it does not
> re-serialise `POST_ROWS` six times. The write lands *after* the L2 sub-split, never
> between the L1 call and it: a category interrupted mid-split must not persist a
> level-1 slice claiming it was searched, since that claim is what the gap logic trusts.
> Worst-case loss drops from the whole sweep to one category, and the interrupted run
> resumes on the categories it never reached.

Neither of these changes what discovery collects. The page universe is identical; only
its resumability and its observability changed.

---

## What the last full run measured — and why this revision exists

Page discovery on 2026-08-24, all eight categories:

| stage | pages |
|---|---|
| returned by `facebook/pages/preview` | 18,062 |
| US-located | 13,265 |
| with a wanted **primary** category | 12,787 |
| primary category `Farm` | 3,837 |

`facebook/posts/estimate` over those surfaces returned **~6,000,000 posts**. Two things
follow, and both are why this is a major revision rather than a patch:

**1. Every async job would have been silently truncated.** 12,787 surfaces is 52 batches
of 250. 6,000,000 ÷ 52 ≈ **115,000 posts per job against a documented 100,000 ceiling**.
Restricting to Farm pages does not save it either — 3,837 surfaces is 16 batches at
~113,000 each. The old design had no axis on which to subdivide further, so `capped`
would have fired on essentially every batch and `n_posts` would have been a floor of
unknown tightness.

> **The fix: `JOB_WINDOW`.** Jobs are now partitioned by **surface batch × time window**,
> not by surface batch alone. At the default `"year"` that is 52 × 5 = 260 jobs of
> ~23,000 posts each — comfortably under the cap. `"quarter"` (988 jobs) and `"month"`
> (2,860 jobs) exist for narrower slicing, at 18 h and 52 h of submission time
> respectively given the 1 query/minute async rate limit.

**2. 6,000,000 posts is twelve weeks of query budget.** The budget is 500,000 records
per rolling 7 days, shared across sync and async. A full-scope `all_posts` pull is not a
single sitting; it is a staged collection. Block 10 therefore keeps a **budget ledger**
in the checkpoint and stops submitting when the next job's estimate would exceed what
the rolling window has left. Re-run next week and it picks up where it stopped.

If twelve weeks is not acceptable, the scope levers, in the order I would pull them:

1. **Narrow the window.** `PostStartDate = "2024-08-01"` covers the outbreak period and
   cuts the pull by roughly half.
2. **`RESTRICT_TO_FARM_ONLY = True`.** ~1.8M posts. Loses the cross-category comparison
   that made `page_category` a grouping dimension in the first place.
3. **`POST_QUERY_MODE = "broad_terms"`.** Cheapest by far, but it destroys the true
   denominator — `n_posts` stops meaning "everything these pages posted", which is the
   measurement this notebook exists to produce. Pull this lever last.

## Sharding the collection across researchers

Twelve budget-weeks is a long time to hold a pull open. The work is therefore split
three ways on **`page_category`**, selected by the `SHARD` name in block 02.

`page_category` is not merely a convenient key — it is the **first grouping key of the
aggregate**, so shards are disjoint in the output and combining is a `concat` with no
re-aggregation. Splitting on state, date, or page-id ranges would put the same aggregate
cell in two people's results and force a merge that re-derives means; `mean_sentiment`
cannot be re-derived from totals, so that merge would silently lose it. A page lands in
exactly one shard because block 09 filters on *primary* category, even though raw
`PAGE_ROWS` overlap between researchers.

| shard | categories | pages (2026-08-24) | est. posts | jobs | submit |
|---|---|---|---|---|---|
| `nick` | Farm, Animal Shelter | 4,247 | ~1.99M | 85 | 1.5 h |
| `max` | Agricultural Service, Agricultural Cooperative | 4,523 | ~2.12M | 95 | 1.7 h |
| `waree` | Veterinarian, Agriculture | 3,932 | ~1.84M | 80 | 1.4 h |
| `smoke` | Zoo | 28 | ~13k | 1 | seconds |

Rebalance off the real per-shard estimates rather than these — 469 posts/page is a flat
average across all categories, and posting rates almost certainly differ by category.

> ⚠️ **Whether this actually buys anything depends on how the record budget is scoped.**
> If 500,000 records / 7 days is **per researcher**, three shards finish in ~4 weeks. If
> it is **per research project**, the shards share one pool and the split buys only
> submission wall-clock (~3x, since the 1 query/minute async limit is per user). Confirm
> which before treating the four-week figure as real.

### Combining

Block 22 does it, and works whether or not researchers share an enclave filesystem: it
reads `shard_<name>_exportagg.*` plus a manifest sidecar from `SHARD_DIR`, preferring
CSV (which survives output review) over pickle (which does not). Run it inside the
enclave on a shared workspace, or outside on three separately-approved exports.

The mechanical merge is trivial. The real risk is **configuration drift** — three
people, three environments, one dataset. Block 22 hard-fails when shards disagree on
`SENTIMENT_ENGINE`, `POST_QUERY_MODE`, `RUN_START`/`RUN_END`, `ENGAGEMENT_MIN`,
`JOB_WINDOW`, or `MIN_CELL_POSTS`. `SENTIMENT_ENGINE` is the dangerous one: if one
researcher gets `vaderSentiment` and another silently falls back to the inline lexicon,
every number still computes, every sanity check still passes, and `mean_sentiment` is
quietly incomparable across a third of the corpus.

### Smoke preset

`SHARD = "smoke"` runs the whole pipeline against **Zoo** — 28 pages, one surface batch,
seconds. Zoo is deliberately excluded from every production shard, so a smoke run can
never be mistaken for or contaminate real data. Block 05A then prints a parity report
covering the environment, the sentiment fingerprint, and a checkpoint round-trip.

> ⚠️ **The parity report contains no counts, deliberately.** Versions, engine name,
> fixed-string sentiment scores, and pass/fail booleans are not Meta data and can be
> shared over Slack or email. A row count *is* derived from Meta data. Do not paste
> "got 847 pages" anywhere — that is an egress decision, not a status update.

## Page discovery is a measured sample, not a census

`facebook/pages/preview` is synchronous-only — there is no `facebook/pages/job` — so
every individual search is capped at 1,000 results in **both** run modes. Discovery is
partitioned one search per category, then sub-split across `PageDescription` terms for
any category that still hits the cap.

**On the last run, 13 slices were still at the 1,000 cap after sub-splitting**, spanning
at least `Farm` and `Agricultural Service` across the terms *livestock, farm, farming,
ranch, agriculture, agricultural*. Those slices are truncated by an unknown amount. A
third level — `PAGE_REFINE_CAPPED`, which re-runs a capped slice as `(term & refiner)`
AND-groups — is available in block 07 and **defaults to off**, because widening the page
universe makes the post-budget problem strictly worse. Turn it on only if the page
universe itself is the object of study.

## Three weightings, because they answer three different questions

The aggregate carries the same underlying reactions three ways, and the difference
between them is the whole point rather than redundancy:

| statistic | weights by | what it answers |
|---|---|---|
| `like_ratio_total` | **reactions** | pooled over the cell. One national brand with 50,000 reactions dominates every small farm in the state. |
| `mean_like_ratio` | **posts** | each post counts once, however popular. |
| `mean_like_ratio_page_eq` | **pages** | each page counts once, however big. Ten small farms no longer collapse into one national brand. |

`max_sad_share_term` is not an average at all. It is the **loudest single distress signal
in the cell among posts that matched a real `PostText` concept** — the maximum of
`sad / (like + sad)` over term-matching posts. One small farm reporting a single dead
bird is the event of interest, and every mean above buries it by construction. The
term restriction is what makes it a bird-flu signal rather than a general-misery signal;
`n_term_ratio_defined` is the count it was taken over, and a max over one post means
something very different from a max over fifty.

`mean_sentiment_page_eq` applies the same page-equal weighting to sentiment. Sentiment is
the dependent variable, so the argument for it is if anything stronger than for the
ratio.

## No median, and why

`median_sentiment` was removed in `20260825E`. It was never a page-equal statistic —
that is `mean_sentiment_page_eq` — it was the median over *posts*, a central-tendency
measure whose entire purpose is resistance to outliers. In the ten-small-farms-and-one-
national-brand cell, the median is guaranteed to report the ordinary farm and ignore the
one reporting a dead bird. It answers the opposite of the question this notebook asks.

Two extremes replace it:

* **`min_sentiment_term`** — the most negative *term-matching* post in the cell. This is
  the exact sentiment twin of `max_sad_share_term`, and the two should be read together:
  one measures how sad the reactions were, the other how negative the text was.
* **`min_sentiment`** — the most negative post regardless of term. Useful as the control
  the term-restricted version is compared against: distress that never used a flu word is
  the baseline that `min_sentiment_term` has to beat to be evidence of anything.

## Model-ready columns: NaN kept, and filled

The term-restricted signals are NaN wherever no term-matching post qualified — either
because no post in the cell matched a `PostText` concept at all, or because the ones that
did got no reactions. Both are common, and both are honest: NaN means *not measurable
here*, which is not the same as zero.

But a downstream model cannot consume NaN without dropping the row, and those rows are
the quiet majority. So each signal is carried twice:

| column | meaning |
|---|---|
| `max_sad_share_term` | the measurement. NaN where nothing qualified. |
| `max_sad_share_term_model` | the same, with NaN filled by `MODEL_FILL_VALUE` (0.0). |

Same for `min_sentiment_term` / `min_sentiment_term_model`. Block 17 reports how many
cells were filled and, importantly, *why* — split by cause.

> **On filling with zero.** This project has a standing rule that suppressed NASS values
> must never be coalesced to 0, because withheld inventory means *unknown* and zeroing it
> is unrecoverably wrong. The rule holds; this is not a violation of it, and the
> difference is worth being explicit about. There, the zero destroys information and
> nothing marks the rows it touched. Here, `n_posts_term`, `n_term_ratio_defined`, and
> `n_term_sentiment_scored` identify the filled set exactly, the NaN column is kept
> alongside, and 0.0 carries the right meaning for these particular measures — "no
> observed distress", "no observed negativity". If any of those three conditions stopped
> being true, the fill would become the NASS mistake.

Other float columns (`mean_sentiment`, `like_ratio_total`, the page-equal means) are NaN
only when a cell had nothing to measure at all, which is rarer and is properly the
modelling notebook's decision rather than this one's. Block 17 prints the NaN rates so
that decision can be made with numbers.

## Thin cells are retained, deliberately

`MIN_CELL_POSTS` defaults to **0 — no suppression** — and that is now a considered
position rather than an unset knob. A cell with `n_posts = 1` is not noise to be tidied
away; under the reasoning above it is very often *the signal*. Suppressing at 5 or 10
would delete exactly the small single-farm observations that `max_sad_share_term` exists
to surface.

> ⚠️ **I am confabulating** on what Meta's output review actually requires. The lever is
> retained (`MIN_CELL_POSTS`) so a reviewer's requirement can be met without a code
> change, but whether thin cells must be suppressed in an export of *counts and ratios
> with no post text* is a question for Meta's review team, not something to infer. Ask
> before assuming either way. Nothing about retention changes inside the enclave — the
> analysis dataset always keeps every cell; suppression is a review-time decision applied
> only to what is requested out.

### A note on names

`SourcePosts` holds **every post from every page in this shard**, not farm posts. It was
called `FarmPosts` through `20260825C` — a leftover from `H5N1_20260819`, where
`PageCategories` really was `["Farm"]`. It stopped being true in `20260822A` when the
category list widened to make `page_category` a grouping dimension, and the name did not
follow. Renamed in `20260825D` because a name that asserts a filter the code does not
apply is worse than a vague one: a reviewer reading block 17 could reasonably assume the
aggregate is farm-scoped.

The lineage now reads straight through: `CleanUSPages` → `SourcePages` → `SourcePosts`.

`FarmPages` keeps its name — it genuinely is the Farm-only subset, and it feeds
`SourcePages` only when `RESTRICT_TO_FARM_ONLY` is set. `FarmPostsFiltered` became
`PostsWithTerm`, which is what it always was: the term-matching subset, retained for
continuity with the older notebooks and used nowhere downstream.

### Aggregate output columns
| Column | Meaning |
|---|---|
| `n_posts` | all posts in the cell |
| `n_posts_term` | posts whose text matches a `PostText` term |
| `n_engagement_ge10` | posts where `like_count + sad_count >= 10` |
| `n_ratio_defined` / `n_ratio_undefined` | posts with `like+sad > 0` / `== 0` |
| `n_sentiment_scored` | posts with text that scored |
| `like_count_total` / `sad_count_total` | summed reactions |
| `n_pages` | distinct pages contributing to the cell |
| `n_term_ratio_defined` | term-matching posts with a defined ratio |
| `like_ratio_total` | `like/(like+sad)` pooled over the cell — reaction-weighted |
| `mean_like_ratio` | mean of `like/(like+sad)` over `n_ratio_defined` posts — post-weighted |
| `mean_like_ratio_page_eq` | mean over pages, each page counting once — page-weighted |
| `mean_sentiment` | compound sentiment over `n_sentiment_scored` posts |
| `min_sentiment` | most negative post in the cell |
| `min_sentiment_term` | most negative **term-matching** post — the sentiment twin of `max_sad_share_term` |
| `n_term_sentiment_scored` | term-matching posts that scored |
| `max_sad_share_term_model` / `min_sentiment_term_model` | the two signals with NaN filled — what the model consumes |
| `mean_sentiment_page_eq` | mean sentiment over pages, each page counting once |
| `max_sad_share_term` | max `sad/(like+sad)` among term-matching posts — the early-warning signal |

### Four things to know before you trust the numbers
1. **`n_posts` is every post these pages made *in `POST_LANGS`*.**
   `POST_QUERY_MODE = "all_posts"` sends no `q`, so no term filter narrows it — but the
   request has always carried a `lang`, which does. Through `20260831A` that was
   hardcoded `"en"`, so every `n_posts` on record is an English-only count and the
   "every post" claim this list used to make was wrong. Report the denominator as
   "posts in `en`, `es`", and if the API rejects a `q`-less query, switch to
   `"broad_terms"` and read `n_posts` as "broad-term posts" on top of that.
2. **Zip and State are the *page's* registered location, not the author's.** Meta
   exposes no author geography. See the `meta-content-library-constraints` memory.
3. **Sentiment is lexicon-based and computed in-enclave.** No model weights are
   downloaded — the enclave has no egress. See block 05 for the fallback chain.
   **As of the 2026-08-25 smoke run the enclave resolves to `inline-lexicon`** —
   fallback #3, the hand-built domain lexicon in block 05, not VADER. That is fine for
   comparability as long as all three shards agree (block 05A's fingerprint proves it),
   but the methods write-up must not describe these scores as VADER scores.
4. **Nothing here leaves the enclave without Meta's output review.** Block 20 applies
   small-cell suppression to make the aggregate reviewable; raw post text is never part
   of the export shape. Checkpoints (blocks 04, 21) write to the enclave's own disk and
   are **not** an export.


In [ ]:
# BLOCK 00 - Imports and environment
# Last changed: Facebook_20260901A
# The timer bootstraps itself here. Block 00 is the block that DEFINES the
# timing helpers, so it stamps its own start by hand and lets _block_end adopt
# the record at the bottom.
import time

_T0 = time.time()
print(f"[BLOCK 00] start  {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_T0))}")

from metacontentlibraryapi import MetaContentLibraryAPIClient as client
from datetime import date, timedelta
import gzip
import hashlib
import json
import math
import os
import re
import unicodedata

import numpy as np
import pandas as pd

# Jupyter auto-displays a cell's last expression, and every block now ENDS with
# a _block_end call, which suppresses that. The three blocks that showed a
# DataFrame that way call display() explicitly instead. print is the fallback
# for a non-IPython kernel.
try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

# ---------------------------------------------------------------------------
# BLOCK TIMING LEDGER
# ---------------------------------------------------------------------------
# Added 20260831A. Block 07 ran for over three hours at full scope before it was
# killed, and nothing in the notebook said how long anything had taken - there
# was no way to tell a slow block from a hung one, or how close the run was to
# finishing. Every block now stamps its own start and end.
#
# Two properties that matter in a notebook rather than a script:
#
#   - re-running a cell REPLACES its record instead of appending a second one,
#     and `runs` counts how many times it has executed. Cells here get re-run
#     constantly and an append-only log would read as nonsense.
#   - a block that raises or is interrupted never reaches its _block_end, so it
#     stays status='running' and timings_frame() measures its elapsed time to
#     NOW. An aborted block is VISIBLE, not absent - which is exactly the case
#     that went unrecorded on the run this was written for.
#
# Wall-clock only: no counts, nothing Meta-derived. The ledger is safe to share
# for coordination, and it is not part of the export.
BLOCK_TIMINGS = {}


def _fmt_hm(seconds):
    """Elapsed as hours and minutes - the unit the long blocks are judged in."""
    h, rem = divmod(int(seconds), 3600)
    return f"{h}h {rem // 60:02d}m"


def _block_start(label):
    """Open (or reopen) a block's timing record and print the wall-clock start."""
    now = time.time()
    prior = BLOCK_TIMINGS.get(label)
    BLOCK_TIMINGS[label] = {
        "block": label,
        "started": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(now)),
        "ended": None,
        "elapsed": None,
        "hours": None,
        "minutes": None,
        "seconds": None,
        "status": "running",
        "runs": prior["runs"] + 1 if prior else 1,
        "_t0": now,
        "_order": prior["_order"] if prior else len(BLOCK_TIMINGS),
    }
    print(f"[{label}] start  {BLOCK_TIMINGS[label]['started']}")
    return now


def _block_end(label, t0=None):
    """Close a block's record and print the wall-clock end plus elapsed."""
    now = time.time()
    rec = BLOCK_TIMINGS.get(label)
    if rec is None:
        # Cell run out of order, or block 00 closing the record it opened by
        # hand before these helpers existed.
        _t0 = now if t0 is None else t0
        rec = BLOCK_TIMINGS[label] = {
            "block": label,
            "started": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(_t0)),
            "runs": 1,
            "_t0": _t0,
            "_order": len(BLOCK_TIMINGS),
        }
    t0 = rec["_t0"] if t0 is None else t0
    elapsed = now - t0
    rec.update(ended=time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(now)),
               elapsed=_fmt_hm(elapsed),
               hours=int(elapsed // 3600),
               minutes=int(elapsed % 3600 // 60),
               seconds=round(elapsed, 1),
               status="done")
    print(f"[{label}] end    {rec['ended']}   elapsed {rec['elapsed']} "
          f"({rec['seconds']:,.1f}s)")
    return elapsed


def timings_frame():
    """The ledger as a DataFrame, in first-run order.

    A block still marked 'running' is measured to NOW and flagged with a
    trailing '+', so an interrupted or still-executing block reads as a
    duration rather than a blank.
    """
    now = time.time()
    rows = []
    for rec in sorted(BLOCK_TIMINGS.values(), key=lambda r: r["_order"]):
        row = {k: v for k, v in rec.items() if not k.startswith("_")}
        if row.get("status") == "running":
            elapsed = now - rec["_t0"]
            row.update(elapsed=_fmt_hm(elapsed) + "+",
                       hours=int(elapsed // 3600),
                       minutes=int(elapsed % 3600 // 60),
                       seconds=round(elapsed, 1))
        rows.append(row)
    out = pd.DataFrame(rows, columns=["block", "started", "ended", "elapsed",
                                      "hours", "minutes", "seconds", "runs",
                                      "status"])
    for _c in ("hours", "minutes", "seconds", "runs"):
        out[_c] = pd.to_numeric(out[_c], errors="coerce")
    # A block that never ended reads better blank than as NaN, which is what
    # to_string renders None as in an object column.
    for _c in ("started", "ended", "elapsed", "status"):
        out[_c] = out[_c].fillna("").astype(str)
    return out


print(f"pandas {pd.__version__}   numpy {np.__version__}")
print(f"cwd    {os.getcwd()}")
_block_end("BLOCK 00", _T0)

In [ ]:
# BLOCK 01 - Search vocabulary
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 01")

# ----------------------------------------------------------------------------
# PAGE DISCOVERY (who is posting)
# ----------------------------------------------------------------------------
# PageCategories is NO LONGER SET HERE. It is derived from SHARD in block 02, so
# that three researchers cannot drift into three different category lists by
# hand-editing. The reference counts below are the 2026-08-24 discovery run and
# exist so block 09 can report drift against a known baseline.
PageCategoryPages = {
    "Farm": 3837,
    "Agricultural Service": 3340,
    "Veterinarian": 1996,
    "Agriculture": 1936,
    "Agricultural Cooperative": 1183,
    "Animal Shelter": 410,
    "Wildlife Sanctuary": 57,   # dropped from production: low return
    "Zoo": 28,                  # dropped from production; reserved for SHARD="smoke"
}

PageDescription = [
    "poultry", "chicken", "turkey", "egg", "hatchery",
    "flock", "livestock", "farm", "farming", "agriculture",
    "agricultural", "ranch", "dairy", "veterinary", "veterinarian",
    "avian",
]

# Third-level split for slices still at the 1,000 cap after sub-splitting by
# PageDescription. Applied as "(term & refiner)" AND-groups, so coverage is
# incomplete by construction: a page whose description carries the term but none
# of these refiners stays invisible. Off by default - see PAGE_REFINE_CAPPED.
PAGE_REFINERS = [
    "poultry", "chicken", "turkey", "duck", "egg", "eggs", "hatchery",
    "flock", "broiler", "layer", "pasture", "organic", "family", "local",
]

PageCountry_Code = "US"

# ----------------------------------------------------------------------------
# POST DISCOVERY (what they wrote)
# ----------------------------------------------------------------------------
# PostText = the *precise* concept list, PER LANGUAGE. Multi-word entries are
# phrases, which the API cannot search, so they are enforced client-side as
# has_post_term.
#
# 20260901A - this was a single English list. Splitting it by language costs NO
# query budget: in the default POST_QUERY_MODE = "all_posts" no q is sent, so
# the term flag is entirely client-side. Only widening the request's `lang`
# (block 02's POST_LANGS) spends anything.
#
# Subtypes and acronyms that are the SAME STRING in every language live apart,
# and are applied to every post regardless of its detected language. "h5n1" is
# "h5n1" in Spanish too, and language detection is not reliable enough on a
# five-word post to be worth gating them behind it.
POST_TERMS_ANY_LANG = ["h5n1", "h5n8", "h7n9", "h9n2", "h10n3", "hpai", "lpai"]

POST_TEXT_BY_LANG = {
    "en": [
        "avian influenza", "avian flu", "chicken flu", "bird flu", "poultry flu",
        "zoonotic flu", "duck flu", "farm flu",
    ],
    # ⚠️ I am confabulating on the exact currency of some of these in US farm-page
    # Spanish - they are constructed by translation, not observed in the corpus.
    # "gripe aviar" and "influenza aviar" are certain; "IAAP" (influenza aviar de
    # alta patogenicidad) is the standard regional equivalent of HPAI but I have
    # not verified it appears in this corpus. Have a Spanish speaker review the
    # list, and check block 14's per-language match rate: a Spanish match rate of
    # zero against a non-zero Spanish post count means the vocabulary is wrong,
    # not that the topic is absent.
    "es": [
        "gripe aviar", "influenza aviar", "gripe aviaria", "influenza aviaria",
        "gripe de las aves", "gripe del pollo", "gripe de pollo",
        "peste aviar", "peste aviaria", "influenza avicola", "gripe avicola",
        "iaap", "iabp",
    ],
}

# Flat union, for reporting and for continuity with the older notebooks.
PostText = POST_TERMS_ANY_LANG + [t for _terms in POST_TEXT_BY_LANG.values()
                                  for t in _terms]

# PostSingleWordText = the *recall* list sent to the API (single tokens only).
# Only reaches the API when POST_QUERY_MODE == "broad_terms".
PostSingleWordText = [
    "influenza", "flu", "gripe", "aviar", "aviaria", "avicola",
    "h5n1", "h5n8", "h7n9", "h9n2", "h10n3", "hpai", "lpai", "iaap",
]

PostStartDate = "2022-01-01"
PostEndDate   = "2026-07-31"

print(f"{len(PageDescription)} description terms, "
      f"{len(PAGE_REFINERS)} refiners")
print(f"{len(PostText)} precise concepts across "
      f"{len(POST_TEXT_BY_LANG)} language(s) + {len(POST_TERMS_ANY_LANG)} language-agnostic; "
      f"{len(PostSingleWordText)} API recall tokens")
print("page categories come from SHARD (block 02)")
print(f"configured post window: {PostStartDate} .. {PostEndDate}")
_block_end("BLOCK 01", _T0)

In [ ]:
# BLOCK 02 - Run mode, budget, and checkpoint configuration
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 02")

# ----------------------------------------------------------------------------
# IDENTITY
# ----------------------------------------------------------------------------
# ONE place. Block 04's checkpoint manifest and block 05A's parity report both
# read this. They used to hardcode it separately, which is why every manifest
# written by 20260825B/C/D claimed to be 20260825A - stale provenance that only
# shows up when someone tries to reconcile a shard months later.
#
# Collapsing three hardcodes into one constant did NOT fix that failure, it only
# moved it: 20260825F shipped with this still reading 20260825E, so that entire
# run checkpointed and reported itself as E. Block 03 now cross-checks the
# constant against the real filename where the environment exposes one. BUMP
# THIS ON EVERY COPY-FORWARD - it is the first edit, not the last.
NOTEBOOK_VERSION = "Facebook_20260901A"

# ----------------------------------------------------------------------------
# MODEL-READY FILL
# ----------------------------------------------------------------------------
# The term-restricted signals are NaN wherever no term-matching post qualified,
# which is honest but unusable downstream: the H5N1 model would drop every such
# row, and those rows are the quiet majority. Each gets a *_model companion
# filled with this value, while the measurement column stays NaN.
#
# 0.0 is defensible here and ONLY here: for max_sad_share_term it means "no
# observed distress", for min_sentiment_term "no observed negativity", and
# n_posts_term / n_term_ratio_defined / n_term_sentiment_scored recover exactly
# which rows were filled. That is what makes this different from coalescing a
# suppressed NASS value to 0, where the zero is unrecoverable and wrong.
MODEL_FILL_VALUE = 0.0

# ----------------------------------------------------------------------------
# LANGUAGE
# ----------------------------------------------------------------------------
# 20260901A. Every prior version hardcoded "lang": "en" in the post request, so
# Spanish posts were not scored badly - they were never COLLECTED. Widening this
# is a collection-scope change, and it draws on the same 500,000-per-rolling-
# 7-days budget as everything else. The ~6,000,000-post estimate on record is
# the ENGLISH-ONLY figure: re-run facebook/posts/estimate before submitting.
#
# It also means n_posts has never been the "every post these pages made"
# denominator the notebook claimed. With this set it is "every post in
# POST_LANGS".
POST_LANGS = ["en", "es"]

# Used when the API returns no lang on a post, and for any text scored without
# an explicit language. Must be one of POST_LANGS.
LANG_DEFAULT = "en"

# How POST_LANGS reaches the API.
#   "auto"         -> block 11 spends ONE synchronous call to find out, then
#                     resolves to one of the two below. The default.
#   "list"         -> one request carrying "en,es".
#   "per_language" -> one request per language. Always works, but DOUBLES the
#                     number of async jobs and therefore the submission
#                     wall-clock, which is already 4.7 h at JOB_WINDOW="year".
#
# ⚠️ I am confabulating on whether `lang` accepts a comma list at all: the docs
# describe it as a single ISO 639-1 code, and nothing in this repo has tested
# it. That is why the default probes instead of assuming. If it turns out to be
# per-language, note that job_key gains a language segment ("b003|2024-01-01|es")
# - a checkpoint written by an earlier version has no such keys, so an old
# checkpoint and a new run do not collide, they simply do not share work.
LANG_REQUEST_MODE = "auto"     # "auto" | "list" | "per_language"

# ----------------------------------------------------------------------------
# EMOJI
# ----------------------------------------------------------------------------
# Emoji contributed exactly nothing before 20260901A - the tokenizer discarded
# every non-ASCII character - so their value here has never actually been
# tested. With this on, every post is scored TWICE: `sentiment` stays text-only
# and keeps its old meaning, `sentiment_emoji` folds emoji valence in. Block 19A
# reports whether they differ; whether they HELP can only be decided outside the
# enclave, against the USDA target.
SCORE_EMOJI = True

# ----------------------------------------------------------------------------
# SHARD - which slice of the collection this run owns
# ----------------------------------------------------------------------------
# Splitting on page_category is what makes the shards recombinable: it is the
# FIRST grouping key of the aggregate, so shards are disjoint in the output and
# block 22 can concat them without re-aggregating. Any other split key would put
# the same aggregate cell in two researchers' output, and mean_sentiment cannot
# be re-derived from totals.
#
# Set this ONE name. Everything else - categories, checkpoint directory, the
# manifest tag - follows from it, so three researchers cannot drift into three
# different category lists by hand-editing.
SHARD = "smoke"      # "nick" | "max" | "waree" | "smoke"

SHARDS = {
    # Counts are the 2026-08-24 8-category baseline, and they are FLOORS: the
    # 2026-08-25 nick run reached 4,246 Farm + 410 Animal Shelter = 4,656 with
    # only two categories searched. Widening discovery raises them further.
    "nick":  ["Farm", "Animal Shelter"],                        # 4,247 pages
    "max":   ["Agricultural Service", "Agricultural Cooperative"],  # 4,523 pages
    "waree": ["Veterinarian", "Agriculture"],                   # 3,932 pages
    # Zoo is excluded from every production shard, so a smoke run can never be
    # mistaken for or contaminate real data.
    "smoke": ["Zoo"],                                           # 28 pages
}
if SHARD not in SHARDS:
    raise ValueError(f"SHARD must be one of {sorted(SHARDS)}, got {SHARD!r}")

PageCategories = SHARDS[SHARD]
IS_SMOKE = SHARD == "smoke"

# Where combined shard files are read from and written to (block 20 / block 22).
SHARD_DIR = os.path.join(os.getcwd(), "h5n1_shards")

# ----------------------------------------------------------------------------
# RUN MODE - preview smoke test vs. the full pull
# ----------------------------------------------------------------------------
# These are two DIFFERENT API mechanisms, not one endpoint with two names:
#
#   "preview" -> SYNCHRONOUS search. client.get("facebook/posts/preview").
#                10 results/page, hard-capped at 1,000 results per search.
#                Seconds. Also truncates page discovery and the date window.
#
#   "full"    -> ASYNCHRONOUS job. client.post("facebook/posts/job").
#                Up to 100,000 results per job, minutes to days, rate-limited
#                to 1 submission/minute. Resumable: re-run to keep polling.
RUN_MODE = "preview"          # "preview" | "full"

# Documented endpoint paths. There is no "facebook/posts/search".
PAGES_PREVIEW_PATH   = "facebook/pages/preview"    # sync only - no async variant
POSTS_PREVIEW_PATH   = "facebook/posts/preview"    # sync,  <= 1,000 results
POSTS_JOB_PATH       = "facebook/posts/job"        # async, <= 100,000 results
POSTS_ESTIMATE_PATH  = "facebook/posts/estimate"   # rough result-size estimate

# --- page discovery ---------------------------------------------------------
PARTITION_PAGE_SEARCH = True   # False = one unpartitioned search, capped at 1,000
PAGE_SEARCH_SLEEP_S   = 1.05   # sync limit is 60 queries/min; each paged call counts

# facebook/pages/preview returns HTTP 500 intermittently. On 2026-08-25 four Farm
# slices - poultry, chicken, egg, dairy - all failed that way and contributed ZERO
# pages, which is most of why that run came back 4.7% short of its baseline. Those
# are the four most poultry-relevant terms in the list, so losing them is a
# systematic bias, not a rounding error. Retry, and retry loudly.
PAGE_SEARCH_RETRIES     = 6    # attempts per API call, not per slice
PAGE_SEARCH_RETRY_SLEEP = 5.0  # seconds, doubling each attempt

# Re-run only the slices that errored, keeping everything already discovered.
# Without this a single 500 means redoing hours of discovery from scratch.
RETRY_FAILED_SLICES = True

# Farm/poultry and Farm/chicken failed on BOTH 2026-08-25 runs, the second time
# through four retries with backoff. Two consecutive whole-slice failures on the
# same two queries is not a transient 500 - it behaves like a query the server
# cannot complete. So when a term slice fails, ask for a strictly SMALLER result
# set rather than asking again: re-query it as "(term & refiner)" groups.
# Coverage is incomplete by construction - a page carrying the term and none of
# the refiners stays invisible - but incomplete beats the zero rows it returns
# now, and those are the two most poultry-relevant terms in the list.
PAGE_NARROW_FAILED = True

# Discovery runs across ALL production categories, not just this shard's.
# The API's `categories` filter is not an exact-primary match and every slice is
# capped at 1,000, so a Farm-primary page can be reachable only through some
# other category's slice. Restricting discovery to the shard's own categories
# therefore finds FEWER of the shard's own pages - that is the second reason the
# 2026-08-25 Farm count came in under baseline. Block 09 still filters to the
# shard, so the shards stay disjoint; only discovery is widened.
DISCOVER_ALL_CATEGORIES = True

# Third level of splitting for slices still capped after the PageDescription
# sub-split. 13 slices were still capped on the 2026-08-24 run. Left OFF because
# a wider page universe makes the post budget problem strictly worse.
PAGE_REFINE_CAPPED = False

# --- preview-mode throttles -------------------------------------------------
PREVIEW_MAX_SURFACES  = 250    # pages fed to the post pull (= one SURFACE_BATCH)
PREVIEW_MAX_API_PAGES = 25     # 25 pages x 10 results = up to 250 pages discovered
PREVIEW_WINDOW_DAYS   = 60     # trailing days of the configured post window

# --- full-mode (async job) settings ----------------------------------------
# SNAPSHOT: results retrievable for up to a year, periodically re-redacted.
# LIVE:     available for 30 days at most. SNAPSHOT is the reproducible choice
#           for a capstone that has to be re-run months later.
JOB_MODE            = "SNAPSHOT"
JOB_NAME            = "H5N1 farm-page posts"
JOB_DESCRIPTION     = ("H5N1 early-warning capstone: posts from US agriculture / "
                       "veterinary / animal-facility Pages.")
JOB_SUBMIT_SLEEP_S  = 65       # async rate limit is 1 query/min - stay under it
JOB_POLL_INTERVAL_S = 60       # seconds between status checks
JOB_POLL_BUDGET_MIN = 30       # one poll round; POLL_ROUNDS rounds per run
JOB_RESULT_CAP      = 100_000  # documented async ceiling

# THE 2026-08-25 FIX. One job per surface batch over the whole window averaged
# ~115,000 posts against a 100,000 ceiling - silent truncation on nearly every
# batch. Jobs are now (surface batch x time window).
#   None      -> one window, the old behaviour. Do not use at full scope.
#   "year"    ->   5 windows,  260 jobs, ~23,000 posts each,  4.7 h to submit
#   "quarter" ->  19 windows,  988 jobs,  ~6,100 posts each, 17.8 h to submit
#   "month"   ->  55 windows, 2860 jobs,  ~2,100 posts each, 51.6 h to submit
JOB_WINDOW = "year"

# --- query budget -----------------------------------------------------------
# 500,000 records per ROLLING 7 DAYS, shared by sync and async. The last
# estimate was ~6,000,000 posts: twelve budget-weeks. Submission stops when the
# next job's estimate would overrun what the rolling window has left, and the
# ledger is checkpointed so a later session resumes instead of restarting.
RECORD_BUDGET     = 500_000
MAX_JOBS_PER_RUN  = 60         # also caps submit wall-clock (x JOB_SUBMIT_SLEEP_S)

# Estimate result size before spending budget (full mode only).
RUN_ESTIMATE = True

# The estimate is a GATE, not a printout. While RUN_ESTIMATE is on and this
# stays False, block 12 prints the plan and STOPS without submitting anything.
CONFIRM_SUBMIT_JOBS = False

# ----------------------------------------------------------------------------
# OTHER TOGGLES
# ----------------------------------------------------------------------------
# "all_posts"   -> no q at all: n_posts is a TRUE denominator. This is the default.
# "broad_terms" -> q = PostSingleWordText (recall-filtered denominator, cheaper).
POST_QUERY_MODE = "all_posts"

RESTRICT_TO_FARM_ONLY = False  # True -> ~3,837 surfaces, ~1.8M posts
ENGAGEMENT_MIN        = 10     # reliability threshold for like_count + sad_count
MIN_CELL_POSTS        = 0      # small-cell suppression on the export table

# ----------------------------------------------------------------------------
# CHECKPOINTING - surviving a session timeout
# ----------------------------------------------------------------------------
# Written to the enclave's OWN disk. This is not an export: nothing crosses the
# Meta boundary, and no checkpoint substitutes for output review.
# Per-shard, so two shards run in one workspace cannot overwrite each other.
CKPT_DIR = os.path.join(os.getcwd(), f"h5n1_ckpt_{SHARD}")

# True  -> a re-run keeps in-memory state, else reloads it from CKPT_DIR.
# False -> ignore checkpoints and start clean, which RE-SPENDS query budget.
RESTORE_FROM_CHECKPOINT = True

# Poll rounds of JOB_POLL_BUDGET_MIN each, checkpointing after every one.
POLL_ROUNDS = 24
_block_end("BLOCK 02", _T0)

In [ ]:
# BLOCK 03 - Resolve run mode and job windows
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 03")

# Everything downstream reads the resolved names below, never the raw config.
if RUN_MODE not in ("preview", "full"):
    raise ValueError(f"RUN_MODE must be 'preview' or 'full', got {RUN_MODE!r}")
if POST_QUERY_MODE not in ("all_posts", "broad_terms"):
    raise ValueError("POST_QUERY_MODE must be 'all_posts' or 'broad_terms', "
                     f"got {POST_QUERY_MODE!r}")
if JOB_MODE not in ("SNAPSHOT", "LIVE"):
    raise ValueError(f"JOB_MODE must be 'SNAPSHOT' or 'LIVE', got {JOB_MODE!r}")
if JOB_WINDOW not in (None, "year", "quarter", "month"):
    raise ValueError(f"JOB_WINDOW must be None/'year'/'quarter'/'month', got {JOB_WINDOW!r}")
if not POST_LANGS or not all(isinstance(l, str) and l for l in POST_LANGS):
    raise ValueError(f"POST_LANGS must be a non-empty list of codes, got {POST_LANGS!r}")
if LANG_DEFAULT not in POST_LANGS:
    raise ValueError(f"LANG_DEFAULT {LANG_DEFAULT!r} must be one of POST_LANGS "
                     f"{POST_LANGS}")
if LANG_REQUEST_MODE not in ("auto", "list", "per_language"):
    raise ValueError("LANG_REQUEST_MODE must be 'auto'/'list'/'per_language', "
                     f"got {LANG_REQUEST_MODE!r}")
# Resolved for real in block 11, which probes when the mode is "auto". Until
# then assume the safe one, so nothing can submit against an untested parameter.
LANG_MODE_RESOLVED = ("per_language" if LANG_REQUEST_MODE == "auto"
                      else LANG_REQUEST_MODE)

# `until` must be strictly in the past or the API returns 400 / subcode 3790079.
_yesterday = (date.today() - timedelta(days=1)).isoformat()
if PostEndDate >= date.today().isoformat():
    print(f"PostEndDate {PostEndDate} is not in the past -> clamped to {_yesterday}")
    PostEndDate = _yesterday

# The smoke preset is applied HERE rather than trusted from block 02, so a
# leftover CONFIRM_SUBMIT_JOBS = True from a previous session cannot turn an
# environment check into a budget-spending run.
if IS_SMOKE:
    _forced = {"RUN_MODE": "preview", "RUN_ESTIMATE": False,
               "CONFIRM_SUBMIT_JOBS": False, "RESTRICT_TO_FARM_ONLY": False,
               "MIN_CELL_POSTS": 0}
    for _k, _v in _forced.items():
        if globals()[_k] != _v:
            print(f"SHARD='smoke': forcing {_k} = {_v!r} (was {globals()[_k]!r})")
        globals()[_k] = _v

# Version self-check. NOTEBOOK_VERSION is hand-set in block 02 and the
# copy-forward step forgets it - 20260825F ran and checkpointed itself as
# "Facebook_20260825E". Nothing downstream could detect that, so every manifest
# and parity report from that run carries the wrong version. Compare it against
# the real filename where the environment exposes one. WARN, never raise: the
# env var is not guaranteed to be set, and a wrong label must not kill a run
# that is holding hours of collection.
_nb_path = os.environ.get("JPY_SESSION_NAME", "")
_nb_stem = os.path.splitext(os.path.basename(_nb_path))[0] if _nb_path else ""
if _nb_stem and _nb_stem != NOTEBOOK_VERSION:
    print(f"!! NOTEBOOK_VERSION says {NOTEBOOK_VERSION!r} but this file is "
          f"{_nb_stem!r}.\n"
          f"   Fix block 02 and re-run blocks 02-03. The checkpoint manifest "
          f"and block 05A\n   read it, and a mislabelled checkpoint cannot be "
          f"reconciled later.")
elif not _nb_stem:
    print(f"note: filename not visible here, so NOTEBOOK_VERSION "
          f"{NOTEBOOK_VERSION!r} is\n      UNVERIFIED - confirm it matches the "
          f"file you actually opened.")

IS_PREVIEW = RUN_MODE == "preview"

# Discovery scope vs. shard scope. These are deliberately different: block 07
# searches DiscoveryCategories, block 09 keeps only PageCategories.
ALL_PRODUCTION_CATEGORIES = sorted({c for k, v in SHARDS.items() if k != "smoke"
                                    for c in v})
DiscoveryCategories = (ALL_PRODUCTION_CATEGORIES
                       if DISCOVER_ALL_CATEGORIES and not IS_SMOKE
                       else PageCategories)

MAX_SURFACES  = PREVIEW_MAX_SURFACES  if IS_PREVIEW else None
MAX_API_PAGES = PREVIEW_MAX_API_PAGES if IS_PREVIEW else None

RUN_END = PostEndDate
if IS_PREVIEW:
    _pv_start = date.fromisoformat(PostEndDate) - timedelta(days=PREVIEW_WINDOW_DAYS - 1)
    RUN_START = max(PostStartDate, _pv_start.isoformat())
else:
    RUN_START = PostStartDate


def _calendar_windows(start, end, grain):
    """Inclusive [since, until] pairs on calendar boundaries.

    Calendar-aligned rather than fixed-width so a re-run with a longer window
    reuses the earlier jobs' keys instead of shifting every boundary.
    """
    if grain is None:
        return [(start, end)]

    out = []
    cur = date.fromisoformat(start)
    last = date.fromisoformat(end)
    while cur <= last:
        if grain == "year":
            nxt = date(cur.year + 1, 1, 1)
        elif grain == "quarter":
            _q = (cur.month - 1) // 3
            nxt = (date(cur.year + 1, 1, 1) if _q == 3
                   else date(cur.year, 3 * (_q + 1) + 1, 1))
        else:  # month
            nxt = (date(cur.year + 1, 1, 1) if cur.month == 12
                   else date(cur.year, cur.month + 1, 1))
        stop = min(nxt - timedelta(days=1), last)
        out.append((cur.isoformat(), stop.isoformat()))
        cur = stop + timedelta(days=1)
    return out


# Preview is a single synchronous search: windowing does not apply.
JOB_WINDOWS = ([(RUN_START, RUN_END)] if IS_PREVIEW
               else _calendar_windows(RUN_START, RUN_END, JOB_WINDOW))

print(f"shard      : {SHARD}   categories {PageCategories}")
print(f"discovery  : {len(DiscoveryCategories)} categories "
      f"{'(all production - widens the page universe)' if DISCOVER_ALL_CATEGORIES and not IS_SMOKE else '(shard only)'}")
print(f"run mode   : {RUN_MODE}"
      + ("   <-- SMOKE TEST, counts are not the real dataset" if IS_PREVIEW
         else f"   <-- async jobs, {JOB_MODE} mode, minutes to days"))
# "preview" in the path below is the ENDPOINT'S name, not this run's RUN_MODE.
# There is no facebook/pages/job, so page discovery is synchronous and capped at
# 1,000 results per search whether RUN_MODE is "preview" or "full".
print(f"pages      : {PAGES_PREVIEW_PATH}"
      f"  (sync-only endpoint - no async variant exists;"
      f" 1,000/search in BOTH run modes)")
print(f"posts      : {POSTS_PREVIEW_PATH if IS_PREVIEW else POSTS_JOB_PATH}"
      f"  ({'sync, 1,000/search' if IS_PREVIEW else 'async, 100,000/job'})")
print(f"post window: {RUN_START} .. {RUN_END}"
      + (f"   (configured: {PostStartDate} .. {PostEndDate})" if IS_PREVIEW else ""))
print(f"surfaces   : {'all' if MAX_SURFACES is None else f'first {MAX_SURFACES}'}")
print(f"query mode : {POST_QUERY_MODE}")
print(f"languages  : {POST_LANGS}   default {LANG_DEFAULT}   "
      f"request mode {LANG_REQUEST_MODE}"
      + ("  (resolved in block 11)" if LANG_REQUEST_MODE == "auto" else ""))
print(f"emoji      : SCORE_EMOJI={SCORE_EMOJI}  (sentiment stays text-only; "
      f"sentiment_emoji is the twin)")
print(f"checkpoints: {CKPT_DIR}")
if IS_SMOKE:
    print("\nSMOKE RUN - environment check only. Not a dataset, and it spends no\n"
          "result budget. Run block 05A and send Nick the parity report.")

if not IS_PREVIEW:
    print(f"job windows: {len(JOB_WINDOWS)} x {JOB_WINDOW}"
          f"   {JOB_WINDOWS[0][0]} .. {JOB_WINDOWS[-1][1]}")
    if JOB_WINDOW is None:
        print("  !! JOB_WINDOW=None at full scope averaged ~115,000 posts/job on the "
              "last run,\n     against a 100,000 ceiling. Expect silent truncation.")
_block_end("BLOCK 03", _T0)

<!-- Last changed: Facebook_20260825A -->
## 0 — Checkpoint machinery

An async pull outlives the session that started it. `JOB_MODE = "SNAPSHOT"` means Meta
holds the results for roughly a year, so the thing that must survive a session timeout
is the **job ID**, not the rows: lose `POST_JOBS` and you resubmit, re-spending budget
you have already paid. Lose the harvested rows and you re-harvest for free.

* `_ckpt_write` / `_ckpt_read` — one object to one file. Raw API rows go to gzipped
  JSONL so no `json_normalize` assumption is baked into the file; DataFrames go to
  gzipped pickle; small dicts and lists go to plain JSON.
* `_ckpt_init(NAME, default)` — in-memory state, else the checkpoint, else a fresh
  default. This is what makes a re-run cheap and a resumed session possible.
* `checkpoint(tag)` — persist every piece of run state that currently exists.

`_ckpt_write` never raises. A checkpoint that kills a twelve-hour run is worse than no
checkpoint at all.

**Resuming a lost session:** run blocks 00–04 with `RESTORE_FROM_CHECKPOINT = True`,
then run down the notebook. Page discovery skips, submitted jobs are not resubmitted,
polling reattaches, and the budget ledger carries its rolling window forward.

> ⚠️ A checkpoint is **not** an export. These files never leave the enclave, and writing
> one is not a substitute for Meta's output review. Same boundary as
> `job.write_data_to_file()`: a within-enclave convenience, nothing more.


In [ ]:
# BLOCK 04 - Checkpoint machinery
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 04")

os.makedirs(CKPT_DIR, exist_ok=True)

# Persisted as gzipped JSONL: raw API payloads, kept in their original shape so a
# reload is not hostage to how json_normalize behaved that day.
_CKPT_ROWSETS = {"page_rows", "post_rows"}

# Persist order: cheapest and most irreplaceable first. POST_JOBS is a few hundred
# bytes and is the only handle on SNAPSHOT results Meta holds for a year; the
# DataFrames at the end are all recomputable from POST_ROWS.
_CKPT_STATE = ("POST_JOBS", "POST_ESTIMATES", "BUDGET_LEDGER", "PAGE_SLICES",
               "POST_BATCH", "PAGE_ROWS", "POST_ROWS",
               "SourcePages", "CleanUSPages", "SourcePosts",
               "AggByCatStateZipDate", "ExportAgg", "ExportAggState")

# Names that changed between notebook versions. A rename must not orphan a
# checkpoint written by an older run - that would quietly discard hours of
# collection. SourcePosts was called FarmPosts through 20260825C.
_CKPT_ALIASES = {"sourceposts": ("farmposts",)}


def _ckpt_write(name, obj):
    """Best-effort persist. NEVER raises - a checkpoint must not kill a long run."""
    try:
        if name in _CKPT_ROWSETS:
            p = os.path.join(CKPT_DIR, name + ".jsonl.gz")
            with gzip.open(p, "wt", encoding="utf-8") as fh:
                for r in obj:
                    fh.write(json.dumps(r, default=str) + "\n")
        elif isinstance(obj, pd.DataFrame):
            p = os.path.join(CKPT_DIR, name + ".pkl.gz")
            obj.to_pickle(p, compression="gzip")
        else:
            p = os.path.join(CKPT_DIR, name + ".json")
            with open(p, "w", encoding="utf-8") as fh:
                json.dump(obj, fh, default=str, indent=2)
        return p
    except Exception as e:
        print(f"  !! checkpoint FAILED for {name}: {type(e).__name__}: {e}")
        return None


def _ckpt_read(name):
    """Load one checkpointed object, or None if it is absent or unreadable.

    Falls back to any legacy name in _CKPT_ALIASES, so a checkpoint written
    before a rename still loads.
    """
    for candidate in (name,) + _CKPT_ALIASES.get(name, ()):
        for ext in (".jsonl.gz", ".pkl.gz", ".json"):
            p = os.path.join(CKPT_DIR, candidate + ext)
            if not os.path.exists(p):
                continue
            try:
                if ext == ".jsonl.gz":
                    with gzip.open(p, "rt", encoding="utf-8") as fh:
                        obj = [json.loads(ln) for ln in fh if ln.strip()]
                elif ext == ".pkl.gz":
                    obj = pd.read_pickle(p, compression="gzip")
                else:
                    with open(p, encoding="utf-8") as fh:
                        obj = json.load(fh)
                if candidate != name:
                    print(f"  note: loaded legacy checkpoint '{candidate}{ext}' "
                          f"for '{name}'")
                return obj
            except Exception as e:
                print(f"  !! checkpoint UNREADABLE {candidate}{ext}: "
                      f"{type(e).__name__}: {e}")
    return None


def _ckpt_init(varname, default):
    """In-memory state, else the checkpoint, else a fresh default.

    Re-running a cell must not re-fetch what has already been paid for, and a
    fresh kernel must be able to pick up where the dead one stopped.

    Job keys are STRINGS ("b003|2024-01-01") precisely so this survives a JSON
    round-trip - JSON has no integer keys, and the old int-keyed POST_JOBS had
    to be coerced back by hand on every restore.
    """
    if not RESTORE_FROM_CHECKPOINT:
        return default
    live = globals().get(varname)
    if live is not None and len(live):
        print(f"  {varname:<14} kept in memory      ({len(live):,})")
        return live
    disk = _ckpt_read(varname.lower())
    if disk is not None and len(disk):
        print(f"  {varname:<14} restored from disk  ({len(disk):,})")
        return disk
    return default


def _ckpt_manifest(tag):
    """Provenance. A checkpoint you cannot attribute to a run config is noise."""
    return {
        "tag": tag,
        "written_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "shard": globals().get("SHARD"),
        "notebook": NOTEBOOK_VERSION,
        "run_mode": globals().get("RUN_MODE"),
        "job_mode": globals().get("JOB_MODE"),
        "job_window": globals().get("JOB_WINDOW"),
        "run_start": str(globals().get("RUN_START")),
        "run_end": str(globals().get("RUN_END")),
        "query_mode": globals().get("POST_QUERY_MODE"),
        "page_categories": globals().get("PageCategories"),
        "sentiment_engine": globals().get("SENTIMENT_ENGINE"),
        "engines_by_lang": globals().get("SENTIMENT_ENGINE_BY_LANG"),
        "lex_fingerprint": globals().get("LEX_FINGERPRINT"),
        "post_langs": globals().get("POST_LANGS"),
        "lang_mode": globals().get("LANG_MODE_RESOLVED"),
        "score_emoji": globals().get("SCORE_EMOJI"),
        "farm_only": globals().get("RESTRICT_TO_FARM_ONLY"),
        "engagement_min": globals().get("ENGAGEMENT_MIN"),
        "n_surfaces": len(globals().get("SourcePages", [])),
        "submit_gated": globals().get("SUBMIT_GATED"),
        "all_jobs_done": globals().get("ALL_JOBS_DONE"),
    }


def checkpoint(tag="", only=None):
    """Persist whatever run state exists right now. Safe to call at any point.

    `only` restricts the write to a subset of _CKPT_STATE. Block 07 uses it to
    checkpoint after every discovery category without re-serialising POST_ROWS,
    which can be millions of rows by the time someone re-runs discovery to widen
    its scope. The manifest is still written every time - a checkpoint you
    cannot attribute to a run config is noise whether it is partial or not.
    """
    _ckpt_write("run_manifest", _ckpt_manifest(tag))
    total = 0
    _names = (_CKPT_STATE if only is None
              else [n for n in _CKPT_STATE if n in set(only)])
    for name in _names:
        obj = globals().get(name)
        if obj is None or not len(obj):
            continue
        p = _ckpt_write(name.lower(), obj)
        if p:
            total += os.path.getsize(p)
            print(f"  {name:<22} {len(obj):>9,}  ->  {os.path.basename(p)}")
    print(f"  checkpoint '{tag}' -> {CKPT_DIR}  ({total / 1e6:.1f} MB)")


print(f"shard          : {globals().get('SHARD')}")
print(f"checkpoint dir : {CKPT_DIR}")
print(f"restore        : {RESTORE_FROM_CHECKPOINT}")
_existing = sorted(os.listdir(CKPT_DIR))
print(f"on disk        : {_existing if _existing else 'nothing yet'}")
_block_end("BLOCK 04", _T0)

<!-- Last changed: Facebook_20260901A -->
## 1 — Sentiment engine

The enclave has no outbound network, so a `transformers` pipeline that pulls weights
from the Hub will fail. The chain below degrades gracefully, **for English**:

1. `vaderSentiment` (pip package, lexicon ships inside the wheel) — preferred.
2. `nltk` VADER — only if `vader_lexicon` is already present on disk.
3. **Inline lexicon** — defined in the block, always works, zero dependencies.

All three return a compound score in `[-1, +1]` using VADER's normalization, so
`mean_sentiment` and the sentiment extremes are comparable whichever engine wins. Record
`SENTIMENT_ENGINE` in the methods write-up — a run scored by fallback #3 is not
interchangeable with one scored by #1. Block 05A now also prints an **NLP capability
probe**, which is what settles empirically whether anything richer was ever installed
here rather than leaving it an assumption.

### Languages

VADER is English-only, so every other language runs on a per-language inline lexicon:
`_LEX_BY_LANG` keyed by the post's own `lang` field. Adding a language is a **data**
edit, not a code edit.

Translation is not an option and should not be attempted. The enclave has no outbound
network, so there is no API to call and no MarianMT/NLLB/Argos weights to download —
scoring in-language is the only route, and it is sufficient here because nothing but
numbers ever leaves. Two consequences worth stating plainly:

> ⚠️ **The lexicons are not calibrated against each other.** A Spanish `−0.6` does not
> provably mean what an English `−0.6` means. Block 05A scores parallel English/Spanish
> sentences and reports the per-pair delta — that number belongs in the methods
> write-up. Block 17 therefore reports `mean_sentiment_en` and `mean_sentiment_es`
> separately; the pooled `mean_sentiment` is kept for continuity but mixes two
> instruments.

> ⚠️ **A language with no lexicon scores `NaN`, never `0.0`.** A zero would read as
> "neutral" and be indistinguishable from a genuinely neutral post — the same
> unrecoverable fill this project forbids for suppressed NASS values. The
> `n_sentiment_scored_*` counts recover exactly which posts those were.

### Emoji

Earlier versions described emoji as "deliberately not scored." That was not quite true:
the tokenizer was `[a-z0-9']+`, which discarded every non-ASCII character, so emoji
contributed **exactly zero** and the question was never actually tested. `20260901A`
tests it.

Every post is now scored twice — `sentiment` (text-only, emoji stripped, unchanged
meaning) and `sentiment_emoji` (emoji valence folded in). Both reach the export. The
emoji contribution always comes from **our** `_EMOJI_LEX`, never from VADER's built-in
emoji table, and the text-only path strips emoji in *every* engine branch: otherwise an
environment that resolved to `vaderSentiment` would silently include emoji in its
"text-only" score and the contrast would measure nothing.

Folding in is exact rather than ad hoc. VADER's compound is `x / sqrt(x² + 15)` and the
inline engine uses the same form, so the text score is de-normalized back to a raw
valence sum, the emoji sum is added, and the result is re-normalized — identical
arithmetic whichever engine produced the text score.

Block 19A reports whether the two variants differ. Whether emoji *help* cannot be
decided in here: that needs the USDA target, which lives outside the enclave.

In [ ]:
# BLOCK 05 - Sentiment engine (multilingual + emoji)
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 05")

# ---------------------------------------------------------------------------
# TOKENIZER
# ---------------------------------------------------------------------------
# 20260901A - the old tokenizer was r"[a-z0-9']+", which silently deleted every
# non-ASCII character. That made two things impossible at once: emoji scored
# zero because they were never tokens, and Spanish scored wrong because "murio"
# and "murio'" are not what "murio" with an accent tokenizes to - accented words
# simply vanished. Unicode word characters, plus accent folding on BOTH the text
# and the lexicon keys, because farm-page Spanish is written unaccented at least
# as often as not.
_TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", re.UNICODE)


def _fold(s):
    """Casefold and strip combining accents. 'Murio' == 'murio' == 'murio'."""
    return "".join(c for c in unicodedata.normalize("NFKD", str(s).casefold())
                   if not unicodedata.combining(c))


def _fold_keys(d):
    """Accent-fold a lexicon's keys, keeping the larger magnitude on collision."""
    out = {}
    for k, v in d.items():
        fk = _fold(k)
        if fk not in out or abs(v) > abs(out[fk]):
            out[fk] = v
    return out


# ---------------------------------------------------------------------------
# EMOJI
# ---------------------------------------------------------------------------
# Matched as GRAPHEME CLUSTERS, not codepoints: a ZWJ sequence like the
# farmer-with-skin-tone emoji is one symbol carrying one meaning, and splitting
# it into "man" + "ear of rice" would both inflate n_emoji and score nonsense.
# The base class is deliberately conservative - (c), (R), (TM) and the text
# arrows are excluded, because they appear constantly in ordinary prose and
# counting them as emoji would inflate every emoji statistic on brand pages.
_EMOJI_BASE = (
    "\U0001F300-\U0001F5FF"      # symbols & pictographs
    "\U0001F600-\U0001F64F"      # emoticons
    "\U0001F680-\U0001F6FF"      # transport & map
    "\U0001F900-\U0001F9FF"      # supplemental symbols & pictographs
    "\U0001FA70-\U0001FAFF"      # symbols & pictographs extended-A
    "\U0001F7E0-\U0001F7EB"      # coloured circles and squares
    "☀-➿"              # misc symbols + dingbats
    "⬀-⯿"              # misc symbols and arrows
    "↔-↪"              # the arrows that really are emoji
    "⌚⌛⏩-⏺▪-◾ℹ"
)
_EMOJI_MOD = "️︎⃣\U0001F3FB-\U0001F3FF"   # VS15/16, keycap, skin tone
_EMOJI_FLAG = "\U0001F1E6-\U0001F1FF"        # regional indicators
_EMOJI_RE = re.compile(
    # Flags FIRST. A flag is a PAIR of regional indicators with no
    # joiner between them, so the general cluster pattern below would
    # split one country into two "emoji" and double-count it.
    "[" + _EMOJI_FLAG + "][" + _EMOJI_FLAG + "]"
    "|[" + _EMOJI_BASE + "][" + _EMOJI_MOD + "]*"
    "(?:‍[" + _EMOJI_BASE + "][" + _EMOJI_MOD + "]*)*"
)

# Same [-4, +4] scale as the word lexicons, so an emoji and a word contribute
# commensurably. Domain-tilted, and two judgement calls are worth stating:
#
#   - The bird/farm pictographs are near ZERO. They are topic markers, not
#     affect: a page posts a chicken emoji on a good day and a bad one alike.
#     Giving them valence would make every poultry page look emotional.
#   - Prayer hands and red heart are ALSO near zero, deliberately. On a farm
#     page they read as condolence at least as often as affection, and that
#     ambiguity is exactly how emoji turn into noise. Scoring them strongly
#     positive is the single most likely way to manufacture a false signal.
#
# Everything here is a hypothesis to be tested by block 19A and, ultimately,
# outside the enclave - not an assumption to build on.
_EMOJI_LEX = {
    # --- distress / loss ---
    "\U0001F622": -2.4,   # crying face
    "\U0001F62D": -3.0,   # loudly crying face
    "\U0001F494": -3.0,   # broken heart
    "\U0001F614": -1.8,   # pensive face
    "\U0001F61E": -2.0,   # disappointed face
    "\U0001F625": -2.2,   # sad but relieved face
    "\U0001F630": -2.2,   # anxious face with sweat
    "\U0001F628": -2.4,   # fearful face
    "\U0001F631": -2.6,   # face screaming in fear
    "\U0001F62B": -2.2,   # tired face
    "\U0001F621": -2.8,   # pouting face
    "\U0001F620": -2.4,   # angry face
    "\U0001F92C": -3.0,   # face with symbols on mouth
    "\U0001F912": -2.0,   # face with thermometer
    "\U0001F915": -1.8,   # face with head bandage
    "\U0001F927": -1.4,   # sneezing face
    "\U0001F637": -1.6,   # face with medical mask
    "\U0001F92E": -2.0,   # face vomiting
    "☠": -3.2,       # skull and crossbones
    "\U0001F480": -3.0,   # skull
    "⚠": -2.0,       # warning
    "\U0001F6D1": -2.0,   # stop sign
    "\U0001F534": -1.0,   # red circle
    "\U0001F198": -2.6,   # SOS
    "\U0001F62C": -1.0,   # grimacing face
    "\U0001F644": -1.0,   # face with rolling eyes
    "\U0001F613": -1.6,   # downcast face with sweat
    # --- positive ---
    "\U0001F600": 2.2, "\U0001F603": 2.4, "\U0001F604": 2.6, "\U0001F601": 2.4,
    "\U0001F606": 2.6, "\U0001F60A": 2.4, "\U0001F642": 1.6, "\U0001F60D": 3.0,
    "\U0001F970": 2.6, "\U0001F929": 2.8, "\U0001F973": 2.8, "\U0001F60E": 2.0,
    "\U0001F44D": 2.0,    # thumbs up
    "\U0001F44F": 2.0,    # clapping hands
    "\U0001F4AA": 1.8,    # flexed biceps
    "❤": 0.4,        # red heart - see note above
    "\U0001F64F": 0.2,    # folded hands - see note above
    "✨": 1.8, "\U0001F389": 2.6, "\U0001F38A": 2.4, "\U0001F31E": 1.6,
    "\U0001F31F": 1.8, "✅": 1.4, "\U0001F49A": 1.4, "\U0001F60B": 2.2,
    "\U0001F44C": 1.8,
    "\U0001F44E": -2.0,   # thumbs down
    # --- topic markers, deliberately ~0 ---
    "\U0001F414": 0.0,    # chicken
    "\U0001F413": 0.0,    # rooster
    "\U0001F425": 0.0,    # front-facing baby chick
    "\U0001F423": 0.0,    # hatching chick
    "\U0001F424": 0.0,    # baby chick
    "\U0001F95A": 0.0,    # egg
    "\U0001F373": 0.0,    # cooking
    "\U0001F983": 0.0,    # turkey
    "\U0001F986": 0.0,    # duck
    "\U0001F9A2": 0.0,    # swan
    "\U0001F426": 0.0,    # bird
    "\U0001F995": 0.0, "\U0001F404": 0.0, "\U0001F416": 0.0, "\U0001F410": 0.0,
    "\U0001F411": 0.0, "\U0001F69C": 0.0, "\U0001F33E": 0.0, "\U0001F33D": 0.0,
    "\U0001F3E1": 0.0, "\U0001F1FA\U0001F1F8": 0.0,
}


def extract_emoji(text):
    """Every emoji grapheme cluster in `text`, in order, duplicates kept."""
    return _EMOJI_RE.findall(str(text)) if text else []


def strip_emoji(text):
    """`text` with every emoji cluster replaced by a space.

    Required for the text-only score in EVERY engine branch, not just ours:
    VADER ships its own emoji lexicon, so on an environment that resolves to
    vaderSentiment the "text-only" score would silently include emoji and the
    emoji contrast would measure nothing at all. The notebook must not be
    correct only by accident of which fallback fired.
    """
    return _EMOJI_RE.sub(" ", str(text)) if text else ""


def _emoji_valences(text):
    """(n_emoji, n_scored, sum_valence). A cluster falls back to its base char,
    so a skin-toned thumbs-up scores as a thumbs-up."""
    found = extract_emoji(text)
    if not found:
        return 0, 0, 0.0
    total, scored = 0.0, 0
    for e in found:
        v = _EMOJI_LEX.get(e)
        if v is None:
            base = "".join(c for c in e if c not in "️︎‍"
                           and not ("\U0001F3FB" <= c <= "\U0001F3FF"))
            v = _EMOJI_LEX.get(base)
            if v is None and base:
                v = _EMOJI_LEX.get(base[0])
        if v is not None:
            total += v
            scored += 1
    return len(found), scored, total


# ---------------------------------------------------------------------------
# WORD LEXICONS, BY LANGUAGE
# ---------------------------------------------------------------------------
# One dict per language on a shared [-4, +4] scale. Adding a language is a DATA
# edit, not a code edit - which is as close to "accommodates any language" as
# this environment allows. Translation is not on the table: the enclave has no
# outbound network, so there is no API to call and no MarianMT/NLLB/Argos
# weights to download. Scoring in-language is the only route, and it is
# sufficient here because nothing but numbers ever leaves.
#
# ⚠️ CALIBRATION. These two lexicons are NOT calibrated against each other. A
# Spanish -0.6 does not provably mean what an English -0.6 means. Block 05A
# scores parallel probe sentences and reports the gap; report per-language means
# separately (block 17 does) and say this in the methods write-up rather than
# quietly pooling them.
_LEX_EN = {
    # --- outbreak / loss (domain-specific negatives) ---
    "cull": -2.6, "culled": -2.8, "culling": -2.8, "depopulate": -3.0,
    "depopulated": -3.0, "depopulation": -3.0, "euthanize": -2.8,
    "euthanized": -2.8, "euthanasia": -2.8, "outbreak": -2.6, "outbreaks": -2.6,
    "infected": -2.4, "infection": -2.3, "infectious": -1.8, "contagious": -1.8,
    "quarantine": -2.0, "quarantined": -2.0, "lockdown": -1.8, "biosecurity": -0.4,
    "dead": -2.9, "death": -2.9, "deaths": -2.9, "died": -2.8, "dying": -2.8,
    "mortality": -2.5, "carcass": -2.2, "carcasses": -2.2, "sick": -2.2,
    "illness": -2.0, "disease": -2.0, "diseased": -2.4, "virus": -1.6,
    "viral": -1.0, "epidemic": -2.8, "pandemic": -2.8, "spread": -1.0,
    "spreading": -1.2, "positive": -0.5, "confirmed": -0.6, "detected": -0.8,
    "detection": -0.8, "loss": -2.4, "losses": -2.4,
    "lost": -2.2, "devastating": -3.4, "devastated": -3.3, "tragedy": -3.2,
    "heartbreaking": -3.3, "heartbroken": -3.3, "crisis": -2.8, "emergency": -2.4,
    "shortage": -2.0, "recall": -1.6, "banned": -1.8, "ban": -1.6,
    "restriction": -1.4, "restrictions": -1.4, "closed": -1.4, "shut": -1.4,
    "struggle": -2.2, "struggling": -2.2, "suffering": -2.8, "risk": -1.4,
    "danger": -2.4, "dangerous": -2.6, "threat": -2.4, "fear": -2.4,
    "afraid": -2.2, "scared": -2.4, "worried": -2.2, "worry": -2.0,
    "concern": -1.4, "concerned": -1.6, "panic": -2.8, "anxious": -2.0,
    # --- general negatives ---
    "bad": -2.5, "awful": -3.0, "terrible": -3.1, "horrible": -3.2,
    "worst": -3.4, "sad": -2.6, "angry": -2.6, "hate": -3.0, "hard": -1.2,
    "difficult": -1.6, "problem": -1.8, "problems": -1.8, "fail": -2.4,
    "failed": -2.4, "failure": -2.6, "damage": -2.2, "destroy": -3.0,
    "destroyed": -3.0, "sorry": -1.2, "unfortunately": -1.8, "never": -1.0,
    # --- general positives ---
    "good": 1.9, "great": 3.1, "excellent": 3.2, "amazing": 3.2,
    "wonderful": 3.1, "fantastic": 3.2, "best": 3.2, "love": 3.2,
    "loved": 2.9, "happy": 2.7, "happiness": 2.8, "joy": 2.8, "excited": 2.4,
    "exciting": 2.4, "beautiful": 2.8, "proud": 2.4, "grateful": 2.6,
    "thank": 2.2, "thanks": 2.2, "thankful": 2.5, "blessed": 2.6,
    "welcome": 1.8, "enjoy": 2.2, "enjoyed": 2.2, "fun": 2.3, "nice": 1.8,
    "safe": 1.6, "safety": 1.0, "healthy": 2.2, "health": 0.8, "recovered": 2.2,
    "recovery": 1.8, "improving": 1.8, "improved": 2.0, "success": 2.7,
    "successful": 2.7, "support": 1.8, "supporting": 1.8, "help": 1.6,
    "helping": 1.6, "hope": 1.9, "hopeful": 2.2, "fresh": 1.6, "delicious": 2.6,
    "open": 1.0, "available": 0.8, "new": 0.6, "clean": 1.4, "strong": 1.8,
    "protect": 1.2, "protected": 1.6, "prevent": 0.8, "vaccine": 0.6,
    "vaccinated": 1.0,
}

# Spanish. Valences mirror the English entry they translate wherever a clean
# one-to-one exists, which is the only calibration discipline available without
# a labelled Spanish corpus - and it is weaker than it looks, so see the
# CALIBRATION note above. Keys are accent-folded at build time, so both
# "murio" and the accented spelling hit the same entry.
_LEX_ES = {
    # --- brote / perdida ---
    "sacrificio": -2.6, "sacrificar": -2.6, "sacrificados": -2.8,
    "sacrificadas": -2.8, "despoblacion": -3.0, "despoblar": -3.0,
    "eutanasia": -2.8, "brote": -2.6, "brotes": -2.6, "infectado": -2.4,
    "infectados": -2.4, "infectada": -2.4, "infectadas": -2.4,
    "infeccion": -2.3, "infecciones": -2.3, "contagio": -2.0,
    "contagios": -2.0, "contagioso": -1.8, "contagiosa": -1.8,
    "cuarentena": -2.0, "confinamiento": -1.8, "bioseguridad": -0.4,
    "muerto": -2.9, "muertos": -2.9, "muerta": -2.9, "muertas": -2.9,
    "muerte": -2.9, "muertes": -2.9, "murio": -2.8, "murieron": -2.8,
    "muriendo": -2.8, "mortalidad": -2.5, "cadaver": -2.2, "cadaveres": -2.2,
    "enfermo": -2.2, "enfermos": -2.2, "enferma": -2.2, "enfermas": -2.2,
    "enfermedad": -2.0, "enfermedades": -2.0, "virus": -1.6, "viral": -1.0,
    "epidemia": -2.8, "pandemia": -2.8, "propagacion": -1.2,
    "propagando": -1.2, "positivo": -0.5, "positivos": -0.5,
    "confirmado": -0.6, "confirmados": -0.6, "detectado": -0.8,
    "detectados": -0.8, "deteccion": -0.8, "perdida": -2.4, "perdidas": -2.4,
    "perdio": -2.2, "perdimos": -2.4, "devastador": -3.4, "devastadora": -3.4,
    "devastado": -3.3, "tragedia": -3.2, "desgarrador": -3.3, "crisis": -2.8,
    "emergencia": -2.4, "escasez": -2.0, "prohibido": -1.8,
    "prohibicion": -1.6, "restriccion": -1.4, "restricciones": -1.4,
    "cerrado": -1.4, "cerrada": -1.4, "lucha": -2.2, "luchando": -2.2,
    "sufrimiento": -2.8, "sufriendo": -2.8, "riesgo": -1.4, "peligro": -2.4,
    "peligroso": -2.6, "peligrosa": -2.6, "amenaza": -2.4, "miedo": -2.4,
    "temor": -2.4, "asustado": -2.4, "asustada": -2.4, "preocupado": -2.2,
    "preocupada": -2.2, "preocupacion": -1.4, "panico": -2.8,
    "ansiedad": -2.0, "ansioso": -2.0,
    # --- negativos generales ---
    "malo": -2.5, "mala": -2.5, "mal": -2.3, "horrible": -3.2,
    "terrible": -3.1, "espantoso": -3.0, "peor": -3.4, "triste": -2.6,
    "tristeza": -2.7, "enojado": -2.6, "enfadado": -2.6, "odio": -3.0,
    "dificil": -1.6, "duro": -1.2, "problema": -1.8, "problemas": -1.8,
    "fallo": -2.4, "fracaso": -2.6, "dano": -2.2, "danos": -2.2,
    "destruido": -3.0, "destruccion": -3.0, "lamentablemente": -1.8,
    "desafortunadamente": -1.8, "lastima": -1.8, "nunca": -1.0,
    # --- positivos generales ---
    "bueno": 1.9, "buena": 1.9, "buenos": 1.9, "buenas": 1.9,
    "genial": 3.1, "excelente": 3.2, "increible": 3.2, "maravilloso": 3.1,
    "fantastico": 3.2, "mejor": 3.2, "amor": 3.2, "encanta": 3.0,
    "feliz": 2.7, "felicidad": 2.8, "alegria": 2.8, "emocionado": 2.4,
    "emocionante": 2.4, "hermoso": 2.8, "hermosa": 2.8, "bonito": 2.4,
    "bonita": 2.4, "lindo": 2.4, "linda": 2.4, "orgulloso": 2.4,
    "orgullosa": 2.4, "agradecido": 2.6, "agradecida": 2.6, "gracias": 2.2,
    "bendecido": 2.6, "bienvenido": 1.8, "bienvenidos": 1.8, "disfrutar": 2.2,
    "disfruten": 2.2, "divertido": 2.3, "seguro": 1.6, "seguridad": 1.0,
    "saludable": 2.2, "salud": 0.8, "recuperado": 2.2, "recuperacion": 1.8,
    "mejorando": 1.8, "mejorado": 2.0, "exito": 2.7, "exitoso": 2.7,
    "apoyo": 1.8, "apoyando": 1.8, "ayuda": 1.6, "ayudando": 1.6,
    "esperanza": 1.9, "fresco": 1.6, "fresca": 1.6, "delicioso": 2.6,
    "deliciosa": 2.6, "abierto": 1.0, "disponible": 0.8, "nuevo": 0.6,
    "nueva": 0.6, "limpio": 1.4, "fuerte": 1.8, "proteger": 1.2,
    "protegido": 1.6, "prevenir": 0.8, "vacuna": 0.6, "vacunado": 1.0,
}

_LEX_BY_LANG = {"en": _fold_keys(_LEX_EN), "es": _fold_keys(_LEX_ES)}

_NEGATIONS_BY_LANG = {
    "en": {"not", "no", "never", "none", "nobody", "nothing", "neither",
           "nowhere", "cannot", "cant", "wont", "dont", "doesnt", "didnt",
           "isnt", "arent", "wasnt", "werent", "hasnt", "havent", "hadnt",
           "aint", "without"},
    "es": {"no", "nunca", "jamas", "nadie", "nada", "ninguno", "ninguna",
           "ni", "tampoco", "sin"},
}
_BOOSTERS_BY_LANG = {
    "en": {"very": 1.45, "really": 1.4, "extremely": 1.6, "so": 1.3,
           "absolutely": 1.5, "completely": 1.5, "totally": 1.45,
           "incredibly": 1.55, "super": 1.4, "highly": 1.4, "massively": 1.5,
           "entire": 1.3, "whole": 1.25, "slightly": 0.7, "somewhat": 0.75,
           "kinda": 0.7, "barely": 0.6, "hardly": 0.6, "little": 0.75,
           "sort": 0.8, "marginally": 0.7},
    "es": {"muy": 1.45, "realmente": 1.4, "extremadamente": 1.6, "tan": 1.3,
           "absolutamente": 1.5, "completamente": 1.5, "totalmente": 1.45,
           "increiblemente": 1.55, "super": 1.4, "sumamente": 1.5,
           "bastante": 1.2, "demasiado": 1.4, "todo": 1.25, "entero": 1.3,
           "ligeramente": 0.7, "algo": 0.75, "apenas": 0.6, "poco": 0.75,
           "casi": 0.8},
}
_NEGATIONS_BY_LANG = {k: {_fold(x) for x in v}
                      for k, v in _NEGATIONS_BY_LANG.items()}
_BOOSTERS_BY_LANG = {k: _fold_keys(v) for k, v in _BOOSTERS_BY_LANG.items()}

SUPPORTED_LANGS = sorted(_LEX_BY_LANG)

# ---------------------------------------------------------------------------
# NORMALIZATION
# ---------------------------------------------------------------------------
# VADER's compound is x / sqrt(x^2 + alpha) with alpha = 15, and the inline
# engine uses the same form. Because it is invertible, the emoji contribution
# can be folded into a text score computed by EITHER engine, identically:
# de-normalize back to the raw valence sum, add the emoji sum, re-normalize.
# That is what keeps the emoji experiment comparable across environments
# instead of using our lexicon here and VADER's built-in emoji table there.
_ALPHA = 15.0


def _normalize(x):
    return float(np.clip(x / math.sqrt(x * x + _ALPHA), -1.0, 1.0))


def _denormalize(c):
    c = float(np.clip(c, -0.999999, 0.999999))
    return c * math.sqrt(_ALPHA / (1.0 - c * c))


def _inline_raw(text, lang):
    """Raw (un-normalized) valence sum for `text` in `lang`; None if unscorable."""
    lex = _LEX_BY_LANG.get(lang)
    if lex is None or not text:
        return None
    toks = [_fold(t) for t in _TOKEN_RE.findall(str(text))]
    if not toks:
        return None
    negations = _NEGATIONS_BY_LANG.get(lang, set())
    boosters = _BOOSTERS_BY_LANG.get(lang, {})
    total = 0.0
    for i, tok in enumerate(toks):
        val = lex.get(tok)
        if val is None:
            continue
        if i > 0 and toks[i - 1] in boosters:      # intensifier immediately before
            val *= boosters[toks[i - 1]]
        if any(t in negations for t in toks[max(0, i - 3):i]):   # negation within 3
            val *= -0.74
        total += val
    return total


# ---------------------------------------------------------------------------
# ENGINE SELECTION, PER LANGUAGE
# ---------------------------------------------------------------------------
# VADER, where present, is English-only - so it is selected for English and the
# inline lexicon still carries every other language. SENTIMENT_ENGINE keeps its
# old meaning (the ENGLISH engine) because the checkpoint manifest and block
# 22's config lock both key off it.
_VADER = None
_VADER_NAME = None
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as _SIA
    _VADER = _SIA()
    _VADER.polarity_scores("a good test")          # prove it runs
    _VADER_NAME = "vaderSentiment"
except Exception:
    try:
        from nltk.sentiment.vader import SentimentIntensityAnalyzer as _SIA
        _VADER = _SIA()
        _VADER.polarity_scores("a good test")      # raises if lexicon absent
        _VADER_NAME = "nltk-vader"
    except Exception:
        _VADER = None


def _raw_score(text, lang):
    """Raw valence sum for one post, or None when the language is unsupported."""
    if lang == "en" and _VADER is not None:
        if not text:
            return None
        return _denormalize(_VADER.polarity_scores(str(text))["compound"])
    return _inline_raw(text, lang)


SENTIMENT_ENGINE_BY_LANG = {
    lang: (_VADER_NAME if lang == "en" and _VADER is not None else "inline-lexicon")
    for lang in SUPPORTED_LANGS
}
SENTIMENT_ENGINE = SENTIMENT_ENGINE_BY_LANG.get("en", "inline-lexicon")

# One fingerprint over every lexicon in play. Block 22 hard-fails on it: if two
# researchers run different Spanish lexicons, every number still computes, every
# sanity check still passes, and mean_sentiment_es is quietly incomparable -
# exactly the failure the SENTIMENT_ENGINE lock was written to catch.
LEX_FINGERPRINT = hashlib.sha256(json.dumps(
    {"words": {k: sorted(v.items()) for k, v in _LEX_BY_LANG.items()},
     "emoji": sorted(_EMOJI_LEX.items()),
     "neg": {k: sorted(v) for k, v in _NEGATIONS_BY_LANG.items()},
     "boost": {k: sorted(v.items()) for k, v in _BOOSTERS_BY_LANG.items()},
     "alpha": _ALPHA},
    sort_keys=True).encode("utf-8")).hexdigest()[:12]


# ---------------------------------------------------------------------------
# PUBLIC API
# ---------------------------------------------------------------------------
def score_one(text, lang=None, with_emoji=False):
    """Compound sentiment on [-1, 1] for one post. NaN when unscorable.

    NaN, never 0.0, for a language with no lexicon. A zero would read as
    "neutral" and be indistinguishable from a genuinely neutral post - the same
    unrecoverable fill this project forbids for suppressed NASS values. The
    n_sentiment_scored_* counts recover exactly which posts these were.
    """
    lang = (LANG_DEFAULT if not lang
            else re.split(r"[-_]", str(lang).lower())[0])
    if lang not in _LEX_BY_LANG:
        return np.nan
    text = "" if text is None else str(text)
    if not text.strip():
        return np.nan
    raw = _raw_score(strip_emoji(text), lang)
    if with_emoji and SCORE_EMOJI:
        _n, _ns, _sum = _emoji_valences(text)
        if raw is None and _ns:
            raw = 0.0                    # emoji-only post: the emoji ARE the text
        if raw is not None:
            raw += _sum
    if raw is None:
        return np.nan
    return _normalize(raw)


def score_sentiment(series, lang=None, with_emoji=False):
    """Compound sentiment for a text Series. `lang` is a parallel Series or a str."""
    s = series.fillna("").astype(str) if hasattr(series, "fillna") else pd.Series(series)
    if lang is None:
        langs = pd.Series(LANG_DEFAULT, index=s.index, dtype=object)
    elif isinstance(lang, str):
        langs = pd.Series(lang, index=s.index, dtype=object)
    else:
        langs = lang.reindex(s.index).fillna(LANG_DEFAULT).astype(str)
    out = [score_one(t, l, with_emoji) for t, l in zip(s, langs)]
    return pd.to_numeric(pd.Series(out, index=s.index), errors="coerce")


def emoji_frame(series):
    """Per-post emoji counts and valence sum, as a DataFrame."""
    s = series.fillna("").astype(str) if hasattr(series, "fillna") else pd.Series(series)
    rows = [_emoji_valences(t) for t in s]
    out = pd.DataFrame(rows, index=s.index,
                       columns=["n_emoji", "n_emoji_scored", "emoji_sum"])
    # Mean valence over SCORED emoji only; NaN where none scored, because
    # "this post's emoji carry no measurable valence" is not "valence zero".
    out["emoji_valence"] = np.where(out["n_emoji_scored"] > 0,
                                    out["emoji_sum"] / out["n_emoji_scored"].replace(0, np.nan),
                                    np.nan)
    out["has_emoji"] = out["n_emoji"] > 0
    return out


print(f"languages       : {SUPPORTED_LANGS}   (default {LANG_DEFAULT})")
for _l in SUPPORTED_LANGS:
    print(f"  {_l}  {len(_LEX_BY_LANG[_l]):>4} words   engine {SENTIMENT_ENGINE_BY_LANG[_l]}")
print(f"emoji lexicon   : {len(_EMOJI_LEX)} entries   SCORE_EMOJI={SCORE_EMOJI}")
print(f"LEX_FINGERPRINT : {LEX_FINGERPRINT}")

_demo = pd.Series([
    "We lost the entire flock to bird flu. Absolutely devastating.",
    "Fresh eggs available today, the girls are healthy and happy!",
    "Perdimos toda la parvada por la gripe aviar. Devastador.",
    "Huevos frescos disponibles hoy, las gallinas estan sanas y felices!",
    "Routine biosecurity update for the barn. \U0001F62D\U0001F494",
    "",
])
_demo_lang = pd.Series(["en", "en", "es", "es", "en", "en"])
print("\ntext only :", score_sentiment(_demo, _demo_lang).round(3).tolist())
print("with emoji:", score_sentiment(_demo, _demo_lang, with_emoji=True).round(3).tolist())
_block_end("BLOCK 05", _T0)

<!-- Last changed: Facebook_20260825B -->
### 05A — Parity report

Run this before anyone spends budget. It answers one question: *will this environment
produce numbers that combine with the other two?*

The sentiment fingerprint is the point. `SENTIMENT_ENGINE` degrades silently through
three fallbacks, and a run scored by the inline lexicon is not interchangeable with one
scored by `vaderSentiment` — but every downstream number still computes and every sanity
check still passes. Scoring fixed strings and comparing the scores catches the
divergence in seconds instead of at combine time.

> ⚠️ **This report is safe to send over Slack or email; a row count is not.** Versions,
> engine names, scores of hardcoded strings, and pass/fail booleans are not Meta data.
> Anything counted from an API response is.


In [ ]:
# BLOCK 05A - Parity report (safe to share: no Meta-derived data)
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 05A")

_parity, _ok = {}, True

# --- environment ------------------------------------------------------------
import platform
import sys as _sys

_parity["python"] = platform.python_version()
_parity["pandas"] = pd.__version__
_parity["numpy"] = np.__version__
_parity["platform"] = platform.system()

try:
    import metacontentlibraryapi as _mcl
    _parity["sdk"] = getattr(_mcl, "__version__", "present (no __version__)")
except Exception as e:
    _parity["sdk"] = f"IMPORT FAILED: {type(e).__name__}"
    _ok = False

_parity["client_usable"] = all(hasattr(client, m) for m in
                               ("get", "post", "has_next_page", "query_next_page"))
_ok &= bool(_parity["client_usable"])

# ---------------------------------------------------------------------------
# NLP CAPABILITY PROBE
# ---------------------------------------------------------------------------
# 20260901A. The design assumes NOTHING is installed - vaderSentiment already
# is not, and the enclave has no outbound network to fetch anything - but an
# assumption that shapes the whole multilingual design deserves to be measured
# rather than asserted. This is what settles whether "proper translation" or an
# off-the-shelf multilingual model was ever an option here.
#
# import-succeeded and usable-with-data are reported SEPARATELY on purpose. An
# import that works and then dies on a missing corpus is exactly the trap the
# nltk-vader fallback already sets: the package is there, the lexicon is not.
def _probe_import(mod):
    try:
        m = __import__(mod)
        return True, getattr(m, "__version__", "no __version__")
    except Exception as e:
        return False, type(e).__name__


def _probe_usable(fn):
    """Run a tiny end-to-end call. True only if it actually produces output."""
    try:
        return bool(fn()) or True
    except Exception as e:
        return f"data missing: {type(e).__name__}"


_PROBE_MODULES = [
    "vaderSentiment", "nltk", "textblob", "pysentimiento", "stanza", "spacy",
    "transformers", "torch", "sentencepiece", "langdetect", "langid", "pycld3",
    "fasttext", "emoji", "argostranslate", "ctranslate2", "sacremoses",
    "polyglot", "sklearn",
]
_probe = {}
for _m in _PROBE_MODULES:
    _found, _ver = _probe_import(_m)
    _probe[_m] = f"{_ver}" if _found else f"absent ({_ver})"

# Data-dependent checks. These are the ones that matter: the package existing
# is not the same as the package working.
_usable = {}
try:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer as _P
    _usable["nltk vader_lexicon"] = _probe_usable(
        lambda: _P().polarity_scores("a good test"))
except Exception as e:
    _usable["nltk vader_lexicon"] = f"absent ({type(e).__name__})"
try:
    import nltk as _nl
    _usable["nltk data path"] = [p for p in _nl.data.path if os.path.isdir(p)] or "none"
except Exception:
    _usable["nltk data path"] = "n/a"
try:
    from textblob import TextBlob as _TB
    _usable["textblob corpora"] = _probe_usable(
        lambda: _TB("a good test").sentiment.polarity)
except Exception as e:
    _usable["textblob corpora"] = f"absent ({type(e).__name__})"
try:
    import spacy as _sp
    _usable["spacy models"] = _sp.util.get_installed_models() or "none installed"
except Exception as e:
    _usable["spacy models"] = f"absent ({type(e).__name__})"
try:
    import huggingface_hub as _hh
    _cache = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
    _usable["hf cached models"] = (sorted(os.listdir(os.path.join(_cache, "hub")))[:10]
                                   if os.path.isdir(os.path.join(_cache, "hub"))
                                   else "no local cache")
except Exception as e:
    _usable["hf cached models"] = f"absent ({type(e).__name__})"

# ---------------------------------------------------------------------------
# SENTIMENT FINGERPRINTS
# ---------------------------------------------------------------------------
# Hardcoded strings, so these scores carry no Meta data. Two environments that
# agree here are scoring the corpus the same way; two that disagree are not.
_PROBES_EN = [
    "we lost the entire flock to avian influenza, devastating",
    "beautiful morning at the farm, the hens are happy",
    "HPAI confirmed. Quarantine in effect until further notice.",
    "",
]
# Parallel Spanish. The pairs are translations of each other, so the per-pair
# delta below is a direct read on how far the two hand-built lexicons sit apart.
# It does NOT make them commensurable - it puts a number on the incomparability
# so the methods write-up can state it instead of assuming it away.
_PROBES_ES = [
    "perdimos toda la parvada por la influenza aviar, devastador",
    "hermosa manana en la granja, las gallinas estan felices",
    "IAAP confirmada. Cuarentena en vigor hasta nuevo aviso.",
    "",
]
# Emoji probes: same sentence with and without, so the delta is the emoji's
# entire contribution.
_PROBES_EMOJI = [
    "the whole barn is gone \U0001F62D\U0001F494",
    "the whole barn is gone",
    "fresh eggs today \U0001F423\U0001F95A",
    "\U0001F44D\U0001F44D\U0001F44D",
]

_parity["sentiment_engine"] = SENTIMENT_ENGINE
_parity["engines_by_lang"] = SENTIMENT_ENGINE_BY_LANG
_parity["lex_fingerprint"] = LEX_FINGERPRINT
_parity["post_langs"] = ",".join(POST_LANGS)
_parity["supported_langs"] = ",".join(SUPPORTED_LANGS)


def _fp(texts, lang, with_emoji=False):
    _s = score_sentiment(pd.Series(texts, dtype=object), lang, with_emoji=with_emoji)
    return " ".join("NaN" if pd.isna(v) else f"{v:+.4f}" for v in _s), _s


try:
    _parity["fingerprint"], _sc_en = _fp(_PROBES_EN, "en")
    _parity["fingerprint_es"], _sc_es = _fp(_PROBES_ES, "es")
    _parity["fingerprint_emoji_off"], _e_off = _fp(_PROBES_EMOJI, "en", False)
    _parity["fingerprint_emoji_on"], _e_on = _fp(_PROBES_EMOJI, "en", True)
except Exception as e:
    _parity["fingerprint"] = f"SCORING FAILED: {type(e).__name__}: {e}"
    _ok = False
    _sc_en = _sc_es = None

# --- checkpoint round trip --------------------------------------------------
# Proves the resume path works HERE, before it matters at hour nine of a poll.
try:
    _probe_obj = {"round": "trip", "unicode": "avian influenza — é \U0001F414", "n": 1}
    _ckpt_write("_parity_probe", _probe_obj)
    _back = _ckpt_read("_parity_probe")
    _rt = _back == _probe_obj

    _df_probe = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]})
    _ckpt_write("_parity_frame", _df_probe)
    _rt &= _ckpt_read("_parity_frame").equals(_df_probe)

    for _f in ("_parity_probe.json", "_parity_frame.pkl.gz"):
        _p = os.path.join(CKPT_DIR, _f)
        if os.path.exists(_p):
            os.remove(_p)
    _parity["checkpoint_roundtrip"] = _rt
    _ok &= _rt
except Exception as e:
    _parity["checkpoint_roundtrip"] = f"FAILED: {type(e).__name__}: {e}"
    _ok = False

_parity["ckpt_dir_writable"] = os.access(CKPT_DIR, os.W_OK)
_ok &= bool(_parity["ckpt_dir_writable"])
_parity["shard"] = SHARD
_parity["categories"] = ",".join(PageCategories)
_parity["notebook"] = NOTEBOOK_VERSION

print("=" * 62)
print(f"PARITY REPORT   shard={SHARD}   {'PASS' if _ok else 'FAIL'}")
print("=" * 62)
for _k, _v in _parity.items():
    print(f"  {_k:<22} {_v}")

print("-" * 62)
print("NLP CAPABILITY PROBE   (import -> version, or why not)")
for _k, _v in _probe.items():
    print(f"  {_k:<22} {_v}")
print("  --- usable with data ---")
for _k, _v in _usable.items():
    print(f"  {_k:<22} {_v}")

if _sc_en is not None and _sc_es is not None:
    print("-" * 62)
    print("EN/ES CALIBRATION   parallel sentences, per-pair delta")
    for _i, (_a, _b) in enumerate(zip(_sc_en, _sc_es)):
        _d = "n/a" if (pd.isna(_a) or pd.isna(_b)) else f"{_b - _a:+.4f}"
        print(f"  pair {_i}   en {_a if pd.isna(_a) else f'{_a:+.4f}'}   "
              f"es {_b if pd.isna(_b) else f'{_b:+.4f}'}   delta {_d}")
    _deltas = [abs(b - a) for a, b in zip(_sc_en, _sc_es)
               if not (pd.isna(a) or pd.isna(b))]
    if _deltas:
        print(f"  mean |delta| {np.mean(_deltas):.4f}   max {max(_deltas):.4f}")
    print("  These lexicons are NOT calibrated to each other. Report per-language")
    print("  means separately; do not read an es -0.6 as an en -0.6.")

print("=" * 62)
print("Safe to paste: versions, engine, fixed-string scores, booleans.")
print("NOT safe to paste: any count taken from an API response.")
if _parity["sentiment_engine"] != "vaderSentiment":
    print(f"\nNOTE: English sentiment engine is '{_parity['sentiment_engine']}', "
          f"not vaderSentiment.\nThat is fine PROVIDED every shard reports the same "
          f"engine, the same\nfingerprints and the same lex_fingerprint - compare "
          f"before anyone starts. It\nis NOT fine to describe the resulting scores "
          f"as VADER scores in the methods\nwrite-up.")

if not _ok:
    print("\n!! Do not start a production shard until this reads PASS.")
elif IS_SMOKE:
    print("\nEnvironment looks good. Run blocks 06-19A on Zoo to prove the full\n"
          "pipeline, then switch SHARD to your production name.")
_block_end("BLOCK 05A", _T0)

<!-- Last changed: Facebook_20260825A -->
## 2 — Pages

Only Pages carry geography, so page discovery has to come first. Country and category
are filtered client-side, not in the request: the API's `admin_countries` is about who
administers the Page, and `categories` is not an exact-primary match.

Discovery is partitioned because it has to be. `facebook/pages/preview` is
synchronous-only, so any single search is capped at 1,000 results in *both* run modes.
Block 07 runs up to three levels:

1. one search per `PageCategories` entry;
2. for any category still returning 1,000 rows, a re-run split across the 16
   `PageDescription` terms;
3. optionally (`PAGE_REFINE_CAPPED`), for any *(category, term)* slice still at 1,000,
   a re-run as `(term & refiner)` AND-groups across `PAGE_REFINERS`.

Every slice is recorded in `PAGE_SLICES`, deduplicated by page id. **Block 07 is meant
to be collapsed** — it prints one line per slice. Block 08 recomputes the funnel and the
truncation report from `PAGE_SLICES` and prints only that.


In [ ]:
# BLOCK 06 - Page search helpers
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 06")

# Page discovery is SYNCHRONOUS ONLY - there is no facebook/pages/job endpoint -
# so every individual search inherits the 1,000-result ceiling, in both modes.
SYNC_SEARCH_CAP = 1000

PAGE_FIELDS = ("id,name,page_categories,about,follower_count,"
               "location.country_code,location.state,location.city,location.zip")


def build_ORterms(terms):
    """Concatenate search terms with | as an OR term.

    The API matches token-by-token with no phrase support, so a multi-word term
    must become an AND group. That creates a term-distance problem, fixed
    client-side by the has_post_term regex in block 14.

    Reserved chars (& | - ( ) and space) are not allowed inside a keyword.
    """
    groups = []
    for t in terms:
        tokens = [tok for tok in re.split(r"[^0-9A-Za-z]+", t) if tok]
        if tokens:
            groups.append(tokens[0] if len(tokens) == 1
                          else "(" + " & ".join(tokens) + ")")
    return " | ".join(groups)


def _with_retry(fn, what):
    """Run one page-search API call, retrying transient failures.

    facebook/pages/preview returns HTTP 500 intermittently and a slice that
    gives up contributes ZERO pages - silent, systematic loss rather than a
    visible error. Retried at the CALL level, not the slice level, so a failure
    on page 40 of 100 does not discard the first 39.
    """
    delay = PAGE_SEARCH_RETRY_SLEEP
    for attempt in range(1, PAGE_SEARCH_RETRIES + 1):
        try:
            return fn()
        except Exception as e:
            if attempt == PAGE_SEARCH_RETRIES:
                raise
            print(f"      {what}: {type(e).__name__} on attempt {attempt}/"
                  f"{PAGE_SEARCH_RETRIES}, retrying in {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


def search_pages(categories, q, max_api_pages=None):
    """One synchronous page search, paginated. Returns (rows, hit_cap, error).

    Paging stops at SYNC_SEARCH_CAP because the API will not serve past it; a
    slice that reaches the cap is TRUNCATED, not complete, and says so.

    A call that exhausts its retries returns the rows it had ALREADY paged plus
    an error string, rather than raising. Throwing away 40 pages of results
    because page 41 returned a 500 is a silent, systematic loss - and on this
    endpoint the slices that fail are not random with respect to the research
    question. `error is not None` is the caller's signal that the slice is
    incomplete by ABSENCE rather than by truncation.
    """
    params = {"q": q,
              "admin_countries": PageCountry_Code,   # comma list, not operators
              "fields": PAGE_FIELDS,
              "categories": categories}
    rows = []

    try:
        resp = _with_retry(
            lambda: client.get(path=PAGES_PREVIEW_PATH, params=params),
            "initial search")
    except Exception as e:
        return rows, False, f"{type(e).__name__}: {e}"

    rows = list(resp.json().get("data", []))
    n_api = 1

    while client.has_next_page(resp) and len(rows) < SYNC_SEARCH_CAP:
        if max_api_pages is not None and n_api >= max_api_pages:
            break
        time.sleep(PAGE_SEARCH_SLEEP_S)      # stay under 60 queries/min
        try:
            resp = _with_retry(lambda: client.query_next_page(resp),
                               f"page {n_api + 1}")
        except Exception as e:
            return (rows, len(rows) >= SYNC_SEARCH_CAP,
                    f"{type(e).__name__}: {e} (after {n_api} api page(s))")
        rows.extend(resp.json().get("data", []))
        n_api += 1

    return rows, len(rows) >= SYNC_SEARCH_CAP, None


def slice_query(level, term):
    """Rebuild the exact query a slice used, so an errored slice can be repaired."""
    if level == 1:
        return build_ORterms(PageDescription)
    if level == 2:
        return build_ORterms([term])
    _t, _r = term.split("+", 1)
    return build_ORterms([f"{_t} {_r}"])


print(f"sync search cap : {SYNC_SEARCH_CAP:,} results per search")
print(f"page fields     : {len(PAGE_FIELDS.split(','))} requested")
print(f"OR query        : {build_ORterms(PageDescription)[:110]} ...")
_block_end("BLOCK 06", _T0)

In [ ]:
# BLOCK 07 - Page discovery - RUN (verbose; collapse me, read block 08)
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 07")

# Discovery is the slowest block in the notebook - synchronous, rate-limited, and
# hours long at full scope. Restoring it rather than repeating it is the point of
# the checkpoint layer.
#
# 20260901A - every slice now prints a projected ETA. The count is a
# projection, not an arithmetic total: the L2 split fires only for
# categories that reach the cap, so how many slices this run will do is not
# knowable in advance. Every line says so with a tilde.
#
# 20260831A - the cold-start path wrote NOTHING until the whole sweep returned.
# At full scope that sweep is 6 L1 categories plus up to 96 L2 term slices, and
# a capped slice is 100 rate-limited API calls, so it runs 3-4 hours. Killing it
# at hour three therefore discarded every row, the query budget spent to get
# them, and any chance of resuming: the gap logic below reads scope back off
# level-1 slices that were never persisted. _discover_categories now checkpoints
# after each category finishes, capping the loss at one category.
#
# 20260825G - the restore path was all-or-nothing, and that hid a real failure. A
# checkpoint written under a NARROWER DiscoveryCategories was reused verbatim, so
# turning DISCOVER_ALL_CATEGORIES on had no effect whatsoever. That is exactly
# what happened on the 2026-08-25 nick run: 18 slices = 2 L1 + 16 L2, shard-only,
# with the widening silently skipped. Scope is now read back off the level-1
# slices and only the GAP is searched.
PAGE_ROWS   = _ckpt_init("PAGE_ROWS", [])    # every row from every slice, pre-dedup
PAGE_SLICES = _ckpt_init("PAGE_SLICES", [])  # one record per search

# ---------------------------------------------------------------------------
# PROGRESS AND ETA
# ---------------------------------------------------------------------------
# 20260901A. Discovery is 3-4 hours and its slice count is not knowable in
# advance: the L2 split fires only for categories that reach the cap. So the
# denominator is PROJECTED, not computed, and every line prints it with a tilde.
# Without this the operator has no way to tell a working run from a hung one,
# which is what put a three-hour run on the chopping block on 2026-08-31.
ETA_WINDOW = 20          # slices averaged for the rate - see below


def _eta_seed():
    """Durations of slices already on record, including from the checkpoint.

    A resumed run restores rec["seconds"] off disk and is calibrated from its
    FIRST slice instead of guessing for the first twenty.
    """
    return [float(r["seconds"]) for r in PAGE_SLICES
            if isinstance(r.get("seconds"), (int, float))
            and not pd.isna(r.get("seconds"))]


_SLICE_SECONDS = _eta_seed()
_ETA_STATE = {"cats_total": 0, "cats_done": 0, "slices_at_cat_start": 0,
              "cat_slices": [], "terms_left": 0, "in_flight": False,
              "l1_unresolved": False}


def _eta_projection():
    """(slices_done, projected_total, seconds_remaining) - all approximate."""
    done = len(_SLICE_SECONDS)
    if not done:
        return done, None, None

    # Rate from the LAST ETA_WINDOW slices, not all of them. A capped slice is
    # 100 rate-limited API calls and an empty one is a single call, so cost
    # varies by an order of magnitude; a run that has moved from Farm into
    # Animal Shelter must not still be paced by Farm's numbers.
    recent = _SLICE_SECONDS[-ETA_WINDOW:]
    rate = sum(recent) / len(recent)

    cats_left = max(0, _ETA_STATE["cats_total"] - _ETA_STATE["cats_done"])
    # Slices per category: observed where we have observations, worst case
    # (1 L1 + every PageDescription term) before the first category completes.
    if _ETA_STATE["cat_slices"]:
        per_cat = sum(_ETA_STATE["cat_slices"]) / len(_ETA_STATE["cat_slices"])
    else:
        per_cat = 1 + len(PageDescription)
    # The category in flight is counted separately: its remaining terms are
    # known exactly, unlike the categories not yet started. in_flight is
    # tracked explicitly rather than inferred from terms_left, because on a
    # category's LAST term terms_left is already 0 while the category is
    # still open - inferring it there counted that category twice and made
    # the estimate jump on every category boundary.
    in_flight_left = _ETA_STATE["terms_left"]
    if (_ETA_STATE["in_flight"] and not in_flight_left
            and _ETA_STATE["l1_unresolved"]):
        # The L1 slice is printing and its split decision is not in yet, so
        # assume this category WILL split. Under-promising is the worse
        # error here: telling someone "0h 00m left" on a block that then
        # runs another hour is exactly the surprise this ETA exists to
        # prevent.
        in_flight_left = max(0.0, per_cat - 1)
    remaining = (in_flight_left
                 + max(0, cats_left - (1 if _ETA_STATE["in_flight"] else 0))
                   * per_cat)
    return done, int(round(done + remaining)), remaining * rate


def _eta_fragment():
    done, total, secs = _eta_projection()
    if total is None or secs is None:
        return ""
    return f"   [{done}/~{total}, ~{_fmt_hm(secs)} left]"


def _run_slice(cat, term, q, level, parent=None):
    """Run one slice, record it, and return (hit_cap, errored).

    A slice that fails keeps whatever it paged before dying - search_pages
    returns partial rows with an error instead of discarding them. `capped` is
    None on an errored slice, which is how block 08 tells absence from
    truncation.

    Duration goes onto the record as `seconds`, so it survives the checkpoint
    and block 08 can report where the hours actually went.
    """
    _t_slice = time.time()
    rows, capped, err = search_pages(cat, q, MAX_API_PAGES)
    _elapsed = time.time() - _t_slice
    _SLICE_SECONDS.append(_elapsed)
    if _ETA_STATE["terms_left"]:
        _ETA_STATE["terms_left"] -= 1

    PAGE_ROWS.extend(rows)
    rec = {"category": cat, "term": term, "level": level, "rows": len(rows),
           "capped": None if err else capped, "sub_split": False,
           "narrowed": False, "seconds": round(_elapsed, 1)}
    if err:
        rec["error"] = err
    PAGE_SLICES.append(rec)
    print(f"  L{level} {cat:<24} {term:<22} {len(rows):>5,} rows"
          + ("  <-- CAPPED" if capped and not err else "")
          + (f"  <-- FAILED {err}" if err else "")
          + _eta_fragment())

    # A term slice that dies wholesale gets a SMALLER query, not another
    # identical one. See PAGE_NARROW_FAILED in block 02 for why a repeat failure
    # is treated as deterministic rather than transient.
    if err and PAGE_NARROW_FAILED and level == 2 and not IS_PREVIEW:
        print(f"      -> {cat}/{term} failed; narrowing across "
              f"{len(PAGE_REFINERS)} refiners")
        rec["narrowed"] = True
        for _ref in PAGE_REFINERS:
            _run_slice(cat, f"{term}+{_ref}",
                       build_ORterms([f"{term} {_ref}"]), level=3)

    return capped, bool(err)


def _discover_categories(cats):
    """L1 sweep over `cats`, sub-splitting any category that caps or fails.

    Factored out so incremental widening and a cold start run the SAME code. A
    page universe assembled across two runs must not have been built by two
    different code paths.

    Checkpoints once per category. The two `continue` guards this loop carried
    became nested `if`s so the write at the bottom is unconditional: a category
    that came back under the cap and needed no sub-split still has to be
    recorded as searched, or the gap logic re-runs it on every pass.
    """
    _q_pages = build_ORterms(PageDescription)
    _ETA_STATE.update(cats_total=len(cats), cats_done=0, cat_slices=[],
                      terms_left=0, in_flight=False, l1_unresolved=False)

    for _cat in cats:
        _t_cat = time.time()
        _ETA_STATE["slices_at_cat_start"] = len(PAGE_SLICES)
        _ETA_STATE["in_flight"] = True
        _ETA_STATE["l1_unresolved"] = True
        _capped_cat, _err_cat = _run_slice(_cat, "<all terms>", _q_pages, level=1)
        _parent = len(PAGE_SLICES) - 1

        # Only a category that is still truncated - or that died wholesale - is
        # worth re-querying per term. The parent slice is EXPECTED to be capped
        # here, which is what triggered the sub-split, so it is excluded from
        # the truncation report in block 08.
        if (_capped_cat or _err_cat) and not IS_PREVIEW:
            print(f"      -> {_cat} {'failed' if _err_cat else 'capped'}; L2 split "
                  f"across {len(PageDescription)} terms")
            PAGE_SLICES[_parent]["sub_split"] = True
            _ETA_STATE["terms_left"] = len(PageDescription)
            _ETA_STATE["l1_unresolved"] = False

            for _term in PageDescription:
                _capped_term, _ = _run_slice(_cat, _term, build_ORterms([_term]),
                                             level=2)
                _parent2 = len(PAGE_SLICES) - 1

                # L3. On 2026-08-24, 13 L2 slices were still capped - Farm and
                # Agricultural Service across livestock/farm/farming/ranch/
                # agriculture/agricultural. AND-refining narrows them, but a
                # page carrying the term and none of the refiners stays
                # invisible.
                if _capped_term and PAGE_REFINE_CAPPED:
                    print(f"          -> {_cat}/{_term} capped; L3 across "
                          f"{len(PAGE_REFINERS)} refiners")
                    PAGE_SLICES[_parent2]["sub_split"] = True
                    for _ref in PAGE_REFINERS:
                        _run_slice(_cat, f"{_term}+{_ref}",
                                   build_ORterms([f"{_term} {_ref}"]), level=3)

        # PER-CATEGORY CHECKPOINT. `_cat` is complete here and only here: its L1
        # slice and every L2/L3 slice beneath it are in PAGE_SLICES, so
        # SEARCHED_CATEGORIES can read it back and the next run skips it.
        # Deliberately AFTER the sub-split rather than straight after the L1
        # call - a category interrupted mid-split must not leave behind a
        # level-1 slice claiming it was searched, because that claim is exactly
        # what the gap logic above trusts.
        #
        # Restricted to the two page structures: a full checkpoint here would
        # re-serialise POST_ROWS, potentially millions of rows, once per
        # category, to persist two lists that are a few hundred KB gzipped.
        # Category finished: fold its real slice count into the projection so
        # the remaining categories are estimated from what actually happened
        # rather than from the worst-case bound.
        _ETA_STATE["terms_left"] = 0
        _ETA_STATE["in_flight"] = False
        _ETA_STATE["l1_unresolved"] = False
        _ETA_STATE["cats_done"] += 1
        _ETA_STATE["cat_slices"].append(
            len(PAGE_SLICES) - _ETA_STATE["slices_at_cat_start"])
        _, _proj_total, _proj_secs = _eta_projection()
        _left = _ETA_STATE["cats_total"] - _ETA_STATE["cats_done"]
        print(f"      {_cat} done in {(time.time() - _t_cat) / 60:.1f} min; "
              f"checkpointing {len(PAGE_ROWS):,} rows so far"
              + (f"\n      {_left} category(ies) left, ~{_proj_total} slice(s) "
                 f"total, ~{_fmt_hm(_proj_secs)} remaining"
                 if _left and _proj_secs is not None else ""))
        checkpoint(f"pages-{_cat}", only=("PAGE_ROWS", "PAGE_SLICES"))


# What this checkpoint has ACTUALLY searched, read back off its level-1 slices
# rather than taken from config - config is the thing someone just changed.
SEARCHED_CATEGORIES = sorted({c.strip()
                              for r in PAGE_SLICES if r.get("level") == 1
                              for c in str(r.get("category", "")).split(",")
                              if c.strip()})
_missing = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]

# A slice that has already been narrowed is not retried: its replacement L3
# slices are sitting in PAGE_SLICES, and re-running it would duplicate them on
# every pass. RESTORE_FROM_CHECKPOINT = False forces a clean sweep instead.
_errored = [r for r in PAGE_SLICES
            if r.get("capped") is None and not r.get("narrowed")]

if PAGE_ROWS and (_missing or (_errored and RETRY_FAILED_SLICES)):
    print(f"{len(PAGE_ROWS):,} page rows present across "
          f"{len(SEARCHED_CATEGORIES)} searched category(ies).\n")

    if _errored and RETRY_FAILED_SLICES:
        # Repair in place. A slice that errored contributed nothing, and redoing
        # the whole of discovery to recover four slices would cost hours.
        print(f"  {len(_errored)} slice(s) errored - repairing those only.\n")
        for _rec in list(_errored):
            PAGE_SLICES.remove(_rec)
            _run_slice(_rec["category"], _rec["term"],
                       slice_query(_rec.get("level", 1), _rec["term"]),
                       level=_rec.get("level", 1))
        checkpoint("pages-repaired")

    if _missing:
        # Incremental widening. Everything already discovered is kept and only
        # the categories this checkpoint never saw are searched, so turning
        # DISCOVER_ALL_CATEGORIES on costs four category sweeps rather than a
        # from-scratch re-run of all of discovery.
        print(f"\n  scope gap: {len(_missing)} of {len(DiscoveryCategories)} "
              f"discovery category(ies) never searched -> {_missing}\n")
        _discover_categories(_missing)
        SEARCHED_CATEGORIES = sorted(set(SEARCHED_CATEGORIES) | set(_missing))
        checkpoint("pages-widened")

elif PAGE_ROWS:
    print(f"{len(PAGE_ROWS):,} page rows already present, all "
          f"{len(DiscoveryCategories)} discovery category(ies) searched, "
          f"nothing left\nto repair - skipping discovery. Set "
          f"RESTORE_FROM_CHECKPOINT = False to force a fresh search.\n")

elif PARTITION_PAGE_SEARCH:
    print(f"partitioned page discovery: {len(DiscoveryCategories)} category "
          f"slices"
          + ("  (preview: no sub-splitting)" if IS_PREVIEW else "")
          + "\n")
    _discover_categories(DiscoveryCategories)
    SEARCHED_CATEGORIES = sorted(DiscoveryCategories)

else:
    print("unpartitioned page discovery - capped at 1,000 rows\n")
    _run_slice(",".join(DiscoveryCategories), "<all terms>",
               build_ORterms(PageDescription), level=1)
    SEARCHED_CATEGORIES = sorted(DiscoveryCategories)

checkpoint("pages-discovered")
_block_end("BLOCK 07", _T0)

In [ ]:
# BLOCK 08 - Page discovery - SUMMARY
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 08")

# Everything block 07 printed line-by-line, recomputed from PAGE_SLICES so the
# verbose block above can stay collapsed.
page_slices = pd.DataFrame(PAGE_SLICES)
for _col, _fill in (("error", None), ("narrowed", False),
                    ("sub_split", False), ("seconds", np.nan)):
    if _col not in page_slices.columns:
        page_slices[_col] = _fill
if len(page_slices):
    page_slices["narrowed"] = page_slices["narrowed"].fillna(False).astype(bool)
    page_slices["sub_split"] = page_slices["sub_split"].fillna(False).astype(bool)

NormalizedRows = (pd.json_normalize(PAGE_ROWS).drop_duplicates("id")
                  if PAGE_ROWS else pd.DataFrame(columns=["id"]))

print(f"{len(PAGE_ROWS):,} rows across {len(page_slices)} slice(s) "
      f"-> {len(NormalizedRows):,} unique pages")

# Scope comes FIRST, because it silently rescales everything under it. The
# categories filter is not an exact-primary match and every slice caps at 1,000,
# so a page whose PRIMARY category is Farm can be reachable only through some
# other category's slice: searching fewer categories finds fewer of your OWN
# pages. A count from a narrow sweep is not comparable to one from a wide sweep.
print(f"\ndiscovery scope : {len(SEARCHED_CATEGORIES)} of "
      f"{len(DiscoveryCategories)} category(ies) searched")
print(f"                  {SEARCHED_CATEGORIES}")
_gap = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]
if _gap:
    print(f"!! {len(_gap)} never searched: {_gap}")
    print("!! Every page and post count below is a FLOOR, the block 12A estimate "
          "included.\n   Re-run block 07 to close the gap before planning "
          "collection.")

if len(page_slices):
    _by_level = (page_slices.groupby("level")
                 .agg(slices=("rows", "size"), rows=("rows", "sum"),
                      capped=("capped", lambda s: int((s == True).sum())),  # noqa: E712
                      failed=("capped", lambda s: int(s.isna().sum()))))
    print("\nby split level:")
    print(_by_level.to_string())

    # Where the hours actually went. Checkpointed with the slices, so this
    # is also what calibrates the next run's ETA from its very first slice
    # instead of guessing for the first twenty.
    _secs = pd.to_numeric(page_slices["seconds"], errors="coerce").dropna()
    if len(_secs):
        print(f"\nslice duration  : {len(_secs)} timed, "
              f"{_fmt_hm(_secs.sum())} total")
        print("  " + "   ".join(f"p{int(q * 100)}={_secs.quantile(q):.0f}s"
                                for q in (0.5, 0.75, 0.9, 1.0)))
    else:
        print("\nslice duration  : not recorded (checkpoint predates 20260901A)")

# A sub-split parent is superseded by its child slices; only unresolved
# truncation counts as truncation.
_capped_slices = (page_slices[(page_slices["capped"] == True)      # noqa: E712
                              & (~page_slices["sub_split"])]
                  if len(page_slices) else pd.DataFrame())
_failed_slices = (page_slices[page_slices["capped"].isna()]
                  if len(page_slices) else pd.DataFrame())

if len(_capped_slices):
    print(f"\n!! {len(_capped_slices)} slice(s) still at the {SYNC_SEARCH_CAP:,} cap - "
          f"truncated by an unknown amount:")
    print(_capped_slices[["level", "category", "term", "rows"]].to_string(index=False))
    print("!! The page universe is a SAMPLE in these slices, not a census. "
          "Record this in the methods write-up.")
    if not PAGE_REFINE_CAPPED:
        print("!! PAGE_REFINE_CAPPED is off - set it True to split these further, "
              "at the cost of a larger and more expensive surface list.")
else:
    print("\nno unresolved slice hit the cap")

if len(_failed_slices):
    _partial = int(_failed_slices["rows"].sum())
    _narrowed = int(_failed_slices["narrowed"].sum())
    print(f"\n!! {len(_failed_slices)} slice(s) errored ({_partial:,} row(s) "
          f"paged before failure were KEPT):")
    print(_failed_slices[["level", "category", "term", "rows", "narrowed",
                          "error"]].to_string(index=False))
    print("!! This is not truncation, it is absence, and it is not random: the "
          "terms that\n   fail are as likely to be the poultry-specific ones as "
          "any other.")
    if _narrowed:
        print(f"!! {_narrowed} of them were re-queried as narrowed "
              f"'(term & refiner)' slices, so they\n   are partly covered - but "
              f"a page carrying the term and none of the refiners\n   is still "
              f"invisible. Say so in the methods write-up.")
    if len(_failed_slices) > _narrowed:
        print("!! The rest contributed only their partial rows. Re-run block 07 "
              "with\n   RETRY_FAILED_SLICES = True to repair just those - "
              "everything already\n   discovered is kept.")
_block_end("BLOCK 08", _T0)

In [ ]:
# BLOCK 09 - Page filter -> SourcePages
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 09")

# Country and PRIMARY-category filtering happen HERE, not in the request.
us = NormalizedRows[NormalizedRows.get("location.country_code",
                                       pd.Series(dtype=object)).fillna("") == "US"]

want = {c.lower() for c in PageCategories}


def primary_category(cell):
    """First element of page_categories, or None."""
    items = cell if isinstance(cell, (list, tuple)) else []
    return str(items[0]) if items else None


us = us.assign(page_category=us["page_categories"].apply(primary_category)
               if "page_categories" in us.columns else None)
CleanUSPages = us[us["page_category"].fillna("").str.lower().isin(want)].copy()

FarmPages = CleanUSPages[CleanUSPages["page_category"].str.lower() == "farm"]

SourcePages = FarmPages if RESTRICT_TO_FARM_ONLY else CleanUSPages

# Preview mode truncates the surface list so the post pull stays a single batch.
if MAX_SURFACES is not None and len(SourcePages) > MAX_SURFACES:
    print(f"{RUN_MODE}: capping surfaces {len(SourcePages):,} -> {MAX_SURFACES}")
    SourcePages = SourcePages.head(MAX_SURFACES)

print(f"{len(NormalizedRows):,} pages -> {len(us):,} US-located "
      f"-> {len(CleanUSPages):,} with a wanted PRIMARY category "
      f"-> {len(FarmPages):,} Farms")
print(f"sourcing posts from {len(SourcePages):,} pages "
      f"({'Farm only' if RESTRICT_TO_FARM_ONLY else 'all wanted categories'})\n")
if len(CleanUSPages):
    print(CleanUSPages["page_category"].value_counts().to_string())

# Drift check against the 2026-08-24 discovery run. The comparison is
# ASYMMETRIC on purpose, because the two directions mean different things.
#
#   SHORTFALL is a problem. It means a truncated or errored page universe, and
#   the post counts built on it are floors that the combined dataset inherits
#   silently.
#
#   SURPLUS is the expected outcome, not an anomaly. The baseline was measured
#   over 8 categories with four Farm slices erroring to ZERO, so it is itself a
#   floor. Wide discovery plus slice repair is supposed to beat it: the
#   2026-08-25 nick run reached 4,246 Farm against a 3,837 baseline (+10.7%)
#   with only two categories searched, and flagging that as drift is a false
#   alarm that trains you to ignore the flag.
print("\nvs 2026-08-24 baseline (itself a FLOOR: 8 categories, 4 Farm slices "
      "errored to zero)")
print(f"   this run searched {len(SEARCHED_CATEGORIES)} of "
      f"{len(DiscoveryCategories)} discovery category(ies)")
_obs = (CleanUSPages["page_category"].value_counts()
        if len(CleanUSPages) else pd.Series(dtype=int))
for _c in PageCategories:
    _exp = PageCategoryPages.get(_c, 0)
    _got = int(_obs.get(_c, 0))
    _pct = (_got / _exp - 1) * 100 if _exp else 0.0
    if _exp and _pct < -10:
        _flag = "  <-- SHORT, investigate"
    elif _exp and _pct > 10:
        _flag = "  (above floor - expected)"
    else:
        _flag = ""
    print(f"  {_c:<26} floor {_exp:>6,}   now {_got:>6,}   {_pct:+6.1f}%{_flag}")

# A page belongs to exactly one shard because this filter is on PRIMARY category.
_stray = set(CleanUSPages["page_category"]) - set(PageCategories)
assert not _stray, f"pages outside this shard's categories leaked through: {_stray}"
_block_end("BLOCK 09", _T0)

<!-- Last changed: Facebook_20260825D -->
## 3 — Posts

`SourcePosts` is kept **whole**. The precise `PostText` concepts are applied later as a
flag, never as a row filter — that is the point of this notebook.

Block 10 only *defines* the fetch machinery. Which path executes depends on `RUN_MODE`:

* **preview** → `client.get("facebook/posts/preview")` per surface batch, paginated,
  capped at 1,000 results per batch.
* **full** → `client.post("facebook/posts/job")` per **(surface batch × time window)**,
  then polling. Submissions are spaced by `JOB_SUBMIT_SLEEP_S` because async search is
  limited to one query per minute.

### The job plan

A *job key* is `b{batch:03d}|{window_start}` — a string, deliberately, so it survives a
JSON round-trip into the checkpoint without hand-coercion. `_job_plan()` is the single
source of truth for what work exists; estimation, submission, and polling all walk it.

### The budget ledger

500,000 records per rolling 7 days. `BUDGET_LEDGER` records what each submission was
estimated to cost and when, `_budget_spent_7d()` sums the live window, and
`submit_post_jobs` stops rather than overrunning it. The ledger is checkpointed, so
next week's session resumes the staged collection instead of restarting it.


In [ ]:
# BLOCK 10 - Post fetch machinery
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 10")

SURFACE_BATCH = 250          # documented max IDs per request
CAP = SYNC_SEARCH_CAP        # sync ceiling; async jobs use JOB_RESULT_CAP

POST_FIELDS = ("id,creation_time,text,lang,surface.id,post_owner_id,"
               "post_owner_type,statistics")

# Module-level so state survives a crash mid-run; _ckpt_init extends that to
# surviving the kernel itself.
POST_ROWS      = _ckpt_init("POST_ROWS", [])       # raw post payloads
POST_BATCH     = _ckpt_init("POST_BATCH", [])      # per-job diagnostics
POST_JOBS      = _ckpt_init("POST_JOBS", {})       # job_key -> job metadata
POST_ESTIMATES = _ckpt_init("POST_ESTIMATES", {})  # job_key -> estimated records
BUDGET_LEDGER  = _ckpt_init("BUDGET_LEDGER", [])   # submissions, for the rolling window


def _epoch(d, end_of_day=False):
    """The async job endpoint's documented samples pass Unix seconds, not ISO dates."""
    ts = pd.Timestamp(d, tz="UTC")
    if end_of_day:
        ts = ts + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    return int(ts.timestamp())


def _base_params(chunk, q, fields, lang=None):
    """Request params for one surface batch.

    `lang` is what this request asks for. In "list" mode the whole of POST_LANGS
    rides on one request; in "per_language" mode the caller passes a single code
    and submits once per language.
    """
    if lang is None:
        lang = (",".join(POST_LANGS) if LANG_MODE_RESOLVED == "list"
                else LANG_DEFAULT)
    p = {"lang": lang,
         "search_scope": "post_text_only",
         "surface_ids": chunk,
         "fields": fields}
    if q:                        # omitted entirely in "all_posts" mode
        p["q"] = q
    return p


def _request_langs():
    """Language values to iterate when building the plan - one, or one each."""
    if LANG_MODE_RESOLVED == "list":
        return [",".join(POST_LANGS)]
    return list(POST_LANGS)


def probe_lang_list_support():
    """One synchronous call to settle whether `lang` accepts a comma list.

    The docs describe `lang` as a single ISO 639-1 code and nothing here has
    ever tested a list, so 20260901A probes rather than assumes. One preview
    request is negligible against the budget, and the failure path is safe: any
    error resolves to per_language, which always works. Called by block 11.

    If it IS per_language, job_key gains a language segment
    ("b003|2024-01-01|es"). Keys written by an earlier version have no such
    segment, so an old checkpoint and a new run never collide - but they also do
    not share work, and the English pass will be re-submitted. Say that in the
    PR body; a reviewer cannot see it in the diff.
    """
    global LANG_MODE_RESOLVED
    if LANG_REQUEST_MODE != "auto":
        LANG_MODE_RESOLVED = LANG_REQUEST_MODE
        return LANG_MODE_RESOLVED
    if len(POST_LANGS) == 1:
        LANG_MODE_RESOLVED = "list"          # nothing to combine
        return LANG_MODE_RESOLVED
    _ids = list(SourcePages["id"])[:1] if len(SourcePages) else []
    if not _ids:
        LANG_MODE_RESOLVED = "per_language"
        print("  lang probe: no surfaces to probe with - staying on 'per_language'")
        return LANG_MODE_RESOLVED
    try:
        client.get(path=POSTS_PREVIEW_PATH,
                   params={"lang": ",".join(POST_LANGS),
                           "search_scope": "post_text_only",
                           "surface_ids": _ids,
                           "fields": "id,lang",
                           "since": RUN_END, "until": RUN_END})
        LANG_MODE_RESOLVED = "list"
        print(f"  lang probe: lang='{','.join(POST_LANGS)}' ACCEPTED "
              f"-> one job per (batch x window)")
    except Exception as e:
        LANG_MODE_RESOLVED = "per_language"
        print(f"  lang probe: comma list REJECTED - {type(e).__name__}: {e}")
        print(f"    -> one job per (batch x window x language): "
              f"{len(POST_LANGS)}x the jobs, {len(POST_LANGS)}x the submission "
              f"wall-clock.")
    return LANG_MODE_RESOLVED


def _batches(page_ids):
    page_ids = list(page_ids)
    for i in range(0, len(page_ids), SURFACE_BATCH):
        yield i // SURFACE_BATCH, page_ids[i:i + SURFACE_BATCH]


def _job_plan(page_ids):
    """Every (surface batch x time window) unit of work, in submission order.

    Windowing is what keeps a job under JOB_RESULT_CAP: one job per batch over
    the whole window averaged ~115,000 posts on the 2026-08-24 estimate, against
    a 100,000 ceiling.
    """
    plan = []
    for b, chunk in _batches(page_ids):
        for since, until in JOB_WINDOWS:
            for lang in _request_langs():
                # The language segment is appended ONLY in per_language mode, so
                # a "list"-mode key is byte-identical to what every earlier
                # version wrote and existing checkpoints keep resuming.
                key = f"b{b:03d}|{since}"
                if LANG_MODE_RESOLVED != "list":
                    key += f"|{lang}"
                plan.append({"job_key": key, "batch": b, "chunk": chunk,
                             "since": since, "until": until, "lang": lang})
    return plan


# ---------------------------------------------------------------------------
# BUDGET
# ---------------------------------------------------------------------------
def _budget_spent_7d(now=None):
    """Estimated records submitted inside the live rolling window."""
    now = time.time() if now is None else now
    cut = now - 7 * 86400
    return sum(e.get("estimated", 0) for e in BUDGET_LEDGER
               if float(e.get("at", 0)) >= cut)


def _budget_remaining():
    return max(0, RECORD_BUDGET - _budget_spent_7d())


def _budget_note(e):
    """Human-readable age of a ledger entry."""
    hrs = (time.time() - float(e.get("at", 0))) / 3600
    return f"{hrs:5.1f} h ago  {e.get('estimated', 0):>8,}  {e.get('job_key', '')}"


# ---------------------------------------------------------------------------
# SYNCHRONOUS path (RUN_MODE == "preview")
# ---------------------------------------------------------------------------
def fetch_posts_sync(page_ids, q, since, until, fields=POST_FIELDS):
    """Paginated sync search per surface batch. Resumable - rerun skips done work."""
    done = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}

    for b, chunk in _batches(page_ids):
      for lang in _request_langs():
        key = f"b{b:03d}|{since}" + (f"|{lang}"
                                     if LANG_MODE_RESOLVED != "list" else "")
        if key in done:
            print(f"  {key}  skip (already done)")
            continue
        try:
            params = _base_params(chunk, q, fields, lang)
            params["since"], params["until"] = since, until

            resp = client.get(path=POSTS_PREVIEW_PATH, params=params)
            recs = list(resp.json().get("data", []))
            while client.has_next_page(resp):
                resp = client.query_next_page(resp)
                recs.extend(resp.json().get("data", []))

            POST_ROWS.extend(recs)
            POST_BATCH.append({"job_key": key, "batch": b, "surfaces": len(chunk),
                               "returned": len(recs), "capped": len(recs) >= CAP})
            print(f"  {key}  {len(recs):>5,} posts"
                  f"{'  <-- CAPPED' if len(recs) >= CAP else ''}"
                  f"   running {len(POST_ROWS):,}")

        except Exception as e:
            POST_BATCH.append({"job_key": key, "batch": b, "surfaces": len(chunk),
                               "returned": None, "capped": None,
                               "error": f"{type(e).__name__}: {e}"})
            print(f"  {key}  FAILED  {type(e).__name__}: {e}")


# ---------------------------------------------------------------------------
# ASYNCHRONOUS path (RUN_MODE == "full")
# ---------------------------------------------------------------------------
def _job_id(payload):
    """Pull the job id out of a /job POST response.

    The docs display the response payload but never name the key, so try the
    plausible shapes and fail loudly with the raw payload rather than guessing.
    """
    if hasattr(payload, "json"):
        payload = payload.json()
    if isinstance(payload, dict):
        for k in ("id", "job_id", "async_job_id", "request_id"):
            if payload.get(k):
                return str(payload[k])
        for k in ("data", "job", "result"):
            inner = payload.get(k)
            if isinstance(inner, list) and inner:
                inner = inner[0]
            if isinstance(inner, dict):
                for kk in ("id", "job_id", "async_job_id"):
                    if inner.get(kk):
                        return str(inner[kk])
    raise KeyError(f"no job id found in submit response: {payload!r}")


def _status_str(st):
    """Normalize get_status() output - dict, response, or bare string."""
    if hasattr(st, "json"):
        try:
            st = st.json()
        except Exception:
            pass
    if isinstance(st, dict):
        for k in ("status", "state", "job_status"):
            if st.get(k):
                return str(st[k]).upper()
    return str(st).upper()


def _job_records(data):
    """Normalize get_data() output to a list of post dicts."""
    if hasattr(data, "json"):
        try:
            data = data.json()
        except Exception:
            pass
    if isinstance(data, dict):
        for k in ("data", "records", "results"):
            if isinstance(data.get(k), list):
                return data[k]
        return []
    return list(data) if isinstance(data, (list, tuple)) else []


def _estimate_count(est):
    """Pull a numeric count out of an /estimate payload of unknown shape."""
    if hasattr(est, "json"):
        try:
            est = est.json()
        except Exception:
            return None
    candidates = [est]
    if isinstance(est, dict):
        for k in ("data", "result", "estimate"):
            v = est.get(k)
            if isinstance(v, list) and v:
                v = v[0]
            if isinstance(v, dict):
                candidates.append(v)
    for c in candidates:
        if isinstance(c, dict):
            for k in ("estimated_count", "estimated_results", "count", "estimate",
                      "total_count", "upper_bound"):
                v = c.get(k)
                if isinstance(v, bool):
                    continue
                if isinstance(v, (int, float)):
                    return int(v)
    return None


def estimate_post_jobs(plan, q, fields=POST_FIELDS, verbose=False):
    """Estimate every planned job. Spends no RESULT budget, but does spend calls.

    Returns (total, n_unparsed). A non-zero n_unparsed means the payload shape is
    not one _estimate_count recognizes, so the total is a FLOOR.
    """
    total, unparsed = 0, 0
    for job in plan:
        key = job["job_key"]
        if key in POST_ESTIMATES:
            total += POST_ESTIMATES[key]
            continue
        try:
            params = _base_params(job["chunk"], q, fields, job.get("lang"))
            params["since"], params["until"] = job["since"], job["until"]
            est = client.get(path=POSTS_ESTIMATE_PATH, params=params).json()
            n = _estimate_count(est)
            if n is None:
                unparsed += 1
                print(f"  {key}  estimate NOT PARSED: {est}")
            else:
                POST_ESTIMATES[key] = n
                total += n
                if verbose:
                    print(f"  {key}  {n:>8,}")
        except Exception as e:
            unparsed += 1
            print(f"  {key}  estimate FAILED  {type(e).__name__}: {e}")
        time.sleep(PAGE_SEARCH_SLEEP_S)      # estimate rides the sync rate limit
    return total, unparsed


def submit_post_jobs(plan, q, fields=POST_FIELDS):
    """Submit planned jobs, budget-gated. Already-submitted keys are skipped.

    Stops - rather than overrunning - when the next job's estimate exceeds what
    the rolling 7-day window has left. The ledger is checkpointed, so a later
    session resumes the staged collection.
    """
    submitted = 0
    for job in plan:
        key = job["job_key"]
        if key in POST_JOBS:
            continue
        if submitted >= MAX_JOBS_PER_RUN:
            print(f"\n  MAX_JOBS_PER_RUN ({MAX_JOBS_PER_RUN}) reached - stopping. "
                  f"Re-run this block to continue.")
            return submitted

        est = POST_ESTIMATES.get(key)
        remaining = _budget_remaining()
        if est is not None and est > remaining:
            print(f"\n  BUDGET STOP at {key}: next job ~{est:,} records, "
                  f"{remaining:,} left in the rolling 7-day window.\n"
                  f"  {len(plan) - submitted:,} job(s) still unsubmitted. Re-run "
                  f"when the window frees up; the ledger is checkpointed.")
            return submitted

        try:
            params = _base_params(job["chunk"], q, fields, job.get("lang"))
            params["since"] = _epoch(job["since"])
            params["until"] = _epoch(job["until"], end_of_day=True)
            params["mode"] = JOB_MODE
            params["name"] = f"{JOB_NAME} [{key}]"
            params["description"] = JOB_DESCRIPTION

            resp = client.post(path=POSTS_JOB_PATH, params=params)
            if not POST_JOBS:                 # show the real payload shape once
                print(f"  submit response payload: {resp.json()}")
            POST_JOBS[key] = {"job_id": _job_id(resp), "batch": job["batch"],
                              "since": job["since"], "until": job["until"],
                              "lang": job.get("lang"),
                              "surfaces": len(job["chunk"]),
                              "status": "IN_PROGRESS", "n": None}
            BUDGET_LEDGER.append({"at": time.time(), "job_key": key,
                                  "estimated": int(est or 0)})
            submitted += 1
            _note = f"   (~{est:,} records)" if est else "   (no estimate)"
            print(f"  {key}  submitted job {POST_JOBS[key]['job_id']}{_note}")
        except Exception as e:
            POST_BATCH.append({"job_key": key, "batch": job["batch"],
                               "surfaces": len(job["chunk"]),
                               "returned": None, "capped": None,
                               "error": f"submit: {type(e).__name__}: {e}"})
            print(f"  {key}  SUBMIT FAILED  {type(e).__name__}: {e}")

        time.sleep(JOB_SUBMIT_SLEEP_S)        # async limit is 1 query/minute
    return submitted


def collect_post_jobs(budget_min=None):
    """Poll submitted jobs, harvesting COMPLETE ones. True when all resolved.

    Bounded by a wall-clock budget because jobs can take days. Re-running is
    safe: harvested keys are recorded in POST_BATCH and skipped.
    """
    budget_min = JOB_POLL_BUDGET_MIN if budget_min is None else budget_min
    deadline = time.time() + budget_min * 60
    harvested = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}

    while True:
        pending = [k for k in POST_JOBS if k not in harvested]
        if not pending:
            return True

        for key in list(pending):
            meta = POST_JOBS[key]
            try:
                job = client.get_async_job(job_id=meta["job_id"])
                status = _status_str(job.get_status())
                meta["status"] = status

                if status.startswith("COMPLETE"):
                    recs = _job_records(job.get_data())
                    POST_ROWS.extend(recs)
                    meta["n"] = len(recs)
                    POST_BATCH.append({"job_key": key, "batch": meta["batch"],
                                       "surfaces": meta["surfaces"],
                                       "returned": len(recs),
                                       "capped": len(recs) >= JOB_RESULT_CAP,
                                       "job_id": meta["job_id"]})
                    harvested.add(key)
                    print(f"  {key}  COMPLETE  {len(recs):>6,} posts"
                          f"{'  <-- CAPPED' if len(recs) >= JOB_RESULT_CAP else ''}"
                          f"   running {len(POST_ROWS):,}")

                elif status.startswith("FAIL") or status.startswith("ERROR"):
                    POST_BATCH.append({"job_key": key, "batch": meta["batch"],
                                       "surfaces": meta["surfaces"],
                                       "returned": None, "capped": None,
                                       "job_id": meta["job_id"],
                                       "error": f"job status {status}"})
                    harvested.add(key)
                    print(f"  {key}  FAILED  job {meta['job_id']} -> {status}")

            except Exception as e:
                print(f"  {key}  poll error  {type(e).__name__}: {e}")

        pending = [k for k in POST_JOBS if k not in harvested]
        if not pending:
            return True
        if time.time() >= deadline:
            print(f"\n  polling budget ({budget_min} min) exhausted; "
                  f"{len(pending)} job(s) still running:")
            for key in pending[:20]:
                print(f"    {key}  {POST_JOBS[key]['job_id']}  "
                      f"{POST_JOBS[key]['status']}")
            if len(pending) > 20:
                print(f"    ... and {len(pending) - 20} more")
            print("  re-run - POST_JOBS is kept, nothing resubmits.")
            return False

        print(f"  {len(pending)} job(s) still running; "
              f"sleeping {JOB_POLL_INTERVAL_S}s")
        time.sleep(JOB_POLL_INTERVAL_S)


print(f"surface batch   : {SURFACE_BATCH}")
print(f"job windows     : {len(JOB_WINDOWS)}")
print(f"languages       : {POST_LANGS}  (request mode {LANG_MODE_RESOLVED}, "
      f"{len(_request_langs())} request(s) per batch x window)")
print(f"restored        : {len(POST_JOBS)} jobs, {len(POST_ROWS):,} rows, "
      f"{len(BUDGET_LEDGER)} ledger entries")
_block_end("BLOCK 10", _T0)

In [ ]:
# BLOCK 11 - Pre-flight - checkpoint, budget, and job status
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 11")

print(f"checkpoint dir : {CKPT_DIR}")
for _f in sorted(os.listdir(CKPT_DIR)):
    print(f"  {_f:<28} {os.path.getsize(os.path.join(CKPT_DIR, _f)) / 1e6:>8.2f} MB")

_harvested = {b["job_key"] for b in POST_BATCH if b.get("returned") is not None}
print(f"\nPAGE_ROWS {len(PAGE_ROWS):,}   POST_ROWS {len(POST_ROWS):,}   "
      f"POST_JOBS {len(POST_JOBS)}   harvested {len(_harvested)}")

print(f"\nrolling 7-day budget: {_budget_spent_7d():,} spent / {RECORD_BUDGET:,} "
      f"-> {_budget_remaining():,} available")
for _e in sorted(BUDGET_LEDGER, key=lambda e: -float(e.get("at", 0)))[:10]:
    print(f"  {_budget_note(_e)}")

if POST_JOBS and not IS_PREVIEW:
    print(f"\nlive status for {len(POST_JOBS)} job(s) "
          f"(one status call each, no result budget spent):")
    _tally = {}
    for _k, _m in sorted(POST_JOBS.items()):
        try:
            _st = _status_str(client.get_async_job(job_id=_m["job_id"]).get_status())
        except Exception as e:
            _st = f"UNREACHABLE ({type(e).__name__})"
        _m["status"] = _st
        _tally[_st] = _tally.get(_st, 0) + 1
    for _st, _n in sorted(_tally.items()):
        print(f"  {_st:<24} {_n:>4}")
    _ready = [k for k, m in POST_JOBS.items()
              if m["status"].startswith("COMPLETE") and k not in _harvested]
    print(f"  COMPLETE but not yet harvested: {len(_ready)} "
          f"(free to fetch - the records are already paid for)")
else:
    print("\nno submitted jobs on record")

# Settle how POST_LANGS reaches the API before anything plans or submits
# against it. One synchronous call when LANG_REQUEST_MODE == "auto", nothing
# otherwise.
if len(POST_LANGS) > 1 or LANG_REQUEST_MODE != "auto":
    print(f"\nlanguage request mode ({LANG_REQUEST_MODE}):")
    probe_lang_list_support()
    print(f"  resolved -> {LANG_MODE_RESOLVED}  "
          f"({len(_request_langs())} request(s) per batch x window)")
_block_end("BLOCK 11", _T0)

In [ ]:
# BLOCK 12 - Post plan and estimate - RUN (verbose; collapse me, read block 12A)
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 12")

_q = None if POST_QUERY_MODE == "all_posts" else build_ORterms(PostSingleWordText)
if _q is None:
    print("POST_QUERY_MODE='all_posts': no q sent, so n_posts is a true denominator.\n"
          "If every job FAILS with a 400, the endpoint requires a query - set\n"
          "POST_QUERY_MODE='broad_terms' and rerun.\n")
else:
    print("POST_QUERY_MODE='broad_terms': n_posts counts broad-term posts only.\n")

ALL_JOBS_DONE = True     # meaningful in full mode only
SUBMIT_GATED  = False    # set by block 12A
PLAN, PLAN_TODO = [], []
_est, _unparsed = 0, 0

if IS_PREVIEW:
    print(f"SYNC pull from {POSTS_PREVIEW_PATH}  ({RUN_START} .. {RUN_END})")
    fetch_posts_sync(SourcePages["id"], q=_q, since=RUN_START, until=RUN_END)
else:
    PLAN = _job_plan(SourcePages["id"])
    PLAN_TODO = [j for j in PLAN if j["job_key"] not in POST_JOBS]

    if RUN_ESTIMATE and PLAN_TODO:
        print(f"ESTIMATE via {POSTS_ESTIMATE_PATH} for {len(PLAN_TODO):,} job(s) "
              f"- spends no result budget. One call each, ~"
              f"{len(PLAN_TODO) * PAGE_SEARCH_SLEEP_S / 60:.0f} min.\n")
        _est, _unparsed = estimate_post_jobs(PLAN_TODO, q=_q)
        checkpoint("estimated")
    elif not PLAN_TODO:
        print("every planned job is already submitted - nothing to estimate.")
    else:
        print("RUN_ESTIMATE is off - no estimate, and no gate. Block 12B will "
              "submit\nblind if CONFIRM_SUBMIT_JOBS is set.")
_block_end("BLOCK 12", _T0)

In [ ]:
# BLOCK 12A - Plan, estimate, and budget SUMMARY
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 12A")

if IS_PREVIEW:
    print("RUN_MODE='preview' - synchronous pull, no plan and no budget gate.")
else:
    _over = [k for k, v in POST_ESTIMATES.items() if v >= JOB_RESULT_CAP]
    _n_batches = math.ceil(len(SourcePages) / SURFACE_BATCH)
    # 20260901A - the language dimension entered PLAN, and this line still
    # printed "batches x windows". In per_language mode that product did
    # not equal the total it was decomposing, on the one block people read
    # to decide whether the shard is a 4-week or a 12-week job.
    _n_langs = len(_request_langs())
    _decomp = _n_batches * len(JOB_WINDOWS) * _n_langs

    print(f"plan               : {len(PLAN):,} job(s) = {_n_batches} batch(es) "
          f"x {len(JOB_WINDOWS)} window(s)"
          + (f" x {_n_langs} language(s)" if _n_langs > 1 else "")
          + (f"   !! does not multiply out to {_decomp:,} - the plan and "
             f"this line disagree" if _decomp != len(PLAN) else ""))
    print(f"already submitted  : {len(PLAN) - len(PLAN_TODO):,}")
    print(f"still to submit    : {len(PLAN_TODO):,}")

    if POST_ESTIMATES:
        print(f"\nestimated remaining: {_est:,} posts across {len(PLAN_TODO):,} "
              f"unsubmitted job(s)")
        print(f"budget             : {RECORD_BUDGET:,} records / rolling 7 days "
              f"-> {_est / RECORD_BUDGET:.1f} budget-week(s)")
        print(f"already spent (7d) : {_budget_spent_7d():,}  "
              f"-> {_budget_remaining():,} available now")
        print(f"jobs over the cap  : {len(_over)} of {len(POST_ESTIMATES)} "
              f"at >= {JOB_RESULT_CAP:,}")
        if _over:
            print(f"  !! those jobs WILL truncate. Narrow JOB_WINDOW "
                  f"('{JOB_WINDOW}' -> 'quarter' -> 'month') and re-plan.")
        if _unparsed:
            print(f"  !! {_unparsed} estimate(s) unparsed - the total is a FLOOR.")

        # Wall-clock, which MAX_JOBS_PER_RUN does not tell you. Submission is
        # bounded by the rolling RECORD budget, not by how many jobs you are
        # willing to fire: at ~23,000 records a job, 500,000 per 7 days admits
        # about 21 of them a week however long you sit here. That is the number
        # that decides whether this shard is a 4-week or a 12-week job.
        _avg = _est / max(len(PLAN_TODO), 1)
        _per_week = int(RECORD_BUDGET // _avg) if _avg else 0
        _weeks = math.ceil(len(PLAN_TODO) / _per_week) if _per_week else 0
        print(f"\nmean job size      : {_avg:,.0f} records")
        print(f"budget admits      : ~{_per_week} job(s) per rolling 7 days "
              f"-> ~{_weeks} week(s) to submit all {len(PLAN_TODO):,}")
        print(f"this session       : at most "
              f"{min(MAX_JOBS_PER_RUN, len(PLAN_TODO)):,} job(s) "
              f"(MAX_JOBS_PER_RUN={MAX_JOBS_PER_RUN}); the budget stops it "
              f"sooner\n                     if the rolling window is already "
              f"spent.")
        print("  !! UNRESOLVED: whether the 500,000/7d budget is scoped per "
              "RESEARCHER or per\n     PROJECT. Per-researcher, the three "
              "shards run concurrently and the week\n     count above is the "
              "real one. Per-project they share it, and the true\n     figure "
              "is roughly three times that. Nobody has checked.")

    print(f"\nscope: {len(SourcePages):,} surfaces from a page universe whose "
          f"slices cap at\n{SYNC_SEARCH_CAP:,}. This is NOT an estimate of all US "
          f"farm-page posts.")

    # The estimate scales with the surface list, so an incomplete page universe
    # produces a total that is low for a reason having nothing to do with how
    # many posts exist. Say which kind of number this is before anyone plans a
    # collection schedule around it.
    _scope_gap = [c for c in DiscoveryCategories if c not in SEARCHED_CATEGORIES]
    if _scope_gap:
        print(f"!! discovery is INCOMPLETE - {len(_scope_gap)} category(ies) "
              f"unsearched: {_scope_gap}.\n   Every total above is a floor and "
              f"will grow. Close the gap in block 07\n   before deciding scope.")

    # --- the gate -----------------------------------------------------------
    if RUN_ESTIMATE and PLAN_TODO and not CONFIRM_SUBMIT_JOBS:
        SUBMIT_GATED = True
        ALL_JOBS_DONE = False
        print("\n" + "=" * 62)
        print("STOPPED before submitting - no result budget has been spent.")
        print("Read the numbers above. To proceed: set CONFIRM_SUBMIT_JOBS = True")
        print("in block 02, re-run blocks 02-03, then run block 12B.")
        print("=" * 62)
        print("\nBlocks 13-21 below will run on an EMPTY frame. That is expected;")
        print("nothing is broken and nothing was submitted.")
    elif not PLAN_TODO:
        print("\nnothing left to submit - go to block 12B to keep polling.")
    else:
        print("\nCONFIRM_SUBMIT_JOBS is set - block 12B will submit.")
_block_end("BLOCK 12A", _T0)

In [ ]:
# BLOCK 12B - Submit and poll - RUN (verbose; collapse me, read block 13)
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 12B")

if IS_PREVIEW:
    print("preview mode - block 12 already fetched synchronously, nothing to submit.")
elif SUBMIT_GATED:
    print("the estimate gate is closed - nothing submitted. See block 12A.")
else:
    print(f"ASYNC submit to {POSTS_JOB_PATH}"
          f"  (<= {MAX_JOBS_PER_RUN} job(s) x {JOB_SUBMIT_SLEEP_S}s spacing "
          f"~ {MAX_JOBS_PER_RUN * JOB_SUBMIT_SLEEP_S / 3600:.1f} h)\n")
    _n_sub = submit_post_jobs(PLAN, q=_q)

    # Job ids before anything else: they are the only handle on results Meta
    # already holds, and losing them means re-spending the query budget.
    checkpoint("post-submit")

    print(f"\npolling: up to {POLL_ROUNDS} round(s) x {JOB_POLL_BUDGET_MIN} min "
          f"(~{POLL_ROUNDS * JOB_POLL_BUDGET_MIN / 60:.1f} h), "
          f"checkpointing after each")
    for _round in range(POLL_ROUNDS):
        print(f"\n  -- poll round {_round + 1}/{POLL_ROUNDS} --")
        ALL_JOBS_DONE = collect_post_jobs()
        checkpoint(f"poll-{_round + 1}")
        if ALL_JOBS_DONE:
            break

checkpoint("posts-fetched")
_block_end("BLOCK 12B", _T0)

In [ ]:
# BLOCK 13 - Post pull - SUMMARY
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 13")

# Everything block 12 printed job-by-job, recomputed so it can stay collapsed.
#
# NAMING (renamed in 20260825D). This frame was called FarmPosts through
# 20260825C, which was a leftover from H5N1_20260819 when PageCategories was
# ["Farm"]. It has never been Farm-only since: it holds every post from every
# page in SourcePages, which is CleanUSPages - all of this shard's categories -
# unless RESTRICT_TO_FARM_ONLY is set. SourcePages -> SourcePosts now reads as
# the lineage it actually is. FarmPages keeps its name; that one really is
# Farm-only.
batch_report = pd.DataFrame(POST_BATCH)

# json_normalize([]) returns a frame with NO columns, so drop_duplicates("id")
# would raise KeyError - which is exactly what the estimate gate produces.
SourcePosts = (pd.json_normalize(POST_ROWS).drop_duplicates("id") if POST_ROWS
             else pd.DataFrame(columns=["id", "creation_time", "text", "lang",
                                        "surface.id", "post_owner_id",
                                        "post_owner_type"]))

# Language, normalized once, here. The API returns ISO 639-1 and may carry a
# region suffix; everything downstream dispatches on the bare code. Absent or
# unparseable falls back to LANG_DEFAULT rather than to NaN, because a NaN here
# would silently drop the post out of every per-language column.
if "lang" in SourcePosts.columns:
    SourcePosts["post_lang"] = (SourcePosts["lang"].astype(str).str.lower()
                                .str.split("[-_]", regex=True).str[0]
                                .replace({"": None, "nan": None, "none": None})
                                .fillna(LANG_DEFAULT))
else:
    SourcePosts["post_lang"] = LANG_DEFAULT

_capped = int(batch_report.get("capped", pd.Series(dtype=float)).fillna(False).sum())
_failed = int(batch_report.get("returned", pd.Series(dtype=float)).isna().sum())
_cap_used = CAP if IS_PREVIEW else JOB_RESULT_CAP

print(f"{len(SourcePosts):,} unique posts from {len(SourcePages):,} pages")
print(f"{len(POST_ROWS):,} raw rows -> {len(POST_ROWS) - len(SourcePosts):,} duplicates "
      f"dropped (windows overlap only at boundaries; a large number here means "
      f"JOB_WINDOWS are wrong)")
print(f"{_capped} of {len(batch_report)} jobs hit the {_cap_used:,} cap; "
      f"{_failed} failed")
if len(SourcePosts):
    _lang_mix = SourcePosts["post_lang"].value_counts()
    print(f"\nlanguages returned ({len(_lang_mix)}): "
          + ", ".join(f"{k} {v:,}" for k, v in _lang_mix.head(10).items()))
    _asked = set(POST_LANGS)
    _missing_lang = sorted(_asked - set(_lang_mix.index))
    if _missing_lang:
        print(f"!! requested but NOT returned: {_missing_lang}. Either these "
              f"pages post\n   none, or the lang parameter did not do what was intended - check\n   block 11's probe result before reading a zero as a finding.")

if not IS_PREVIEW and len(POST_JOBS):
    _done = sum(1 for m in POST_JOBS.values() if str(m.get("status", "")).startswith("COMPLETE"))
    print(f"\njobs: {len(POST_JOBS)} submitted, {_done} complete, "
          f"{len(POST_JOBS) - _done} outstanding")
    print(f"rolling 7-day budget: {_budget_spent_7d():,} / {RECORD_BUDGET:,} spent")
    if len(batch_report) and "returned" in batch_report:
        _by_window = (batch_report.assign(
            window=batch_report["job_key"].str.split("|").str[1])
            .groupby("window")["returned"].agg(["size", "sum"]))
        print("\nposts by job window:")
        print(_by_window.to_string())

if _capped:
    print(f"\n!! {_capped} job(s) at the cap - those counts are FLOORS, not totals. "
          f"Narrow JOB_WINDOW and resubmit the affected keys.")
if SUBMIT_GATED:
    print("\n!! Estimate gate is closed - no posts fetched. Blocks below run on an "
          "EMPTY frame, which is handled but produces nothing.")
elif not ALL_JOBS_DONE:
    print("\n!! Jobs are still running - SourcePosts is PARTIAL. Re-run block 12 "
          "before trusting anything below.")
_block_end("BLOCK 13", _T0)

<!-- Last changed: Facebook_20260825D -->
## 4 — Term flag

`build_ORterms` sent single tokens to the API because there is no phrase search, so a
post containing "flu" anywhere qualifies. Block 14 re-imposes the real `PostText`
concepts — multi-word terms as adjacency patterns — and records the result as
`has_post_term` + `matched_terms`. **No rows are dropped.**

> **Fixed 2026-08-25.** The previous implementation was
> `hits.apply(lambda r: ", ".join(hits.columns[r]), axis=1)`, which raised
> *"Cannot set a DataFrame with multiple columns to the single column matched_terms"*.
> `DataFrame.apply(axis=1)` on a **zero-row** frame returns a DataFrame, not a Series —
> and a zero-row `SourcePosts` is exactly what the estimate gate produces. The numpy path
> below returns a plain list at every size, and is roughly two orders of magnitude
> faster across millions of rows, which matters now that the corpus is measured in
> millions rather than thousands.


In [ ]:
# BLOCK 14 - Term flag (requirement 1), per language
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 14")

def term_re(term):
    """Multi-word terms become adjacency patterns; single tokens stay bounded tokens."""
    return re.compile(
        r"\b" + r"[\s\-/]+".join(map(re.escape, term.split())) + r"\b",
        re.IGNORECASE | re.UNICODE,
    )


# 20260901A - terms are per language now. A post is tested against the
# language-agnostic tokens (h5n1, hpai, ... - identical strings everywhere) plus
# the term list for ITS OWN language, taken from the API's `lang` field.
#
# This costs no query budget. In the default POST_QUERY_MODE = "all_posts" no q
# is sent at all, so the term flag is purely client-side: adding a language's
# vocabulary here is free, and only widening the request's `lang` (block 10)
# spends anything.
TERMS_ANY = {t: term_re(t) for t in POST_TERMS_ANY_LANG}
TERMS_BY_LANG = {lang: {t: term_re(t) for t in terms}
                 for lang, terms in POST_TEXT_BY_LANG.items()}
TERMS = dict(TERMS_ANY)
for _lang_terms in TERMS_BY_LANG.values():
    TERMS.update(_lang_terms)

_txt = (SourcePosts["text"].fillna("") if "text" in SourcePosts.columns
        else pd.Series("", index=SourcePosts.index, dtype=object))
_lang = (SourcePosts["post_lang"] if "post_lang" in SourcePosts.columns
         else pd.Series(LANG_DEFAULT, index=SourcePosts.index, dtype=object))

# One boolean column per term, then masked to the posts whose language that
# term belongs to. A Spanish phrase matching inside an English post would be a
# false positive on a coincidence, and the language-agnostic tokens are exempt
# from the mask precisely because they carry no language.
hits = pd.DataFrame(index=SourcePosts.index)
for _t, _rx in TERMS_ANY.items():
    hits[_t] = _txt.str.contains(_rx)
for _lg, _terms in TERMS_BY_LANG.items():
    _in_lang = (_lang == _lg)
    for _t, _rx in _terms.items():
        hits[_t] = _txt.str.contains(_rx) & _in_lang

# --- the 2026-08-25 fix -----------------------------------------------------
# hits.apply(..., axis=1) returns a DATAFRAME when hits has zero rows, and
# pandas refuses to assign a multi-column frame to one column. Going through
# numpy returns a plain list at every size - including empty - and skips the
# per-row Python call for the ~99% of posts that match nothing.
_term_names = np.array(list(hits.columns), dtype=object)
_hit_matrix = hits.to_numpy(dtype=bool) if len(hits.columns) else \
    np.zeros((len(SourcePosts), 0), dtype=bool)
_any_hit = _hit_matrix.any(axis=1) if _hit_matrix.size else \
    np.zeros(len(SourcePosts), dtype=bool)

matched = np.full(len(SourcePosts), "", dtype=object)
for _i in np.flatnonzero(_any_hit):
    matched[_i] = ", ".join(_term_names[_hit_matrix[_i]])

SourcePosts["has_post_term"] = _any_hit
SourcePosts["matched_terms"] = matched

# Kept only for continuity with the older notebooks - not used downstream.
PostsWithTerm = SourcePosts.loc[SourcePosts["has_post_term"]]

_rate = SourcePosts["has_post_term"].mean() if len(SourcePosts) else 0.0
print(f"{len(SourcePosts):,} posts kept; {int(_any_hit.sum()):,} ({_rate:.2%}) "
      f"match a PostText term")
print(f"term vocabulary: {len(TERMS_ANY)} language-agnostic + "
      + " + ".join(f"{len(v)} {k}" for k, v in TERMS_BY_LANG.items()))
if len(SourcePosts):
    print("\nper-term hits:")
    print(hits.sum().sort_values(ascending=False).to_string())
    # Match rate BY LANGUAGE. A language whose posts never match is either
    # genuinely absent from the corpus or missing vocabulary here, and those
    # two look identical in the pooled number above.
    _by_lang = (pd.DataFrame({"lang": _lang, "hit": _any_hit})
                .groupby("lang")["hit"].agg(["size", "sum", "mean"]))
    print("\nmatch rate by language:")
    print(_by_lang.rename(columns={"size": "posts", "sum": "matched",
                                   "mean": "rate"}).to_string())
    _unsupported = sorted(set(_lang.dropna().unique()) - set(SUPPORTED_LANGS))
    if _unsupported:
        print(f"\n!! {len(_unsupported)} language(s) present with NO term list: "
              f"{_unsupported[:12]}")
        print("   Those posts can only match the language-agnostic tokens, so "
              "their\n   n_posts_term is a floor. Add a term list or read the "
              "counts as such.")
else:
    print("SourcePosts is empty - nothing to flag (see block 13).")
_block_end("BLOCK 14", _T0)

<!-- Last changed: Facebook_20260825A -->
## 5 — Page geography and category

`page_category` / `state` / `zip` describe the **posting Page**, not the author. Meta
exposes no author geography and no county, which is why the modeling grain for social
signal is state-or-coarser. Zip is included because it is the finest thing available,
not because it is a reliable locator of the person who wrote the post.


In [ ]:
# BLOCK 15 - Attach page geography and category (requirement 2)
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 15")

PageLookup = (
    SourcePages[["id", "name", "page_category",
                 "location.state", "location.city", "location.zip"]]
    .rename(columns={"id": "page_id",
                     "name": "page_name",
                     "location.state": "state",
                     "location.city": "city",
                     "location.zip": "zip"})
)

SourcePosts = SourcePosts.merge(PageLookup, left_on="surface.id", right_on="page_id",
                            how="left")

# Grouping keys must not be NaN or the cells silently vanish from the aggregate.
SourcePosts["page_category"] = SourcePosts["page_category"].fillna("UNKNOWN")
SourcePosts["state"] = SourcePosts["state"].fillna("").replace("", "UNKNOWN")
SourcePosts["zip"] = (SourcePosts["zip"].fillna("").astype(str).str.strip()
                    .str.split("-").str[0].replace("", "UNKNOWN"))

SourcePosts["post_date"] = pd.to_datetime(
    SourcePosts["creation_time"], errors="coerce", utc=True
).dt.tz_convert(None).dt.date

_unmatched = SourcePosts["page_id"].isna().sum()
print(f"{_unmatched:,} posts did not join to a page "
      f"({_unmatched / max(len(SourcePosts), 1):.1%})")
print(f"{SourcePosts['post_date'].isna().sum():,} posts have an unparseable creation_time")
if len(SourcePosts):
    print(f"{(SourcePosts['zip'] == 'UNKNOWN').mean():.1%} of posts have no page zip")
_block_end("BLOCK 15", _T0)

<!-- Last changed: Facebook_20260825E -->
## 6 — Engagement, reliability flag, and sentiment

Post-level measures only. The decile bucketing that used to live here is gone — the
aggregate carries means, page-equal means, and extremes instead of twenty count
columns.


In [ ]:
# BLOCK 16 - Engagement, reliability, sentiment (requirements 3-6)
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 16")

def _num(df, col):
    """Numeric column or a zero column if the API omitted it."""
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0.0, index=df.index)


SourcePosts["like_count"] = _num(SourcePosts, "statistics.like_count")
SourcePosts["sad_count"]  = _num(SourcePosts, "statistics.sad_count")
SourcePosts["like_sad"]   = SourcePosts["like_count"] + SourcePosts["sad_count"]

# --- requirement 4: reliability flag ---------------------------------------
SourcePosts["engagement_ge10"] = SourcePosts["like_sad"] >= ENGAGEMENT_MIN

# --- requirement 5: like share of like+sad ----------------------------------
# NaN where like+sad == 0: an undefined ratio, not a zero one.
SourcePosts["like_ratio"] = np.where(
    SourcePosts["like_sad"] > 0,
    SourcePosts["like_count"] / SourcePosts["like_sad"].replace(0, np.nan),
    np.nan,
)

# --- sad share, and the term-restricted view of it -------------------------
# sad_share is exactly 1 - like_ratio wherever the ratio is defined. It is
# carried separately because the signal of interest is a HIGH sad share, and
# restricting it to posts that matched a real PostText concept is what makes it
# a bird-flu signal rather than a general-misery signal.
SourcePosts["sad_share"] = np.where(
    SourcePosts["like_sad"] > 0,
    SourcePosts["sad_count"] / SourcePosts["like_sad"].replace(0, np.nan),
    np.nan,
)
SourcePosts["sad_share_term"] = SourcePosts["sad_share"].where(SourcePosts["has_post_term"])

# --- requirement 6: sentiment, scored TWICE --------------------------------
# 20260901A. Emoji contributed exactly nothing before this version - the
# tokenizer discarded them - so whether they carry signal was never tested here.
# The prior cohort found emoji ADDED NOISE; the Laison/Lyme work found the
# opposite. That disagreement is settled with our own data, not inherited.
#
# The test cannot run in this enclave: it needs the USDA target, which lives in
# Postgres outside and can never be joined to Meta data inside. So both variants
# are produced and both are exported; block 19A reports whether they even
# differ, and the does-it-help comparison happens downstream.
#
#   sentiment        text only, emoji STRIPPED. Unchanged meaning from earlier
#                    versions, so everything already collected stays comparable.
#   sentiment_emoji  the same text score with emoji valence folded in.
_text = (SourcePosts["text"] if "text" in SourcePosts.columns
         else pd.Series("", index=SourcePosts.index, dtype=object))
_plang = (SourcePosts["post_lang"] if "post_lang" in SourcePosts.columns
          else pd.Series(LANG_DEFAULT, index=SourcePosts.index, dtype=object))

SourcePosts["sentiment"] = score_sentiment(_text, _plang, with_emoji=False)
SourcePosts["sentiment_emoji"] = (
    score_sentiment(_text, _plang, with_emoji=True) if SCORE_EMOJI
    else SourcePosts["sentiment"])

_ef = emoji_frame(_text)
for _c in ("n_emoji", "n_emoji_scored", "emoji_valence", "has_emoji"):
    SourcePosts[_c] = _ef[_c]

# The term-restricted views, for the same reason sad_share_term exists: an
# extreme is only an H5N1 signal if the post was about H5N1.
SourcePosts["sentiment_term"] = SourcePosts["sentiment"].where(
    SourcePosts["has_post_term"])
SourcePosts["sentiment_emoji_term"] = SourcePosts["sentiment_emoji"].where(
    SourcePosts["has_post_term"])

_n = max(len(SourcePosts), 1)
print(f"like+sad >= {ENGAGEMENT_MIN}: {int(SourcePosts['engagement_ge10'].sum()):,} "
      f"({SourcePosts['engagement_ge10'].sum() / _n:.1%}) of {len(SourcePosts):,} posts")
print(f"like_ratio defined for {int(SourcePosts['like_ratio'].notna().sum()):,} posts "
      f"(mean {SourcePosts['like_ratio'].mean():.3f})")
print(f"sentiment scored for {int(SourcePosts['sentiment'].notna().sum()):,} posts "
      f"(mean {SourcePosts['sentiment'].mean():.3f})")

# Scored-vs-unscored BY LANGUAGE. An unsupported language scores NaN, never 0.0
# - a zero would be indistinguishable from a genuinely neutral post - so this
# table is the only place the unscorable share is visible.
if len(SourcePosts):
    _lang_report = (pd.DataFrame({"lang": _plang,
                                  "scored": SourcePosts["sentiment"].notna()})
                    .groupby("lang")["scored"].agg(["size", "sum", "mean"])
                    .rename(columns={"size": "posts", "sum": "scored",
                                     "mean": "rate"})
                    .sort_values("posts", ascending=False))
    print("\nsentiment coverage by language "
          f"(engines {SENTIMENT_ENGINE_BY_LANG}):")
    print(_lang_report.head(15).to_string())
    _unscored = int(SourcePosts["sentiment"].isna().sum() -
                    (_text.fillna("").astype(str).str.strip() == "").sum())
    if _unscored > 0:
        print(f"!! {_unscored:,} non-empty post(s) could not be scored - no lexicon "
              f"for their\n   language. They are NaN, not 0.0, and "
              f"n_sentiment_scored recovers them.")

print(f"\nsad_share defined for {int(SourcePosts['sad_share'].notna().sum()):,} posts; "
      f"{int(SourcePosts['sad_share_term'].notna().sum()):,} of those match a PostText term")
print(f"emoji: {int(SourcePosts['has_emoji'].sum()):,} post(s) carry at least one "
      f"({SourcePosts['has_emoji'].mean() if len(SourcePosts) else 0:.1%}); "
      f"see block 19A")
_block_end("BLOCK 16", _T0)

<!-- Last changed: Facebook_20260825C -->
## 7 — The aggregate table

One row per **(PageCategory, State, Zip, Date)**, built by `aggregate_posts`, which block
20 reuses at the state grain so the two tables cannot drift apart.

Every `n_*` column is a count of posts except `n_pages`, so `n_posts` is the denominator
for `n_posts_term` and `n_engagement_ge10`, and `n_ratio_defined + n_ratio_undefined ==
n_posts`.

The three ratio columns are three different weightings of the same reactions —
reaction-weighted (`like_ratio_total`), post-weighted (`mean_like_ratio`), and
page-weighted (`mean_like_ratio_page_eq`) — and `max_sad_share_term` is not a weighting
at all but the single loudest term-matching distress signal in the cell. See the intro
for why all four are carried.

All of them are NaN in a cell where nothing qualified — that is correct, and must not be
filled with 0. A NaN `max_sad_share_term` means "no term-matching post here had any
reactions", which is not the same as "no distress here".


In [ ]:
# BLOCK 17 - Aggregate table
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 17")

GRP       = ["page_category", "state", "zip", "post_date"]
GRP_STATE = ["page_category", "state", "post_date"]

# 20260901A - language is carried as COLUMNS, not as a grouping key. Adding
# `lang` to the grain would multiply the cell count, thin every cell, worsen
# small-cell suppression, and force a matching change in block 22's GRP_C and in
# the downstream fact table. Columns keep the grain, keep the shards disjoint,
# and still let anything downstream separate or pool the languages.
LANG_SUFFIXES = [(l, l) for l in POST_LANGS] + [("other", "lang_other")]


def aggregate_posts(df, keys):
    """Build the aggregate at any grain. Used for BOTH the zip-day and state-day
    tables, so the two can never drift apart.

    Three weightings of the same reactions live here on purpose:

      like_ratio_total         pooled over reactions. A national brand with
                               50,000 reactions dominates the cell.
      mean_like_ratio          mean over POSTS - each post counts once.
      mean_like_ratio_page_eq  mean over PAGES - each page counts once, however
                               big, so ten small farms do not collapse into one
                               national brand.

    max_sad_share_term is not an average: it is the loudest single distress
    signal in the cell among posts matching a real PostText concept. One small
    farm reporting a dead bird is the event of interest, and every mean above
    buries it by construction.

    Sentiment appears twice - text-only and emoji-inclusive - because whether
    emoji help is an open question this notebook cannot settle (see block 19A).
    """
    base = (
        df.groupby(keys, dropna=False)
        .agg(
            n_posts              = ("id", "size"),
            n_posts_term         = ("has_post_term", "sum"),
            n_engagement_ge10    = ("engagement_ge10", "sum"),
            n_ratio_defined      = ("like_ratio", "count"),
            n_sentiment_scored   = ("sentiment", "count"),
            n_term_ratio_defined = ("sad_share_term", "count"),
            n_term_sentiment_scored = ("sentiment_term", "count"),
            # The emoji variant needs its OWN denominators. It scores a
            # strict superset of the text-only variant: a post whose text
            # is nothing but emoji is NaN text-only and defined with emoji
            # folded in, so n_sentiment_scored would undercount it.
            n_sentiment_emoji_scored = ("sentiment_emoji", "count"),
            n_term_sentiment_emoji_scored = ("sentiment_emoji_term", "count"),
            n_emoji_posts        = ("has_emoji", "sum"),
            like_count_total     = ("like_count", "sum"),
            sad_count_total      = ("sad_count", "sum"),
            mean_like_ratio      = ("like_ratio", "mean"),
            mean_sentiment       = ("sentiment", "mean"),
            # No median. A median is a central-tendency statistic, and every
            # central-tendency statistic buries the single distressed small farm
            # this notebook exists to find. The extremes replace it.
            min_sentiment        = ("sentiment", "min"),
            min_sentiment_term   = ("sentiment_term", "min"),
            max_sad_share_term   = ("sad_share_term", "max"),
            # --- emoji-inclusive twins -----------------------------------
            mean_sentiment_emoji = ("sentiment_emoji", "mean"),
            min_sentiment_emoji  = ("sentiment_emoji", "min"),
            min_sentiment_emoji_term = ("sentiment_emoji_term", "min"),
            mean_emoji_valence   = ("emoji_valence", "mean"),
        )
    )

    # Page-equal weighting. Pooling WITHIN a page is deliberate - a page's own
    # ratio should use all of its reactions. It is only ACROSS pages that size
    # is ignored.
    per_page = (
        df.groupby(keys + ["page_id"], dropna=False)
        .agg(like=("like_count", "sum"), sad=("sad_count", "sum"),
             sent=("sentiment", "mean"))
    )
    _ls = per_page["like"] + per_page["sad"]
    per_page["ratio"] = np.where(_ls > 0, per_page["like"] / _ls.replace(0, np.nan),
                                 np.nan)

    page_eq = (
        per_page.groupby(level=list(range(len(keys))))
        .agg(n_pages                 = ("ratio", "size"),
             mean_like_ratio_page_eq = ("ratio", "mean"),
             mean_sentiment_page_eq  = ("sent", "mean"))
    )

    out = base.join(page_eq)

    # --- per-language columns ----------------------------------------------
    # A hand-built Spanish lexicon is NOT calibrated against the English one, so
    # a pooled mean_sentiment mixes two instruments. It is kept for continuity,
    # but these columns are what make the mixture visible and separable.
    _lg = df["post_lang"].where(df["post_lang"].isin(POST_LANGS), other="other")
    lang_int_cols, lang_float_cols = [], []
    for _key, _sfx in LANG_SUFFIXES:
        _sub = df[_lg == _key]
        _names = {f"n_posts_{_sfx}": ("id", "size"),
                  f"n_sentiment_scored_{_sfx}": ("sentiment", "count"),
                  f"mean_sentiment_{_sfx}": ("sentiment", "mean")}
        lang_int_cols += [f"n_posts_{_sfx}", f"n_sentiment_scored_{_sfx}"]
        lang_float_cols += [f"mean_sentiment_{_sfx}"]
        if len(_sub):
            out = out.join(_sub.groupby(keys, dropna=False).agg(**_names))
        else:
            for _c in _names:
                out[_c] = np.nan

    out["n_ratio_undefined"] = out["n_posts"] - out["n_ratio_defined"]

    _tot = out["like_count_total"] + out["sad_count_total"]
    out["like_ratio_total"] = np.where(
        _tot > 0, out["like_count_total"] / _tot.replace(0, np.nan), np.nan)

    int_cols = ["n_posts", "n_pages", "n_posts_term", "n_engagement_ge10",
                "n_ratio_defined", "n_ratio_undefined", "n_sentiment_scored",
                "n_term_ratio_defined", "n_term_sentiment_scored",
                "n_sentiment_emoji_scored", "n_term_sentiment_emoji_scored",
                "n_emoji_posts", "like_count_total",
                "sad_count_total"] + lang_int_cols
    float_cols = ["like_ratio_total", "mean_like_ratio", "mean_like_ratio_page_eq",
                  "mean_sentiment", "mean_sentiment_page_eq",
                  "min_sentiment", "min_sentiment_term", "max_sad_share_term",
                  "mean_sentiment_emoji", "min_sentiment_emoji",
                  "min_sentiment_emoji_term", "mean_emoji_valence",
                  "share_non_en"] + lang_float_cols

    out[lang_int_cols] = out[lang_int_cols].fillna(0)
    _en = f"n_posts_{LANG_DEFAULT}"
    out["share_non_en"] = np.where(
        out["n_posts"] > 0,
        1.0 - out.get(_en, pd.Series(0, index=out.index)) / out["n_posts"].replace(0, np.nan),
        np.nan)

    # Model-ready companions. The measurement columns above stay NaN-honest -
    # NaN means "not measurable here", which is true and must stay sayable. The
    # *_model columns are what the H5N1 model consumes, so it never has to drop
    # a row. Which rows were filled is fully recoverable from the n_term_*
    # counts, so nothing is lost by having both.
    model_cols = ["max_sad_share_term_model", "min_sentiment_term_model",
                  "min_sentiment_emoji_term_model"]
    out["max_sad_share_term_model"] = out["max_sad_share_term"].fillna(
        MODEL_FILL_VALUE)
    out["min_sentiment_term_model"] = out["min_sentiment_term"].fillna(
        MODEL_FILL_VALUE)
    out["min_sentiment_emoji_term_model"] = out["min_sentiment_emoji_term"].fillna(
        MODEL_FILL_VALUE)

    out[int_cols] = out[int_cols].fillna(0).astype(int)

    return (out[int_cols + float_cols + model_cols].reset_index()
            .sort_values(keys).reset_index(drop=True))


AggByCatStateZipDate = aggregate_posts(SourcePosts, GRP)

print(f"{len(AggByCatStateZipDate):,} aggregate rows covering "
      f"{AggByCatStateZipDate['n_posts'].sum():,} posts from "
      f"{AggByCatStateZipDate['n_pages'].sum():,} page-cells")
if len(AggByCatStateZipDate):
    _a = AggByCatStateZipDate
    _n = len(_a)
    _thin = (_a["n_posts"] == 1).mean()
    print(f"{_thin:.1%} of cells hold a single post - RETAINED, see block 20")

    # Why max_sad_share_term is NaN, split by cause. The two causes mean
    # different things and it is worth knowing which one dominates.
    _have = int(_a["max_sad_share_term"].notna().sum())
    _no_term = int((_a["n_posts_term"] == 0).sum())
    _term_no_react = int(((_a["n_posts_term"] > 0)
                          & (_a["n_term_ratio_defined"] == 0)).sum())
    print(f"\nmax_sad_share_term measured in {_have:,} of {_n:,} cells "
          f"({_have / _n:.1%})")
    print(f"  NaN - no term-matching post at all      : {_no_term:,}")
    print(f"  NaN - term post(s) but zero reactions   : {_term_no_react:,}")
    print(f"  -> max_sad_share_term_model fills both with {MODEL_FILL_VALUE}; "
          f"n_posts_term and\n     n_term_ratio_defined recover which rows those "
          f"were.")

    _hs = int(_a["min_sentiment_term"].notna().sum())
    print(f"min_sentiment_term measured in {_hs:,} of {_n:,} cells "
          f"({_hs / _n:.1%}); min_sentiment_term_model has no NaN")

    # Language mix. A shard where share_non_en is ~0 has nothing multilingual to
    # say, and that is a finding about the corpus rather than a bug.
    print("\nlanguage mix across cells:")
    for _key, _sfx in LANG_SUFFIXES:
        _c = f"n_posts_{_sfx}"
        print(f"  {_c:<24} {int(_a[_c].sum()):>10,} posts   "
              f"mean_sentiment_{_sfx} measured in "
              f"{int(_a[f'mean_sentiment_{_sfx}'].notna().sum()):,} cells")
    print(f"  share_non_en (post-weighted): "
          f"{1 - _a[f'n_posts_{LANG_DEFAULT}'].sum() / max(_a['n_posts'].sum(), 1):.2%}")
display(AggByCatStateZipDate.head(20))
_block_end("BLOCK 17", _T0)

<!-- Last changed: Facebook_20260825A -->
## 8 — Sanity checks

Run these before believing anything. Each one has failed at least once in a
comparable pipeline.


In [ ]:
# BLOCK 18 - Sanity checks
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 18")

checks = []


def chk(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})


a = AggByCatStateZipDate

chk("aggregate post count == SourcePosts row count",
    a["n_posts"].sum() == len(SourcePosts),
    f"{a['n_posts'].sum():,} vs {len(SourcePosts):,}")

chk("n_posts_term <= n_posts", (a["n_posts_term"] <= a["n_posts"]).all())
chk("n_engagement_ge10 <= n_posts", (a["n_engagement_ge10"] <= a["n_posts"]).all())
chk("n_sentiment_scored <= n_posts", (a["n_sentiment_scored"] <= a["n_posts"]).all())

chk("ratio defined + undefined == n_posts",
    (a["n_ratio_defined"] + a["n_ratio_undefined"] == a["n_posts"]).all())

chk("mean_like_ratio in [0, 1] where defined",
    a["mean_like_ratio"].dropna().between(0.0, 1.0).all())
chk("mean_like_ratio NaN exactly where n_ratio_defined == 0",
    (a["mean_like_ratio"].isna() == (a["n_ratio_defined"] == 0)).all())

chk("mean_sentiment in [-1, 1] where defined",
    a["mean_sentiment"].dropna().between(-1.0, 1.0).all())
chk("mean_sentiment NaN exactly where n_sentiment_scored == 0",
    (a["mean_sentiment"].isna() == (a["n_sentiment_scored"] == 0)).all())

chk("no NaN grouping keys", not a[GRP].isna().any().any())

# --- the 20260825C statistics ----------------------------------------------
chk("n_pages >= 1 wherever there are posts",
    (a.loc[a["n_posts"] > 0, "n_pages"] >= 1).all())
chk("n_pages <= n_posts", (a["n_pages"] <= a["n_posts"]).all(),
    "a page cannot contribute more cells than it has posts")
chk("n_term_ratio_defined <= n_posts_term",
    (a["n_term_ratio_defined"] <= a["n_posts_term"]).all())
chk("like_ratio_total in [0, 1] where defined",
    a["like_ratio_total"].dropna().between(0.0, 1.0).all())
chk("mean_like_ratio_page_eq in [0, 1] where defined",
    a["mean_like_ratio_page_eq"].dropna().between(0.0, 1.0).all())
chk("page-equal ratio NaN exactly where no post had reactions",
    (a["mean_like_ratio_page_eq"].isna() == (a["n_ratio_defined"] == 0)).all())
chk("page-equal sentiment NaN exactly where nothing scored",
    (a["mean_sentiment_page_eq"].isna() == (a["n_sentiment_scored"] == 0)).all())
chk("mean_sentiment_page_eq in [-1, 1] where defined",
    a["mean_sentiment_page_eq"].dropna().between(-1.0, 1.0).all())
chk("max_sad_share_term in [0, 1] where defined",
    a["max_sad_share_term"].dropna().between(0.0, 1.0).all())
chk("max_sad_share_term NaN exactly where no term post had reactions",
    (a["max_sad_share_term"].isna() == (a["n_term_ratio_defined"] == 0)).all())

# --- the 20260825E statistics ----------------------------------------------
chk("n_term_sentiment_scored <= n_posts_term",
    (a["n_term_sentiment_scored"] <= a["n_posts_term"]).all())
chk("min_sentiment in [-1, 1] where defined",
    a["min_sentiment"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment_term in [-1, 1] where defined",
    a["min_sentiment_term"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment <= mean_sentiment wherever both are defined",
    (a["min_sentiment"] <= a["mean_sentiment"] + 1e-12).fillna(True).all())
chk("min_sentiment_term NaN exactly where no term post scored",
    (a["min_sentiment_term"].isna() == (a["n_term_sentiment_scored"] == 0)).all())
chk("median_sentiment is gone", "median_sentiment" not in a.columns,
    "a median buries the single distressed small farm - that is why it was cut")

# --- model-ready companions -------------------------------------------------
chk("model columns contain no NaN",
    not a[["max_sad_share_term_model", "min_sentiment_term_model",
           "min_sentiment_emoji_term_model"]].isna().any().any(),
    "the H5N1 model must never have to drop a row")
chk("model columns equal the measurement wherever it is defined",
    (a.loc[a["max_sad_share_term"].notna(), "max_sad_share_term_model"]
     .equals(a.loc[a["max_sad_share_term"].notna(), "max_sad_share_term"]))
    and (a.loc[a["min_sentiment_term"].notna(), "min_sentiment_term_model"]
         .equals(a.loc[a["min_sentiment_term"].notna(), "min_sentiment_term"])),
    "a fill must not alter a measured value")
chk("filled rows are recoverable from the counts",
    (a["max_sad_share_term"].isna() == (a["n_term_ratio_defined"] == 0)).all(),
    "n_term_ratio_defined == 0 is exactly the filled set")

# --- collection completeness -----------------------------------------------
chk("no job failed", _failed == 0, f"{_failed} failed jobs")
chk("no job hit the result cap", _capped == 0,
    f"{_capped} capped at {_cap_used:,} - counts are floors, not totals")
chk("all async jobs finished", ALL_JOBS_DONE,
    "jobs still running - SourcePosts is partial" if not ALL_JOBS_DONE else "")
chk("every planned job was submitted",
    IS_PREVIEW or SUBMIT_GATED or not len(_job_plan(SourcePages["id"])) or
    all(j["job_key"] in POST_JOBS for j in _job_plan(SourcePages["id"])),
    "budget or MAX_JOBS_PER_RUN stopped submission - collection is STAGED, "
    "re-run block 12 next window")

chk("post_date inside the requested window",
    (pd.Series(pd.to_datetime(a["post_date"], errors="coerce")).between(
        pd.Timestamp(RUN_START), pd.Timestamp(RUN_END)) | a["post_date"].isna()).all(),
    f"{RUN_START} .. {RUN_END} ({RUN_MODE})")

# Page discovery is sync-only in both modes, so slices can be silently truncated.
_n_capped_slices = int(((page_slices["capped"] == True)        # noqa: E712
                        & (~page_slices["sub_split"])).sum()) if len(page_slices) else 0
_n_failed_slices = int(page_slices["capped"].isna().sum()) if len(page_slices) else 0
chk("no page-discovery slice hit the sync cap", _n_capped_slices == 0,
    f"{_n_capped_slices} of {len(page_slices)} slices capped at "
    f"{SYNC_SEARCH_CAP:,} - the page universe is a sample there, not a census "
    f"(13 on 2026-08-24)")
chk("no page-discovery slice failed", _n_failed_slices == 0,
    f"{_n_failed_slices} slice(s) errored and contributed ZERO pages - re-run "
    f"block 07 with RETRY_FAILED_SLICES=True before trusting this shard")
chk("discovery ran across all production categories",
    IS_SMOKE or DISCOVER_ALL_CATEGORIES,
    "shard-only discovery finds fewer of the shard's OWN pages, because capped "
    "slices mean more slices find more unique pages")

chk("RUN_MODE is 'full'", RUN_MODE == "full",
    "preview output is a smoke test, not a dataset")

# --- shard integrity --------------------------------------------------------
chk("aggregate holds only this shard's categories",
    set(a["page_category"]) <= set(PageCategories) | {"UNKNOWN"},
    f"{sorted(set(a['page_category']) - set(PageCategories) - {'UNKNOWN'})}")
chk("shard is a production shard", not IS_SMOKE,
    "SHARD='smoke' is an environment check on Zoo, not a dataset")
chk("no UNKNOWN page_category",
    "UNKNOWN" not in set(a["page_category"]),
    "posts that did not join to a page cannot be attributed to a shard")
chk("JOB_WINDOW set in full mode", IS_PREVIEW or JOB_WINDOW is not None,
    "one job per batch over the whole window averaged ~115,000 posts on the "
    "2026-08-24 estimate, against a 100,000 cap")

# --- the 20260901A statistics: emoji ---------------------------------------
# The emoji variant is a parallel measurement, not a refinement, so it gets the
# same treatment as the text-only one. Its denominator is its OWN count column:
# sentiment_emoji is defined for a strict SUPERSET of the posts sentiment is,
# because a post whose text is nothing but emoji scores only with them folded in.
chk("n_emoji_posts <= n_posts", (a["n_emoji_posts"] <= a["n_posts"]).all())
chk("n_sentiment_emoji_scored <= n_posts",
    (a["n_sentiment_emoji_scored"] <= a["n_posts"]).all())
chk("emoji variant scores a superset of the text-only variant",
    (a["n_sentiment_emoji_scored"] >= a["n_sentiment_scored"]).all(),
    "an emoji-only post is NaN text-only and defined with emoji")
chk("mean_sentiment_emoji in [-1, 1] where defined",
    a["mean_sentiment_emoji"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment_emoji in [-1, 1] where defined",
    a["min_sentiment_emoji"].dropna().between(-1.0, 1.0).all())
chk("min_sentiment_emoji_term in [-1, 1] where defined",
    a["min_sentiment_emoji_term"].dropna().between(-1.0, 1.0).all())
chk("mean_sentiment_emoji NaN exactly where nothing scored",
    (a["mean_sentiment_emoji"].isna() == (a["n_sentiment_emoji_scored"] == 0)).all())
chk("min_sentiment_emoji <= mean_sentiment_emoji wherever both are defined",
    (a["min_sentiment_emoji"] <= a["mean_sentiment_emoji"] + 1e-12).fillna(True).all())
chk("min_sentiment_emoji_term NaN exactly where no term post scored",
    (a["min_sentiment_emoji_term"].isna()
     == (a["n_term_sentiment_emoji_scored"] == 0)).all())
chk("n_term_sentiment_emoji_scored <= n_posts_term",
    (a["n_term_sentiment_emoji_scored"] <= a["n_posts_term"]).all())
# mean_emoji_valence is on the LEXICON scale [-4, +4], not the compound scale.
# It is the only float column here that is not a [-1, 1] sentiment.
chk("mean_emoji_valence in [-4, 4] where defined",
    a["mean_emoji_valence"].dropna().between(-4.0, 4.0).all(),
    "raw lexicon valence, not a compound score")
chk("emoji model column equals the measurement wherever defined",
    a.loc[a["min_sentiment_emoji_term"].notna(), "min_sentiment_emoji_term_model"]
    .equals(a.loc[a["min_sentiment_emoji_term"].notna(), "min_sentiment_emoji_term"]),
    "a fill must not alter a measured value")

# --- the 20260901A statistics: language ------------------------------------
# The per-language columns partition n_posts exactly. If they stop summing, a
# post has a lang the aggregate did not account for and some cell is silently
# short - which no other check here would notice.
_lang_count_cols = [f"n_posts_{_s}" for _k, _s in LANG_SUFFIXES]
chk("per-language counts partition n_posts",
    (a[_lang_count_cols].sum(axis=1) == a["n_posts"]).all(),
    f"{_lang_count_cols} vs n_posts")
for _k, _s in LANG_SUFFIXES:
    chk(f"n_posts_{_s} <= n_posts", (a[f"n_posts_{_s}"] <= a["n_posts"]).all())
    chk(f"n_sentiment_scored_{_s} <= n_posts_{_s}",
        (a[f"n_sentiment_scored_{_s}"] <= a[f"n_posts_{_s}"]).all())
    chk(f"mean_sentiment_{_s} in [-1, 1] where defined",
        a[f"mean_sentiment_{_s}"].dropna().between(-1.0, 1.0).all())
    chk(f"mean_sentiment_{_s} NaN exactly where nothing scored",
        (a[f"mean_sentiment_{_s}"].isna()
         == (a[f"n_sentiment_scored_{_s}"] == 0)).all())
chk("share_non_en in [0, 1] where defined",
    a["share_non_en"].dropna().between(0.0, 1.0).all())
chk("share_non_en agrees with the counts",
    ((a["share_non_en"] - (1 - a[f"n_posts_{LANG_DEFAULT}"] / a["n_posts"]))
     .abs() < 1e-9).fillna(True).all())
# An unsupported language scores NaN, never 0.0 - so a shard carrying
# lang_other posts has a scored-post count BELOW its post count, on purpose.
# This is a report, not a failure.
chk("posts in an unsupported language are unscored, not scored as neutral",
    a["n_sentiment_scored_lang_other"].sum() == 0
    or "lang_other" not in [_s for _k, _s in LANG_SUFFIXES],
    f"{int(a['n_posts_lang_other'].sum()):,} post(s) outside POST_LANGS; "
    f"{int(a['n_sentiment_scored_lang_other'].sum()):,} of them scored - should "
    f"be 0 unless a lexicon exists for that language")

report = pd.DataFrame(checks)
print(f"{int(report['pass'].sum())} of {len(report)} checks passed\n")
print(report.to_string(index=False))
_block_end("BLOCK 18", _T0)

In [ ]:
# BLOCK 19 - Distribution report
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 19")

# Where the mass actually sits - useful for deciding whether zip is a usable grain.
print("posts per aggregate cell:")
print(AggByCatStateZipDate["n_posts"].describe().round(2).to_string())
if len(AggByCatStateZipDate):
    print(f"\ncells with a single post: "
          f"{(AggByCatStateZipDate['n_posts'] == 1).mean():.1%}")
    print("\ntop categories by post volume:")
    print(AggByCatStateZipDate.groupby("page_category")["n_posts"]
          .sum().sort_values(ascending=False).to_string())
    print("\ntop states by post volume:")
    print(AggByCatStateZipDate.groupby("state")["n_posts"]
          .sum().sort_values(ascending=False).head(15).to_string())
_block_end("BLOCK 19", _T0)

<!-- Last changed: Facebook_20260901A -->
### 19A — Does the emoji column earn its place?

The prior cohort (Eilken et al.) found emoji **added noise** and destabilised their
time-series models. The Laison/Lyme work found emoji enrichment *helped*. Both cannot be
right about this corpus, and neither finding transfers for free — so `20260901A` carries
both variants and measures.

**What this block can answer:** whether emoji change the score at all, how often, by how
much, how often they flip its sign, and whether the emoji actually appearing in the
corpus are in `_EMOJI_LEX` at all. If the two variants correlate at 0.999 the downstream
test is not worth running, and that is a real result worth reporting rather than a
non-finding.

**What it cannot answer:** whether emoji improve early warning. That comparison needs
USDA outbreak counts, which live in Postgres outside the enclave and can never be joined
to Meta data inside it. It runs downstream, on the exported aggregate, against
`feature_county_day` — and note there that Meta's geography is the **page's** state and
zip, not the author's and not a county, so the join is state-level.

Read the term-matching subset, not the headline. A distress emoji on a post about
nothing is not an H5N1 signal however strongly it scores; if emoji carry anything, it has
to show up on posts that matched a real `PostText` concept.

> ⚠️ **Every number this block prints is Meta-derived**, the emoji frequency table
> included. Read it on screen. The aggregate columns leave only through output review,
> like everything else — a share or a count pasted into Slack is an egress decision, not
> a status update.

In [ ]:
# BLOCK 19A - Emoji contribution report
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 19A")

# Descriptive only, and deliberately so. The question "do emoji improve the
# early-warning signal" cannot be answered in here: it needs the USDA target,
# which lives in Postgres outside the enclave and can never be joined to Meta
# data inside it. What this block establishes is the necessary precondition -
# whether emoji change the score AT ALL, and where. If the answer is "almost
# never", the downstream test is not worth running.
#
# The prior cohort (Eilken et al.) reported emoji ADDED NOISE and destabilised
# their time-series models. This does not assume they were right or wrong.
#
# ⚠️ EGRESS. Every number below is Meta-derived. Read it on screen; do not paste
# it into Slack, email, or a status update. The exported aggregate columns leave
# only through Meta's output review, like everything else.
if not SCORE_EMOJI:
    print("SCORE_EMOJI is off - sentiment_emoji is a copy of sentiment, and there "
          "is\nnothing to compare. Set SCORE_EMOJI = True in block 02.")
elif not len(SourcePosts):
    print("SourcePosts is empty - nothing to report (see block 13).")
else:
    _n = len(SourcePosts)
    _with = SourcePosts["has_emoji"]
    _n_with = int(_with.sum())

    print(f"{_n_with:,} of {_n:,} posts carry an emoji ({_n_with / _n:.1%})")
    print(f"  emoji found      : {int(SourcePosts['n_emoji'].sum()):,}")
    print(f"  with a valence   : {int(SourcePosts['n_emoji_scored'].sum()):,} "
          f"({SourcePosts['n_emoji_scored'].sum() / max(SourcePosts['n_emoji'].sum(), 1):.1%} "
          f"of them are in the lexicon)")

    # Coverage is the first thing to check. A low in-lexicon share means the
    # comparison below is measuring the lexicon's gaps, not the emoji's effect.
    if SourcePosts["n_emoji"].sum() and \
            SourcePosts["n_emoji_scored"].sum() / SourcePosts["n_emoji"].sum() < 0.5:
        print("  !! under half of observed emoji are in _EMOJI_LEX. Extend it from "
              "the\n     frequency table below before reading anything into the "
              "deltas.")

    _both = SourcePosts["sentiment"].notna() & SourcePosts["sentiment_emoji"].notna()
    _delta = (SourcePosts["sentiment_emoji"] - SourcePosts["sentiment"])[_both]
    _moved = _delta[_delta != 0]

    print(f"\nboth variants scored for {int(_both.sum()):,} posts")
    print(f"  score CHANGED in {len(_moved):,} ({len(_moved) / max(int(_both.sum()), 1):.2%})")
    if len(_moved):
        print(f"  |delta| mean {_moved.abs().mean():.4f}   "
              f"median {_moved.abs().median():.4f}   max {_moved.abs().max():.4f}")
        print("  |delta| quantiles: " + "  ".join(
            f"p{int(q * 100)}={_moved.abs().quantile(q):.4f}"
            for q in (0.5, 0.75, 0.9, 0.99)))
        _flip = ((np.sign(SourcePosts["sentiment"][_both]) != 0)
                 & (np.sign(SourcePosts["sentiment_emoji"][_both])
                    != np.sign(SourcePosts["sentiment"][_both])))
        print(f"  SIGN FLIPS: {int(_flip.sum()):,} "
              f"({_flip.mean():.2%} of scored posts)")
        _corr = SourcePosts.loc[_both, ["sentiment", "sentiment_emoji"]].corr().iloc[0, 1]
        print(f"  correlation between the two variants: {_corr:.4f}")
        if _corr > 0.99:
            print("  -> effectively the same variable. Emoji are unlikely to change "
                  "any\n     downstream result; say so rather than carrying two "
                  "columns forever.")

    # The term-matching subset is where an emoji signal would have to show up if
    # it exists at all - a distress emoji on a post about nothing is not an
    # H5N1 signal, however strongly it scores.
    _t = SourcePosts["has_post_term"] & _both
    if int(_t.sum()):
        _dt = (SourcePosts["sentiment_emoji"] - SourcePosts["sentiment"])[_t]
        _dtm = _dt[_dt != 0]
        print(f"\nterm-matching posts: {int(_t.sum()):,} scored, "
              f"{len(_dtm):,} changed ({len(_dtm) / max(int(_t.sum()), 1):.2%})")
        if len(_dtm):
            print(f"  |delta| mean {_dtm.abs().mean():.4f}   "
                  f"max {_dtm.abs().max():.4f}   mean signed {_dtm.mean():+.4f}")
            print(f"  emoji_valence on term posts: mean "
                  f"{SourcePosts.loc[_t, 'emoji_valence'].mean():.3f}")
    else:
        print("\nno term-matching post was scored - nothing to say about the subset "
              "that\nmatters. Widen the corpus before drawing a conclusion.")

    # Frequency table, for extending _EMOJI_LEX. In-lexicon emoji are marked so
    # the gaps are obvious at a glance.
    _counter = {}
    for _lst in SourcePosts.loc[_with, "text"].fillna("").map(extract_emoji):
        for _e in _lst:
            _counter[_e] = _counter.get(_e, 0) + 1
    if _counter:
        _top = sorted(_counter.items(), key=lambda kv: -kv[1])[:20]
        print("\ntop 20 emoji (⚠️ Meta-derived - do not paste out):")
        for _e, _c in _top:
            _v = _EMOJI_LEX.get(_e)
            print(f"  {_e}  {_c:>8,}   " +
                  (f"valence {_v:+.1f}" if _v is not None else "NOT IN LEXICON"))

    print("\nNext step is OUTSIDE the enclave: compare mean_sentiment against")
    print("mean_sentiment_emoji as predictors on feature_county_day. Note that")
    print("Meta's geography is the PAGE's state/zip - not the author's, and not a")
    print("county - so that join is state-level and the resolution mismatch is a")
    print("real limitation, not a detail.")
_block_end("BLOCK 19A", _T0)

<!-- Last changed: Facebook_20260825D -->
## 9 — Export shape

⚠️ **Nothing leaves the enclave without Meta's output review.** Block 20 only *builds*
the reviewable object; it deliberately does not write a file. It also refuses to treat a
preview run or an unfinished async pull as an exportable result.

`MIN_CELL_POSTS` suppresses thin cells and **defaults to 0 — retain everything**. That
is a considered position: a single-post cell is very often the signal rather than noise,
and suppressing at 5 or 10 deletes exactly the small single-farm observations
`max_sad_share_term` exists to surface. If a reviewer requires suppression the lever is
there, and the block reports how many early-warning cells the threshold would destroy
before you accept it.

**I am confabulating** on whether Meta's output review requires small-cell suppression
for an export of counts and ratios carrying no post text. Ask them rather than inferring
it; nothing inside the enclave needs it either way.

The state rollup is rebuilt from `SourcePosts` through the same `aggregate_posts` helper,
not summed out of the zip table. Summing recovers the totals but not the page-equal means
or the sentiment columns — which is exactly why the previous version had to drop
sentiment from the rollup. One code path, two grains, no drift.

If output review clears the aggregate, note that `job.write_data_to_file(directory=...,
filename=...)` writes **raw post rows**, not this table — a within-enclave convenience,
not an export route. The same is true of every checkpoint file.


In [ ]:
# BLOCK 20 - Export shape
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 20")

if IS_PREVIEW:
    print("RUN_MODE='preview' - this is a smoke test, not an exportable result.\n"
          "Set RUN_MODE='full', rerun the notebook, then build the export.\n")
elif SUBMIT_GATED:
    print("The estimate gate stopped submission - there is nothing to export.\n")
elif not ALL_JOBS_DONE:
    print("Async jobs are still running - this aggregate is PARTIAL.\n"
          "Re-run block 12 until every job is COMPLETE before exporting.\n")

ExportAgg = AggByCatStateZipDate.copy()

if MIN_CELL_POSTS > 0:
    _keep = ExportAgg["n_posts"] >= MIN_CELL_POSTS
    _lost_sig = int(ExportAgg.loc[~_keep, "max_sad_share_term"].notna().sum())
    print(f"suppressing {int((~_keep).sum()):,} of {len(ExportAgg):,} cells "
          f"below n_posts={MIN_CELL_POSTS} "
          f"({int(ExportAgg.loc[~_keep, 'n_posts'].sum()):,} posts)")
    if _lost_sig:
        print(f"  !! {_lost_sig:,} of the suppressed cells carry a "
              f"max_sad_share_term - that is the early-warning signal being "
              f"deleted. Confirm review actually requires this.")
    ExportAgg = ExportAgg[_keep].reset_index(drop=True)
else:
    # Deliberate, not an unset knob: a single-post cell is very often the signal
    # rather than noise, and suppressing at 5 or 10 deletes exactly the small
    # single-farm observations max_sad_share_term exists to surface.
    _thin = int((ExportAgg["n_posts"] == 1).sum())
    print(f"MIN_CELL_POSTS=0 - thin cells RETAINED by design "
          f"({_thin:,} single-post cells kept).")
    print("  Set MIN_CELL_POSTS only if Meta's output review requires it; nothing "
          "inside\n  the enclave needs it, and it costs signal.")

# State-level fallback if zip-day is judged too granular by output review.
# Rebuilt from SourcePosts through the SAME aggregate_posts helper rather than
# summed out of the zip table. Summing could recover the totals, but neither the
# page-equal means nor the sentiment columns survive that route - which is why
# the previous version had to drop sentiment from the rollup entirely.
ExportAggState = aggregate_posts(SourcePosts, GRP_STATE)

print(f"\nExportAgg      : {len(ExportAgg):,} rows x {ExportAgg.shape[1]} cols")
print(f"ExportAggState : {len(ExportAggState):,} rows x {ExportAggState.shape[1]} cols "
      f"(same columns as the zip grain, sentiment included)")

print(f"\nrun_mode={RUN_MODE}  window={RUN_START}..{RUN_END}  "
      f"engine={SENTIMENT_ENGINE}  query_mode={POST_QUERY_MODE}  "
      f"farm_only={RESTRICT_TO_FARM_ONLY}  engagement_min={ENGAGEMENT_MIN}")
print(f"langs={POST_LANGS} ({LANG_MODE_RESOLVED})  score_emoji={SCORE_EMOJI}  "
      f"lex_fingerprint={LEX_FINGERPRINT}")
if not IS_PREVIEW:
    print(f"job_mode={JOB_MODE}  job_window={JOB_WINDOW}  jobs={len(POST_JOBS)}  "
          f"all_complete={ALL_JOBS_DONE}")

# ---------------------------------------------------------------------------
# Write this shard's contribution. CSV as well as pickle: a pickle cannot pass
# output review, so if researchers turn out NOT to share a filesystem the CSV is
# the artifact each person carries out and block 22 recombines outside.
# ---------------------------------------------------------------------------
os.makedirs(SHARD_DIR, exist_ok=True)

ExportAgg = ExportAgg.assign(shard=SHARD)
ExportAggState = ExportAggState.assign(shard=SHARD)

_shard_manifest = dict(_ckpt_manifest("shard-export"),
                       categories=PageCategories,
                       min_cell_posts=MIN_CELL_POSTS,
                       n_rows=len(ExportAgg))

for _name, _obj in (("exportagg", ExportAgg), ("exportaggstate", ExportAggState)):
    _stem = os.path.join(SHARD_DIR, f"shard_{SHARD}_{_name}")
    _obj.to_csv(_stem + ".csv", index=False)
    _obj.to_pickle(_stem + ".pkl.gz", compression="gzip")
with open(os.path.join(SHARD_DIR, f"shard_{SHARD}_manifest.json"), "w",
          encoding="utf-8") as _fh:
    json.dump(_shard_manifest, _fh, default=str, indent=2)

print(f"\nshard '{SHARD}' written to {SHARD_DIR}")
for _f in sorted(f for f in os.listdir(SHARD_DIR) if f.startswith(f"shard_{SHARD}_")):
    print(f"  {_f}")
if IS_SMOKE:
    print("\n!! SMOKE shard - delete these before combining anything real.")

display(ExportAgg.head(20))
_block_end("BLOCK 20", _T0)

<!-- Last changed: Facebook_20260825A -->
## 10 — Final checkpoint

Persist the derived frames too. They are all recomputable from `POST_ROWS`, so this is
convenience rather than insurance — but it is the difference between reopening the
notebook tomorrow and re-running blocks 12–19 first.

Same boundary as everywhere else: this writes inside the enclave. `ExportAgg` becoming a
file here does **not** make it exported.


In [ ]:
# BLOCK 21 - Final checkpoint
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 21")

checkpoint("final")

print(f"\nrolling 7-day budget: {_budget_spent_7d():,} / {RECORD_BUDGET:,} spent, "
      f"{_budget_remaining():,} available")

if not IS_PREVIEW:
    _plan_n = len(_job_plan(SourcePages["id"]))
    _left = _plan_n - len(POST_JOBS)
    if _left > 0:
        print(f"{_left:,} of {_plan_n:,} planned job(s) still unsubmitted - this "
              f"collection is STAGED.\nRe-run block 12 when the rolling window frees "
              f"up; the ledger and job ids are checkpointed.")

print(f"\nshard '{SHARD}' -> {CKPT_DIR}")
print("\nto resume in a fresh session: set the same SHARD, then run blocks\n"
      "00-04 with\n"
      "RESTORE_FROM_CHECKPOINT = True, then run down the notebook. Page discovery\n"
      "skips, submitted jobs are not resubmitted, polling reattaches, and the\n"
      "budget ledger carries its rolling window forward.")
_block_end("BLOCK 21", _T0)

<!-- Last changed: Facebook_20260825B -->
## 11 — Combining the shards

Run this once all three shards exist. It works in either arrangement:

* **shared enclave workspace** — everyone writes into the same `SHARD_DIR`, combine
  inside, and put the merged aggregate through output review once;
* **isolated workspaces** — each researcher takes their own `shard_<name>_exportagg.csv`
  through review, and the three approved CSVs are combined outside the enclave.

The block prefers CSV over pickle for exactly that reason: a pickle cannot pass review.

The concat is the easy half. What this block is really for is refusing to merge shards
that were not produced the same way — mismatched sentiment engine, query mode, date
window, engagement threshold, job window, or suppression threshold. Every one of those
produces a combined table that looks entirely healthy and is quietly wrong.


In [ ]:
# BLOCK 22 - Combine shards
# Last changed: Facebook_20260901A
_T0 = _block_start("BLOCK 22")

COMBINE_SHARDS = ["nick", "max", "waree"]   # "smoke" is deliberately not here

# Fields that MUST agree across shards or the combined table is not one dataset.
# 20260901A adds three. lex_fingerprint is the important one: a hand-built
# Spanish lexicon is not calibrated against the English one, so if two
# researchers run DIFFERENT Spanish lexicons every number still computes, every
# sanity check still passes, and mean_sentiment_es is quietly incomparable
# across a third of the corpus - the exact failure sentiment_engine was locked
# to prevent, one level down.
_LOCKED = ("sentiment_engine", "query_mode", "run_start", "run_end",
           "engagement_min", "job_window", "min_cell_posts",
           "lex_fingerprint", "post_langs", "score_emoji")


def _load_shard(name, which="exportagg"):
    """CSV first - a pickle cannot pass output review, so CSV is the portable one."""
    stem = os.path.join(SHARD_DIR, f"shard_{name}_{which}")
    if os.path.exists(stem + ".csv"):
        # dtype=str on the key columns is NOT optional: read_csv infers "05010"
        # as the integer 5010, silently renumbering every zip in CT/MA/ME/NH/
        # NJ/RI/VT - and a renumbered key will not match its own shard's rows.
        df = pd.read_csv(stem + ".csv",
                         dtype={"zip": str, "state": str,
                                "page_category": str, "shard": str})
        if "post_date" in df.columns:
            df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce").dt.date
        if "zip" in df.columns:
            df["zip"] = df["zip"].str.strip()
        return df, "csv"
    if os.path.exists(stem + ".pkl.gz"):
        return pd.read_pickle(stem + ".pkl.gz", compression="gzip"), "pkl"
    raise FileNotFoundError(f"no {which} for shard '{name}' in {SHARD_DIR}")


# What is actually on disk, so a missing shard reads as "not produced yet"
# rather than as a failure. A smoke shard is never combined - it is an
# environment check on Zoo, and combining it would put test data in the dataset.
_present = sorted({f.split("_")[1] for f in os.listdir(SHARD_DIR)
                   if f.startswith("shard_")}) if os.path.isdir(SHARD_DIR) else []
print(f"SHARD_DIR      : {SHARD_DIR}")
print(f"shards on disk : {_present or 'none'}")
print(f"shards wanted  : {COMBINE_SHARDS}")
if "smoke" in _present:
    print("  note: 'smoke' is the Zoo environment check and is never combined.")
_waiting = [n for n in COMBINE_SHARDS if n not in _present]
if _waiting:
    print(f"  waiting on    : {_waiting}  (not yet produced)")
print()

frames, manifests, problems = {}, {}, []

for _name in COMBINE_SHARDS:
    try:
        frames[_name], _fmt = _load_shard(_name)
        with open(os.path.join(SHARD_DIR, f"shard_{_name}_manifest.json"),
                  encoding="utf-8") as _fh:
            manifests[_name] = json.load(_fh)
        print(f"  {_name:<8} {len(frames[_name]):>8,} rows  ({_fmt})  "
              f"engine={manifests[_name].get('sentiment_engine')}")
    except Exception as e:
        problems.append(f"shard '{_name}' has not been produced yet")
        print(f"  {_name:<8} not produced yet")

# --- the config lock --------------------------------------------------------
print("\nconfig lock:")
for _f in _LOCKED:
    vals = {n: m.get(_f) for n, m in manifests.items()}
    agree = len(set(map(str, vals.values()))) <= 1
    print(f"  {_f:<18} {'OK  ' if agree else 'DIFF'}  {vals}")
    if not agree:
        problems.append(f"{_f} differs across shards: {vals}")

# --- disjointness -----------------------------------------------------------
cats = {n: set(df['page_category']) for n, df in frames.items()}
for _a in cats:
    for _b in cats:
        if _a < _b and cats[_a] & cats[_b]:
            problems.append(f"shards {_a}/{_b} share categories {cats[_a] & cats[_b]}")

if problems:
    print("\n!! NOT COMBINING - resolve these first:")
    for _p in problems:
        print(f"  - {_p}")
    CombinedAgg = None
else:
    GRP_C = ["page_category", "state", "zip", "post_date"]
    CombinedAgg = (pd.concat(frames.values(), ignore_index=True)
                   .sort_values(GRP_C).reset_index(drop=True))

    dup = CombinedAgg.duplicated(subset=GRP_C).sum()
    assert dup == 0, f"{dup} duplicate aggregate cells - shards are not disjoint"

    print(f"\nCombinedAgg: {len(CombinedAgg):,} rows, "
          f"{CombinedAgg['n_posts'].sum():,} posts")
    print(CombinedAgg.groupby("shard")
          .agg(rows=("n_posts", "size"), posts=("n_posts", "sum"),
               categories=("page_category", "nunique")).to_string())
    print(f"\nstates {CombinedAgg['state'].nunique()}   "
          f"dates {CombinedAgg['post_date'].nunique()}   "
          f"categories {CombinedAgg['page_category'].nunique()}")
    display(CombinedAgg.head(20))
_block_end("BLOCK 22", _T0)

<!-- Last changed: Facebook_20260831A -->
## 9 — Timing

Every code block brackets itself: it prints the wall-clock time on entry, and on exit
prints the wall-clock time again plus how long it took in hours and minutes. Block 23
prints the whole ledger.

This exists because of a specific failure. On 2026-08-31 a `SHARD="nick"`,
`RUN_MODE="full"` run sat on block 07 for over three hours and was killed, and there was
no way to tell from the notebook whether it was working or hung — no elapsed time, no
per-category progress, and no partial checkpoint. Both halves of that are fixed here:
block 07 now checkpoints per category, and every block now reports its own duration.

Read the ledger for two things:

* **`status`.** `done` means the block reached its `_block_end`. `running` means it did
  not — it raised, it was interrupted, or it is still going — and its elapsed is measured
  to *now* and flagged with a trailing `+`. A block that was killed shows up as a
  duration, which is the case that previously left no trace at all.
* **`runs`.** Re-running a cell replaces its record rather than appending a second one,
  so the table stays one row per block; `runs` is how many times that cell has executed
  in this kernel. The timings are always the most recent execution.

Only blocks **07** and **12B** should show hours. Anything else in hours is a symptom.

> The ledger is wall-clock only — no counts, nothing derived from Meta data — so it can
> be pasted into Slack for coordination, unlike anything that reports rows. It is
> deliberately **not** checkpointed: timings describe the run, not the data, and a
> restored checkpoint would carry a dead session's durations forward as if they were
> this one's.

In [ ]:
# BLOCK 23 - Block timing report
# Last changed: Facebook_20260831A
_T0 = _block_start("BLOCK 23")

# The whole ledger, in the order the blocks first ran. Read it top-down. Two
# blocks are supposed to cost real time - 07 (page discovery, synchronous and
# rate-limited) and 12B (submit and poll, 1 query/minute) - and anything else
# showing hours is a symptom rather than a workload.
#
# status='running' means that block never reached its _block_end: it raised, or
# it was interrupted, or it is still executing in another cell. Its elapsed is
# measured to NOW and flagged with a trailing '+'.
BlockTimings = timings_frame()

print(f"{len(BlockTimings)} block(s) executed in this kernel\n")
print(BlockTimings.to_string(index=False))

_secs = BlockTimings["seconds"].fillna(0)
print(f"\ntotal measured : {_fmt_hm(_secs.sum())} ({_secs.sum():,.1f}s) across "
      f"{int(BlockTimings['runs'].fillna(0).sum())} block run(s)")

_unfinished = BlockTimings[BlockTimings["status"] == "running"]
if len(_unfinished):
    print(f"!! {len(_unfinished)} block(s) never finished: "
          f"{', '.join(_unfinished['block'])}")

if len(BlockTimings):
    print("\nslowest:")
    for _r in BlockTimings.nlargest(3, "seconds").itertuples():
        print(f"  {_r.block:<11} {_r.elapsed:<10} {_r.started} -> {_r.ended}")

# Wall-clock only - no counts, no Meta-derived values - so this table can be
# shared for coordination between shards. It is NOT part of the export, and it
# is not written to the checkpoint: timings describe the run, not the data, and
# a restored checkpoint would carry a previous session's durations forward as if
# they belonged to this one.
_block_end("BLOCK 23", _T0)